# Inspect the Expanded Dataset
### Dataset Inventory

Objective: Inspect the expanded music dataset and identify the available melody, chord, beat, and key annotation files.
Before modifying the filtering criteria or implementing a new key estimation algorithm, the dataset structure must be examined. Particular attention will be given to the availability of absolute chord information, which is necessary for deriving musical keys independently of existing Roman numeral annotations.
This initial inspection does not modify any source files.


In [2]:
from pathlib import Path
import json
import pandas as pd


# -----------------------------------
# 1. Locate project and dataset
# -----------------------------------

PROJECT_ROOT = next(
    (
        parent
        for parent in [Path.cwd(), *Path.cwd().parents]
        if (parent / "datasets" / "mtg-jamendo").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Check your notebook's working directory."
    )

OUTPUTS_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "mtg-jamendo"
    / "melody-chord-extraction-pipeline"
    / "outputs"
)

assert OUTPUTS_DIR.is_dir(), (
    f"Dataset output directory not found: {OUTPUTS_DIR}"
)

print("PROJECT DIRECTORY")
print("--------------------------------")
print("Project:", PROJECT_ROOT)
print("Outputs:", OUTPUTS_DIR)


# -----------------------------------
# 2. Discover annotation files
# -----------------------------------

file_patterns = {
    "melody": "melody/combined.json",
    "chords": "roman/submission.json",
    "beats": "beats/beats.json",
    "key": "key.json"
}

track_files = {}

for category, pattern in file_patterns.items():

    paths = list(
        OUTPUTS_DIR.rglob(pattern)
    )

    # For each annotation file, identify
    # its containing track directory.

    track_files[category] = {
        path.parent.parent
        if "/" in pattern
        else path.parent
        for path in paths
    }


# -----------------------------------
# 3. Calculate file availability
# -----------------------------------

all_tracks = set().union(
    *track_files.values()
)

inventory_rows = []

for category, tracks in track_files.items():

    inventory_rows.append({
        "Annotation": category,
        "Tracks": len(tracks)
    })

inventory_df = pd.DataFrame(
    inventory_rows
)

print("\nANNOTATION FILE AVAILABILITY")
print("--------------------------------")

print(
    "Discovered track directories:",
    len(all_tracks)
)

display(inventory_df)


# -----------------------------------
# 4. Find complete and incomplete tracks
# -----------------------------------

complete_tracks = set.intersection(
    *track_files.values()
)

missing_counts = {
    category: len(all_tracks - tracks)
    for category, tracks in track_files.items()
}

print("\nDATA COMPLETENESS")
print("--------------------------------")

print(
    "Tracks with all four annotation files:",
    len(complete_tracks)
)

print("\nMissing annotations:")

for category, count in missing_counts.items():
    print(f"{category:10s}: {count}")


# -----------------------------------
# 5. Select one complete track
# -----------------------------------

assert complete_tracks, (
    "No complete tracks were found. "
    "Inspect the directory structure."
)

sample_track = sorted(
    complete_tracks
)[0]

print("\nSAMPLE TRACK")
print("--------------------------------")

print(
    "Track directory:",
    sample_track.name
)


# -----------------------------------
# 6. Inspect available JSON files
# -----------------------------------

print("\nAVAILABLE JSON FILES")
print("--------------------------------")

sample_json_files = sorted(
    sample_track.rglob("*.json")
)

for path in sample_json_files[:30]:

    print(
        path.relative_to(sample_track)
    )

if len(sample_json_files) > 30:
    print(
        f"... and {len(sample_json_files) - 30} more"
    )


# -----------------------------------
# 7. Inspect annotation structures
# -----------------------------------

print("\nANNOTATION STRUCTURES")
print("--------------------------------")

for category, relative_path in (
    file_patterns.items()
):

    path = sample_track / relative_path

    with open(path, "r") as f:
        data = json.load(f)

    print(f"\n{category.upper()}")
    print("-" * 35)

    print("Data type:", type(data).__name__)

    if isinstance(data, dict):

        print(
            "Top-level keys:",
            list(data.keys())[:25]
        )

        # Inspect nested structures without
        # printing entire annotation files.

        for key, value in data.items():

            if isinstance(value, list) and value:

                print(
                    f"List '{key}':",
                    len(value),
                    "items"
                )

                if isinstance(value[0], dict):

                    print(
                        "First item fields:",
                        list(value[0].keys())[:25]
                    )

    elif isinstance(data, list):

        print(
            "Number of items:",
            len(data)
        )

        if data and isinstance(data[0], dict):

            print(
                "First item fields:",
                list(data[0].keys())[:25]
            )


print("\nDataset inventory completed!")


PROJECT DIRECTORY
--------------------------------
Project: /Users/praise/Developer/vocal-harmonization
Outputs: /Users/praise/Developer/vocal-harmonization/datasets/mtg-jamendo/melody-chord-extraction-pipeline/outputs

ANNOTATION FILE AVAILABILITY
--------------------------------
Discovered track directories: 1603


,Annotation,Tracks
0,melody,1603
1,chords,1603
2,beats,1603
3,key,1603



DATA COMPLETENESS
--------------------------------
Tracks with all four annotation files: 1603

Missing annotations:
melody    : 0
chords    : 0
beats     : 0
key       : 0

SAMPLE TRACK
--------------------------------
Track directory: track_0000382

AVAILABLE JSON FILES
--------------------------------
beats/beats.json
bpdr/boundaries.json
bpdr/record.json
chords/full.json
chords/submission.json
key.json
key_comparison.json
melody/combined.json
melody/game_vocal.json
melody/sheetsage2_instrumental.json
roman/full.json
roman/submission.json

ANNOTATION STRUCTURES
--------------------------------

MELODY
-----------------------------------
Data type: dict
Top-level keys: ['schema_version', 'track_id', 'mtg_track_id', 'method', 'source_mp3', 'source_fingerprint', 'time_origin', 'pitch_unit', 'duration_sec', 'notes', 'selection']
List 'notes': 313 items
First item fields: ['onset_sec', 'offset_sec', 'midi_pitch', 'midi_pitch_rounded', 'velocity', 'source']

CHORDS
----------------------

##### Findings
The expanded dataset contains 1,603 tracks, all containing melody, chord, beat, and key annotation files. The pipeline additionally provides absolute chord annotations, Roman numeral mappings, and key-comparison information.
Unlike Phase 2, which applied strict key-agreement filtering, Phase 3 will investigate independent key estimation using chord progressions and melodic pitch distributions. The objective is to retain more usable songs while maintaining reliable harmonic labels.

### Absolute Chord and Key Annotation Inspection
Objective: Examine the structure of absolute chord annotations and existing key estimates to determine which information can be used for independent key estimation.
Chord-based key estimation requires absolute chord roots, chord qualities, and durations. These features can provide evidence of tonal centers and harmonic relationships. Melody-based estimates will serve as an independent comparison.
This inspection will help select appropriate key estimation methods without relying on existing Roman numeral mappings.

In [3]:
import json
from pathlib import Path


# -----------------------------------
# 1. Files to inspect
# -----------------------------------

files_to_inspect = [
    "chords/submission.json",
    "chords/full.json",
    "key.json",
    "key_comparison.json"
]


# -----------------------------------
# 2. Helper: preview JSON values
# -----------------------------------

def preview_value(value, max_chars=700):

    text = json.dumps(
        value,
        indent=2,
        default=str
    )

    if len(text) > max_chars:
        text = text[:max_chars] + "\n... (truncated)"

    return text


# -----------------------------------
# 3. Inspect each file
# -----------------------------------

for relative_path in files_to_inspect:

    path = sample_track / relative_path

    print("\n" + "=" * 65)
    print(relative_path.upper())
    print("=" * 65)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    with open(path, "r") as f:
        data = json.load(f)

    print("\nTop-level structure:")

    if isinstance(data, dict):

        print(list(data.keys()))

        # -----------------------------------
        # A. Inspect metadata
        # -----------------------------------

        print("\nMETADATA")
        print("--------------------------------")

        for key, value in data.items():

            if isinstance(value, list):
                continue

            print(
                f"\n{key}:"
            )

            print(
                preview_value(value, max_chars=450)
            )

        # -----------------------------------
        # B. Inspect event lists
        # -----------------------------------

        print("\nANNOTATION LISTS")
        print("--------------------------------")

        for key, value in data.items():

            if not isinstance(value, list):
                continue

            print(
                f"\nList: {key}"
            )

            print(
                "Number of items:",
                len(value)
            )

            if value:

                print("\nFirst item:")

                print(
                    preview_value(
                        value[0],
                        max_chars=1200
                    )
                )

                if len(value) > 1:

                    print("\nSecond item:")

                    print(
                        preview_value(
                            value[1],
                            max_chars=1200
                        )
                    )

    else:

        print(
            preview_value(
                data,
                max_chars=2000
            )
        )


print("\nAnnotation inspection completed!")


CHORDS/SUBMISSION.JSON

Top-level structure:
['schema_version', 'track_id', 'mtg_track_id', 'source_mp3', 'source_fingerprint', 'time_origin', 'duration_sec', 'model', 'quality_flags', 'no_chord_fraction', 'events']

METADATA
--------------------------------

schema_version:
1

track_id:
"382"

mtg_track_id:
"track_0000382"

source_mp3:
"vocal-tagged-audio/82/382.low.mp3"

source_fingerprint:
{
  "size_bytes": 2394721,
  "mtime_ns": 1627592200000000000
}

time_origin:
"full_song_start"

duration_sec:
211.04

model:
{
  "name": "lv-chordia",
  "version": "1.1.0",
  "vocabulary": "submission"
}

no_chord_fraction:
0.0236

ANNOTATION LISTS
--------------------------------

List: quality_flags
Number of items: 0

List: events
Number of items: 76

First item:
{
  "start_sec": 0.0,
  "end_sec": 0.56,
  "chord_label": "N",
  "root": null,
  "quality": null,
  "bass_degree": null
}

Second item:
{
  "start_sec": 0.56,
  "end_sec": 4.37,
  "chord_label": "F:maj/3",
  "root": "F",
  "quality": 

##### Findings
Inspection confirmed that the extraction pipeline provides absolute chord annotations in both submission and full vocabularies. The submission vocabulary was selected as the primary source because the proposed model predicts basic Roman numeral chord classes. The full vocabulary will be retained for possible comparative experiments.
The pipeline already implements melody-based Krumhansl–Kessler key estimation and duration-weighted chord-based key estimation. These estimates will serve as reference methods rather than automatically determining the new dataset's key labels. Phase 3 will investigate a chord-informed key estimation approach incorporating harmonic relationships, chord durations, and tonal context.

### Key Estimation Agreement Analysis
Objective: Analyze key-estimation agreement and confidence across all 1,603 tracks.
This experiment compares existing melody-derived and chord-derived key estimates, identifying exact agreement, relative major/minor relationships, parallel major/minor relationships, and other disagreements. Confidence distributions and review flags will also be examined to identify opportunities for increasing dataset retention without accepting unreliable annotations.

In [4]:
import json
import pandas as pd
from collections import Counter


# -----------------------------------
# 1. Helper functions
# -----------------------------------

def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def key_relationship(melody, chord):

    m_pc = melody.get("tonic_pc")
    c_pc = chord.get("tonic_pc")

    m_mode = melody.get("mode")
    c_mode = chord.get("mode")

    if (
        m_pc is None
        or c_pc is None
        or m_mode not in ("major", "minor")
        or c_mode not in ("major", "minor")
    ):
        return "unavailable"

    m_pc = int(m_pc)
    c_pc = int(c_pc)

    # Same tonic and mode
    if m_pc == c_pc and m_mode == c_mode:
        return "exact_agreement"

    # Same tonic, different mode
    if m_pc == c_pc and m_mode != c_mode:
        return "parallel_major_minor"

    # Relative major/minor:
    # C major (0) and A minor (9), for example.

    if m_mode != c_mode:

        major_pc = (
            m_pc if m_mode == "major" else c_pc
        )

        minor_pc = (
            m_pc if m_mode == "minor" else c_pc
        )

        if major_pc == (minor_pc + 3) % 12:
            return "relative_major_minor"

    return "other_disagreement"


# -----------------------------------
# 2. Analyze all tracks
# -----------------------------------

key_records = []

for track_dir in sorted(all_tracks):

    try:

        key_data = read_json(
            track_dir / "key.json"
        )

        comparison = read_json(
            track_dir / "key_comparison.json"
        )

        chord_data = read_json(
            track_dir / "chords/submission.json"
        )

        melody_data = read_json(
            track_dir / "melody/combined.json"
        )

        melody_key = (
            key_data.get("melody_key", {})
            .get("selected", {})
        )

        chord_key_info = comparison.get(
            "chord_key", {}
        )

        chord_key = chord_key_info.get(
            "selected", {}
        )

        relationship = key_relationship(
            melody_key,
            chord_key
        )

        key_records.append({

            "track_id": track_dir.name,

            "relationship": relationship,

            "melody_tonic":
                melody_key.get("tonic"),

            "melody_mode":
                melody_key.get("mode"),

            "melody_confidence":
                melody_key.get("confidence"),

            "chord_tonic":
                chord_key.get("tonic"),

            "chord_mode":
                chord_key.get("mode"),

            "chord_confidence":
                chord_key.get("confidence"),

            "pipeline_agreement":
                comparison.get("agreement"),

            "review_flag":
                comparison.get("review_flag"),

            "review_reasons":
                comparison.get("review_reasons", []),

            "melody_notes":
                len(melody_data.get("notes", [])),

            "chord_events":
                len(chord_data.get("events", [])),

            "usable_chord_segments":
                chord_key_info.get(
                    "usable_chord_segments"
                ),

            "no_chord_fraction":
                chord_data.get("no_chord_fraction"),

            "duration_sec":
                chord_data.get("duration_sec")
        })

    except Exception as error:

        print(
            f"Failed: {track_dir.name} | {error}"
        )


key_records_df = pd.DataFrame(
    key_records
)


# -----------------------------------
# 3. Dataset overview
# -----------------------------------

print("KEY ESTIMATION DATASET")
print("--------------------------------")

print("Tracks analyzed:", len(key_records_df))


# -----------------------------------
# 4. Key relationship distribution
# -----------------------------------

relationship_counts = (
    key_records_df["relationship"]
    .value_counts()
)

relationship_df = (
    relationship_counts
    .rename_axis("Relationship")
    .reset_index(name="Tracks")
)

relationship_df["Percentage"] = (
    relationship_df["Tracks"]
    / len(key_records_df)
    * 100
)

print("\nKEY RELATIONSHIPS")
print("--------------------------------")

display(
    relationship_df.round(2)
)


# -----------------------------------
# 5. Confidence distributions
# -----------------------------------

print("\nMELODY KEY CONFIDENCE")
print("--------------------------------")

display(
    key_records_df[
        "melody_confidence"
    ].value_counts(dropna=False)
)


print("\nCHORD KEY CONFIDENCE")
print("--------------------------------")

display(
    key_records_df[
        "chord_confidence"
    ].value_counts(dropna=False)
)


# -----------------------------------
# 6. Confidence vs agreement
# -----------------------------------

print("\nKEY RELATIONSHIP BY CONFIDENCE")
print("--------------------------------")

confidence_table = pd.crosstab(
    [
        key_records_df["melody_confidence"],
        key_records_df["chord_confidence"]
    ],
    key_records_df["relationship"]
)

display(confidence_table)


# -----------------------------------
# 7. Review flags
# -----------------------------------

print("\nREVIEW FLAGS")
print("--------------------------------")

display(
    key_records_df[
        "review_flag"
    ].value_counts(dropna=False)
)

reason_counts = Counter()

for reasons in key_records_df["review_reasons"]:
    if isinstance(reasons, list):
        reason_counts.update(reasons)

print("\nMOST COMMON REVIEW REASONS")
print("--------------------------------")

display(
    pd.DataFrame(
        reason_counts.most_common(15),
        columns=["Reason", "Tracks"]
    )
)


# -----------------------------------
# 8. Dataset quality overview
# -----------------------------------

print("\nDATASET QUALITY OVERVIEW")
print("--------------------------------")

display(
    key_records_df[
        [
            "melody_notes",
            "chord_events",
            "usable_chord_segments",
            "no_chord_fraction",
            "duration_sec"
        ]
    ].describe().round(2)
)


# -----------------------------------
# 9. Inspect disagreements
# -----------------------------------

print("\nEXAMPLE KEY DISAGREEMENTS")
print("--------------------------------")

disagreements = key_records_df[
    key_records_df["relationship"]
    != "exact_agreement"
]

display(
    disagreements[
        [
            "track_id",
            "relationship",
            "melody_tonic",
            "melody_mode",
            "melody_confidence",
            "chord_tonic",
            "chord_mode",
            "chord_confidence"
        ]
    ].head(15)
)


print("\nKey agreement analysis completed!")

Failed: track_0205076 | 'NoneType' object has no attribute 'get'
Failed: track_0585146 | 'NoneType' object has no attribute 'get'
Failed: track_0585147 | 'NoneType' object has no attribute 'get'
Failed: track_0585149 | 'NoneType' object has no attribute 'get'
Failed: track_0585157 | 'NoneType' object has no attribute 'get'
Failed: track_0648408 | 'NoneType' object has no attribute 'get'
Failed: track_0659887 | 'NoneType' object has no attribute 'get'
Failed: track_0659895 | 'NoneType' object has no attribute 'get'
Failed: track_0939272 | 'NoneType' object has no attribute 'get'
Failed: track_0939273 | 'NoneType' object has no attribute 'get'
Failed: track_0939274 | 'NoneType' object has no attribute 'get'
Failed: track_0939275 | 'NoneType' object has no attribute 'get'
Failed: track_0939276 | 'NoneType' object has no attribute 'get'
Failed: track_0989846 | 'NoneType' object has no attribute 'get'
Failed: track_1004129 | 'NoneType' object has no attribute 'get'
Failed: track_1075215 | '

,Relationship,Tracks,Percentage
0,exact_agreement,830,52.90
1,other_disagreement,491,31.29
2,parallel_major_minor,136,8.67
3,relative_major_minor,112,7.14



MELODY KEY CONFIDENCE
--------------------------------


melody_confidence
high      819
low       398
medium    352
Name: count, dtype: int64


CHORD KEY CONFIDENCE
--------------------------------


...


DATASET QUALITY OVERVIEW
--------------------------------


,melody_notes,chord_events,usable_chord_segments,no_chord_fraction,duration_sec
count,1569.00,1569.00,1569.00,1569.00,1569.00
mean,492.05,71.24,68.62,0.09,227.31
std,289.23,46.01,45.92,0.13,108.72
min,3.00,2.00,1.00,0.00,30.48
25%,313.00,35.00,32.00,0.02,169.00
50%,448.00,68.00,65.00,0.04,216.83
75%,608.00,99.00,96.00,0.10,267.12
max,2463.00,399.00,387.00,0.97,1703.37



EXAMPLE KEY DISAGREEMENTS
--------------------------------


,track_id,relationship,melody_tonic,melody_mode,melody_confidence,chord_tonic,chord_mode,chord_confidence
1,track_0000383,other_disagreement,B,minor,medium,G,major,high
2,track_0000384,other_disagreement,Bb,minor,high,Ab,minor,low
4,track_0000387,other_disagreement,F,minor,medium,Db,major,low
6,track_0000389,other_disagreement,F,major,low,C,major,medium
9,track_0000392,other_disagreement,Gb,major,medium,Db,major,medium
10,track_0000393,other_disagreement,Db,major,low,Gb,major,high
12,track_0000395,other_disagreement,Eb,major,medium,Bb,major,low
13,track_0002171,other_disagreement,Bb,major,low,D,minor,high
15,track_0007392,other_disagreement,B,major,low,A,minor,high
16,track_0007395,relative_major_minor,A,minor,low,C,major,high



Key agreement analysis completed!


### Manual Key Verification
Randomly selected tracks with disagreements between previously estimated melody and chord keys for manual musical verification. Tracks were sampled from different sections of the dataset, excluding the earliest and latest 100 tracks. The objective is to assess which estimation method better represents the perceived tonal center.

In [5]:
import pandas as pd
import json

# Exclude the first and last 100 tracks
df = (
    key_records_df
    .sort_values("track_id")
    .iloc[100:-100]
    .copy()
)

# Keep only tracks with key disagreements
df = df[
    df["relationship"] != "exact_agreement"
].copy()

# Divide tracks into 4 groups by track number
df["group"] = pd.qcut(
    df["track_id"].str.extract(r"(\d+)$")[0].astype(int),
    q=4,
    labels=False
)

# Randomly select 3 from each group
SEED = 42

samples = pd.concat([
    group.sample(n=min(3, len(group)), random_state=SEED + i)
    for i, group in df.groupby("group")
]).sort_values("track_id")

# Include original audio paths
track_lookup = {
    path.name: path for path in all_tracks
}

samples["audio_file"] = [
    read_json(
        track_lookup[tid] / "chords/submission.json"
    )["source_mp3"]
    for tid in samples["track_id"]
]

# Display comparison
display(samples[[
    "track_id",
    "relationship",
    "melody_tonic",
    "melody_mode",
    "melody_confidence",
    "chord_tonic",
    "chord_mode",
    "chord_confidence",
    "audio_file"
]])

print(f"Selected {len(samples)} random tracks.")

,track_id,relationship,melody_tonic,melody_mode,melody_confidence,chord_tonic,chord_mode,chord_confidence,audio_file
298,track_0571479,other_disagreement,E,major,high,Gb,minor,high,vocal-tagged-audio/79/571479.low.mp3
341,track_0603758,relative_major_minor,G,major,low,E,minor,high,vocal-tagged-audio/58/603758.low.mp3
382,track_0659892,other_disagreement,F,major,high,Bb,minor,high,vocal-tagged-audio/92/659892.low.mp3
419,track_0785281,other_disagreement,Ab,major,high,Gb,minor,medium,vocal-tagged-audio/81/785281.low.mp3
453,track_0854720,relative_major_minor,A,major,low,Gb,minor,medium,vocal-tagged-audio/20/854720.low.mp3
519,track_0920193,other_disagreement,Gb,major,high,Db,major,high,vocal-tagged-audio/93/920193.low.mp3
761,track_1049878,other_disagreement,A,minor,low,D,major,medium,vocal-tagged-audio/78/1049878.low.mp3
892,track_1075195,other_disagreement,E,major,low,Gb,minor,high,vocal-tagged-audio/95/1075195.low.mp3
1046,track_1096796,other_disagreement,B,minor,high,G,major,medium,vocal-tagged-audio/96/1096796.low.mp3
1071,track_1098835,other_disagreement,B,major,medium,E,major,medium,vocal-tagged-audio/35/1098835.low.mp3


Selected 12 random tracks.


### Manual Key and Harmonic Consistency Investigation
Objective: Investigate key-estimation disagreements through manual musical verification and analysis of absolute chord progressions, melody pitch classes, and harmonic diversity.
Preliminary listening tests identified several cases of relative major/minor ambiguity, repeated two-chord progressions, potential power-chord classification errors, and unusual tuning. These findings suggest that strict agreement between independently estimated keys is unsuitable as the sole dataset-selection criterion.
This investigation examines the extracted musical evidence before designing revised key estimation and data-quality filters.

In [7]:
from collections import Counter
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Tracks manually investigated
# -----------------------------------

MANUAL_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0659892",
    "track_0785281",
    "track_0854720",
    "track_0920193",
    "track_1049878",
    "track_1075195",
    "track_1096796",
    "track_1098835",
    "track_1129652"
]

# Add 3 random disagreements from
# elsewhere in the dataset.

candidates = key_records_df.copy()

candidates["track_number"] = (
    candidates["track_id"]
    .str.extract(r"(\d+)$")[0]
    .astype(int)
)

candidates = candidates[
    (candidates["relationship"] != "exact_agreement")
    & (candidates["track_number"] > 40000)
    & (~candidates["track_id"].isin(MANUAL_TRACKS))
]

RANDOM_TRACKS = (
    candidates.sample(
        n=3,
        random_state=42
    )["track_id"].tolist()
)

INSPECTION_TRACKS = MANUAL_TRACKS + RANDOM_TRACKS

track_lookup = {
    path.name: path
    for path in all_tracks
}

print("ADDITIONAL RANDOM TRACKS")
print("--------------------------------")
print(RANDOM_TRACKS)


# -----------------------------------
# 2. Note names
# -----------------------------------

NOTE_NAMES = [
    "C", "C#", "D", "Eb", "E", "F",
    "F#", "G", "Ab", "A", "Bb", "B"
]

def note_name(midi):
    midi = int(round(midi))
    return f"{NOTE_NAMES[midi % 12]}{midi // 12 - 1}"


def format_key(key):
    return (
        f"{key.get('tonic', '?')} "
        f"{key.get('mode', '?')} "
        f"({key.get('confidence', '?')})"
    )


# -----------------------------------
# 3. Inspect song-level chord diversity
# -----------------------------------

summary_rows = []

for track_id in INSPECTION_TRACKS:

    folder = track_lookup[track_id]

    melody = read_json(
        folder / "melody/combined.json"
    )

    chords = read_json(
        folder / "chords/submission.json"
    )

    comparison = read_json(
        folder / "key_comparison.json"
    )

    events = [
        e for e in chords["events"]
        if e.get("root") is not None
        and e["chord_label"] != "N"
    ]

    roots = {
        e["root"]
        for e in events
    }

    chord_labels = {
        e["chord_label"]
        for e in events
    }

    duration_counts = Counter()

    for event in events:
        duration_counts[event["chord_label"]] += (
            event["end_sec"] - event["start_sec"]
        )

    top_chords = ", ".join(
        f"{label} ({seconds:.0f}s)"
        for label, seconds
        in duration_counts.most_common(5)
    )

    mk = comparison.get("melody_key") or {}
    ck = (
        (comparison.get("chord_key") or {})
        .get("selected") or {}
    )

    summary_rows.append({
        "Track": track_id,
        "Melody key": format_key(mk),
        "Chord key": format_key(ck),
        "Notes": len(melody["notes"]),
        "Unique roots": len(roots),
        "Unique chord labels": len(chord_labels),
        "Top chords by duration": top_chords
    })


print("\nSONG-LEVEL HARMONIC SUMMARY")
print("--------------------------------")

display(
    pd.DataFrame(summary_rows)
)


# -----------------------------------
# 4. Detailed melody–chord inspection
# -----------------------------------

# Selected interesting examples,
# plus the additional random tracks.

DETAIL_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0785281",
    "track_1096796",
    "track_1129652"
] + RANDOM_TRACKS

FIRST_SECONDS = 60
MAX_CHORD_EVENTS = 15


# -----------------------------------
# 5. Print melody notes over each chord
# -----------------------------------

for track_id in DETAIL_TRACKS:

    folder = track_lookup[track_id]

    melody = read_json(
        folder / "melody/combined.json"
    )

    chords = read_json(
        folder / "chords/submission.json"
    )

    comparison = read_json(
        folder / "key_comparison.json"
    )

    mk = comparison.get("melody_key") or {}
    ck = (
        (comparison.get("chord_key") or {})
        .get("selected") or {}
    )

    notes = sorted(
        melody["notes"],
        key=lambda n: n["onset_sec"]
    )

    events = [
        e for e in chords["events"]
        if e["start_sec"] < FIRST_SECONDS
        and e["end_sec"] > 0
    ][:MAX_CHORD_EVENTS]

    print("\n" + "=" * 75)
    print("TRACK:", track_id)
    print("Melody key:", format_key(mk))
    print("Chord key: ", format_key(ck))
    print("=" * 75)

    for event in events:

        start = event["start_sec"]

        end = min(
            event["end_sec"],
            FIRST_SECONDS
        )

        overlapping = [
            n for n in notes
            if n["onset_sec"] < end
            and n["offset_sec"] > start
            and n.get("midi_pitch_rounded") is not None
        ]

        # Duration-weighted melody notes
        # overlapping this chord event.

        pitch_durations = Counter()

        for note in overlapping:

            overlap = max(
                0,
                min(note["offset_sec"], end)
                - max(note["onset_sec"], start)
            )

            pitch = note_name(
                note["midi_pitch_rounded"]
            )

            pitch_durations[pitch] += overlap

        top_notes = ", ".join(
            f"{pitch} ({duration:.1f}s)"
            for pitch, duration
            in pitch_durations.most_common(7)
        )

        if not top_notes:
            top_notes = "No melody"

        print(
            f"{start:5.1f}–{end:5.1f}s | "
            f"{event['chord_label']:12s} | "
            f"{top_notes}"
        )

print("\nInspection completed!")

ADDITIONAL RANDOM TRACKS
--------------------------------
['track_1106439', 'track_1075177', 'track_0802276']

SONG-LEVEL HARMONIC SUMMARY
--------------------------------


,Track,Melody key,Chord key,Notes,Unique roots,Unique chord labels,Top chords by duration
0,track_0571479,E major (high),Gb minor (high),453,3,5,"E:maj (85s), F#:min (67s), B:min (44s), F#:min..."
1,track_0603758,G major (low),E minor (high),490,5,8,"E:min (50s), G:maj (34s), E:maj (25s), A:maj (..."
2,track_0659892,F major (high),Bb minor (high),170,1,2,"Bb:min7 (124s), Bb:min9 (4s)"
3,track_0785281,Ab major (high),Gb minor (medium),176,7,12,"D:maj (46s), A:maj (25s), F#:min (15s), G:maj ..."
4,track_0854720,A major (low),Gb minor (medium),397,4,4,"F#:min (97s), A:maj (90s), D:maj (33s), B:min ..."
5,track_0920193,Gb major (high),Db major (high),642,4,6,"F#:maj (101s), C#:maj (90s), A:maj (3s), F#:mi..."
6,track_1049878,A minor (low),D major (medium),591,8,13,"G:maj (81s), D:maj (60s), B:7 (30s), A:maj (21..."
7,track_1075195,E major (low),Gb minor (high),210,3,4,"F#:min7 (109s), F#:min (22s), C#:min7 (17s), B..."
8,track_1096796,B minor (high),G major (medium),484,4,7,"D:maj (131s), C:maj (66s), E:min (23s), G:maj/..."
9,track_1098835,B major (medium),E major (medium),615,6,10,"C#:min (59s), B:maj (54s), A:maj (48s), E:maj ..."



TRACK: track_0571479
Melody key: E major (high)
Chord key:  Gb minor (high)
  0.0–  0.0s | N            | C#5 (0.0s)
  0.0–  2.2s | F#:min7      | A4 (0.8s), F#4 (0.6s), C#5 (0.5s), E4 (0.3s)
  2.2–  4.2s | B:min        | F#4 (0.8s), A4 (0.7s), E4 (0.3s), B4 (0.1s)
  4.2–  6.7s | F#:min       | A4 (0.8s), F#4 (0.6s), C#5 (0.6s), B3 (0.3s), E4 (0.3s)
  6.7–  8.4s | B:min        | F#4 (0.8s), A4 (0.7s), B4 (0.1s)
  8.4– 11.2s | F#:min       | A4 (0.8s), F#4 (0.6s), C#5 (0.6s), E4 (0.5s), B3 (0.3s)
 11.2– 13.2s | B:min        | F#4 (0.8s), A4 (0.7s), E4 (0.3s), B4 (0.2s), B3 (0.0s)
 13.2– 15.7s | F#:min       | A4 (0.9s), C#5 (0.5s), F#4 (0.5s), E4 (0.3s), B3 (0.3s)
 15.7– 17.9s | B:min        | F#4 (1.1s), A4 (0.6s), E4 (0.3s), B3 (0.2s)
 17.9– 20.2s | F#:min       | B4 (0.6s), E5 (0.5s), A4 (0.3s), F#4 (0.3s), B3 (0.1s)
 20.2– 22.1s | B:min        | B4 (1.1s), A4 (0.3s), F#4 (0.3s)
 22.1– 24.7s | F#:min       | E5 (0.6s), B4 (0.6s), F#4 (0.3s), A4 (0.3s)
 24.7– 26.8s | B:min        | B

### Chord Root Diversity
Objective: Analyze the number of distinct harmonic roots across all 1,603 tracks to evaluate a proposed minimum chord-diversity criterion.
Absolute chord annotations from chords/submission.json are used to count unique chord roots independently of chord quality. Enharmonically equivalent roots, such as F♯ and G♭, are treated as identical.
Two measurements are calculated: total distinct roots and meaningful distinct roots based on their cumulative durations. This analysis will determine how many tracks can be retained when requiring at least four harmonic roots. No tracks are removed at this stage.

In [8]:
import re
import pandas as pd
from collections import defaultdict, Counter


# -----------------------------------
# 1. Convert chord roots to pitch classes
# -----------------------------------

NATURAL_PC = {
    "C": 0, "D": 2, "E": 4,
    "F": 5, "G": 7, "A": 9, "B": 11
}

def root_to_pc(root):

    if not isinstance(root, str):
        return None

    root = (
        root.strip()
        .replace("♯", "#")
        .replace("♭", "b")
    )

    match = re.fullmatch(
        r"([A-Ga-g])([#b]*)",
        root
    )

    if not match:
        return None

    note = match.group(1).upper()
    accidentals = match.group(2)

    offset = (
        accidentals.count("#")
        - accidentals.count("b")
    )

    return (NATURAL_PC[note] + offset) % 12


# -----------------------------------
# 2. Analyze all 1,603 tracks
# -----------------------------------

MIN_ROOT_DURATION = 2.0
MIN_ROOT_FRACTION = 0.01

root_records = []
unknown_roots = Counter()

for track_dir in sorted(all_tracks):

    data = read_json(
        track_dir / "chords/submission.json"
    )

    root_durations = defaultdict(float)
    valid_events = 0

    for event in data.get("events", []):

        root = event.get("root")

        if root is None:
            continue

        pc = root_to_pc(root)

        if pc is None:
            unknown_roots[str(root)] += 1
            continue

        duration = max(
            0.0,
            float(event["end_sec"])
            - float(event["start_sec"])
        )

        if duration <= 0:
            continue

        root_durations[pc] += duration
        valid_events += 1

    total_chord_duration = sum(
        root_durations.values()
    )

    # Meaningful root threshold:
    # At least 2 seconds AND at least
    # 1% of the chord-annotated duration.

    threshold = max(
        MIN_ROOT_DURATION,
        MIN_ROOT_FRACTION * total_chord_duration
    )

    meaningful_roots = {
        root: duration
        for root, duration in root_durations.items()
        if duration >= threshold
    }

    root_records.append({
        "track_id": track_dir.name,
        "unique_roots": len(root_durations),
        "meaningful_roots": len(meaningful_roots),
        "chord_events": valid_events,
        "chord_duration_sec": total_chord_duration
    })


root_df = pd.DataFrame(root_records)

assert len(root_df) == len(all_tracks)


# -----------------------------------
# 3. Distribution of unique roots
# -----------------------------------

def count_categories(series):

    return {
        "0 roots": (series == 0).sum(),
        "1 root": (series == 1).sum(),
        "2 roots": (series == 2).sum(),
        "3 roots": (series == 3).sum(),
        "4+ roots": (series >= 4).sum()
    }


raw_counts = count_categories(
    root_df["unique_roots"]
)

meaningful_counts = count_categories(
    root_df["meaningful_roots"]
)

distribution = pd.DataFrame({
    "Root category": list(raw_counts.keys()),
    "Raw root count": list(raw_counts.values()),
    "Meaningful root count": list(
        meaningful_counts.values()
    )
})


# -----------------------------------
# 4. Display results
# -----------------------------------

print("CHORD ROOT DIVERSITY ANALYSIS")
print("--------------------------------")

print("Total tracks:", len(root_df))

display(distribution)


# -----------------------------------
# 5. Retention with 4+ roots
# -----------------------------------

total = len(root_df)

raw_eligible = (
    root_df["unique_roots"] >= 4
).sum()

meaningful_eligible = (
    root_df["meaningful_roots"] >= 4
).sum()

print("\nDATASET RETENTION")
print("--------------------------------")

print(
    f"4+ raw roots: "
    f"{raw_eligible} tracks "
    f"({100 * raw_eligible / total:.2f}%)"
)

print(
    f"4+ meaningful roots: "
    f"{meaningful_eligible} tracks "
    f"({100 * meaningful_eligible / total:.2f}%)"
)


# -----------------------------------
# 6. Root diversity statistics
# -----------------------------------

print("\nROOT DIVERSITY STATISTICS")
print("--------------------------------")

display(
    root_df[
        ["unique_roots", "meaningful_roots"]
    ].describe().round(2)
)


# -----------------------------------
# 7. Validate parsing
# -----------------------------------

print("\nUNRECOGNIZED ROOT LABELS")
print("--------------------------------")

print(dict(unknown_roots))

print("\nAnalysis completed!")

CHORD ROOT DIVERSITY ANALYSIS
--------------------------------
Total tracks: 1603


,Root category,Raw root count,Meaningful root count
0,0 roots,33,33
1,1 root,59,77
2,2 roots,116,139
3,3 roots,188,238
4,4+ roots,1207,1116



DATASET RETENTION
--------------------------------
4+ raw roots: 1207 tracks (75.30%)
4+ meaningful roots: 1116 tracks (69.62%)

ROOT DIVERSITY STATISTICS
--------------------------------


,unique_roots,meaningful_roots
count,1603.00,1603.00
mean,4.92,4.54
std,2.24,2.09
min,0.00,0.00
25%,4.00,3.00
50%,5.00,4.00
75%,6.00,6.00
max,12.00,12.00



UNRECOGNIZED ROOT LABELS
--------------------------------
{}

Analysis completed!


### Chord Quality Analysis for Key Estimation
Objective: Examine the absolute chord-quality distribution before implementing an independent chord-informed key estimator.
The proposed key estimation method will first identify likely diatonic pitch collections using chord roots, qualities, and durations. It will then evaluate possible major and minor tonal centers using harmonic context.
Songs with insufficient evidence to distinguish relative major and minor keys will be marked as tonally ambiguous rather than assigned an arbitrary mode.
The analysis uses chords/submission.json and does not depend on previously estimated keys or Roman numeral labels.

In [9]:
from collections import Counter, defaultdict
import pandas as pd

# -----------------------------------
# 1. Select candidate tracks
# -----------------------------------

candidate_tracks = set(
    root_df.loc[
        root_df["meaningful_roots"] >= 4,
        "track_id"
    ]
)

print("Candidate tracks:", len(candidate_tracks))


# -----------------------------------
# 2. Analyze chord qualities
# -----------------------------------

quality_events = Counter()
quality_durations = defaultdict(float)

unrecognized = Counter()

for track_dir in sorted(all_tracks):

    if track_dir.name not in candidate_tracks:
        continue

    data = read_json(
        track_dir / "chords/submission.json"
    )

    for event in data.get("events", []):

        root = event.get("root")
        quality = event.get("quality")

        if root is None:
            continue

        if quality is None:
            unrecognized[
                event.get("chord_label", "UNKNOWN")
            ] += 1
            continue

        duration = max(
            0.0,
            float(event["end_sec"])
            - float(event["start_sec"])
        )

        quality_events[quality] += 1
        quality_durations[quality] += duration


# -----------------------------------
# 3. Display chord-quality vocabulary
# -----------------------------------

quality_df = pd.DataFrame([
    {
        "Quality": quality,
        "Events": count,
        "Duration (hours)": (
            quality_durations[quality] / 3600
        )
    }

    for quality, count in quality_events.items()
])

quality_df = quality_df.sort_values(
    "Duration (hours)",
    ascending=False
).reset_index(drop=True)

quality_df["Duration (%)"] = (
    quality_df["Duration (hours)"]
    / quality_df["Duration (hours)"].sum()
    * 100
)

print("\nCHORD QUALITY DISTRIBUTION")
print("--------------------------------")

display(quality_df.round(2))


# -----------------------------------
# 4. Inspect selected song progressions
# -----------------------------------

INSPECT_TRACKS = [
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652"
]

print("\nSAMPLE CHORD PROGRESSIONS")
print("--------------------------------")

for track_id in INSPECT_TRACKS:

    folder = track_lookup[track_id]

    data = read_json(
        folder / "chords/submission.json"
    )

    events = [
        e for e in data["events"]
        if e.get("root") is not None
    ]

    # Remove consecutive identical chords.
    progression = []

    for event in events:

        label = event["chord_label"]

        if not progression or progression[-1] != label:
            progression.append(label)

    print(f"\n{track_id}")

    print(
        " → ".join(progression[:25])
    )


print("\nUNRECOGNIZED CHORD QUALITIES")
print("--------------------------------")

print(dict(unrecognized))

print("\nAnalysis completed!")

Candidate tracks: 1116

CHORD QUALITY DISTRIBUTION
--------------------------------


,Quality,Events,Duration (hours),Duration (%)
0,maj,54329,36.76,53.80
1,min,27131,23.26,34.04
2,min7,4735,3.93,5.75
3,7,3608,2.50,3.66
4,maj7,1842,1.41,2.06
5,sus4,214,0.14,0.20
6,dim,226,0.12,0.18
7,hdim7,173,0.08,0.12
8,sus4(b7),61,0.05,0.07
9,min9,42,0.04,0.05



SAMPLE CHORD PROGRESSIONS
--------------------------------

track_0603758
E:maj → G:maj → A:maj → C:maj → D:maj → E:min → G:maj → A:min → C:maj → D:maj → E:maj → G:maj → E:min → C:maj → E:min → G:maj → A:maj → C:maj → D:maj → E:min → G:maj → A:maj → D:maj/2 → C:maj → D:maj

track_0854720
A:maj → F#:min → A:maj → F#:min → A:maj → F#:min → A:maj → F#:min → B:min → D:maj → A:maj → F#:min → B:min → D:maj → A:maj → F#:min → A:maj → F#:min → A:maj → F#:min → B:min → D:maj → A:maj → F#:min → B:min

track_1049878
E:maj/3 → D:maj/2 → A:min → E:maj/3 → D:maj → A:min → B:min → B:7 → B:min7 → B:7 → G:maj → D:maj → A:maj → B:min → B:7 → G:maj → D:maj → A:min → F:maj → C:maj → G:maj → B:min7 → B:7 → G:maj → D:maj

track_1096796
D:maj → C:maj → G:maj → C:maj → G:maj/3 → C:maj → G:maj → C:maj → G:maj/3 → C:maj → D:maj → C:maj → E:min/5 → C:maj → D:maj → C:maj → E:min/5 → C:maj → D:maj → C:maj → E:min/5 → D:maj → G:maj/5 → D:maj → G:maj/5

track_1129652
E:min → C:maj → G:maj → E:min → C:maj → E:min → 

### Key Estimation from Absolute Chord Information
Objective: Develop the first stage of an independent chord-based key estimation method using absolute chord roots, qualities, and durations.
Each chord is converted into its constituent pitch classes. The algorithm evaluates how well these pitch classes fit each of the 12 possible diatonic major-scale collections. Since relative major and natural minor share the same pitch collection, they are evaluated together initially.
Additional harmonic evidence is collected to support later tonal-center identification, including tonic-root duration and dominant-to-tonic root movements.
The algorithm does not use previously estimated keys or Roman numeral annotations. At this stage, results are exploratory and do not represent validated final key predictions.

In [10]:
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Chord-quality pitch intervals
# -----------------------------------

CHORD_INTERVALS = {
    "maj": (0, 4, 7),
    "min": (0, 3, 7),
    "7": (0, 4, 7, 10),
    "maj7": (0, 4, 7, 11),
    "min7": (0, 3, 7, 10),
    "dim": (0, 3, 6),
    "dim7": (0, 3, 6, 9),
    "hdim7": (0, 3, 6, 10),
    "aug": (0, 4, 8),
    "sus2": (0, 2, 7),
    "sus4": (0, 5, 7),
    "sus4(b7)": (0, 5, 7, 10),
    "9": (0, 4, 7, 10, 2),
    "min9": (0, 3, 7, 10, 2)
}

MAJOR_SCALE = (0, 2, 4, 5, 7, 9, 11)

PC_NAMES = [
    "C", "C#", "D", "Eb", "E", "F",
    "F#", "G", "Ab", "A", "Bb", "B"
]


# -----------------------------------
# 2. Extract chord information
# -----------------------------------

def extract_chord_evidence(events):

    chords = []

    for event in events:

        root_pc = root_to_pc(
            event.get("root")
        )

        quality = event.get("quality")

        if (
            root_pc is None
            or quality not in CHORD_INTERVALS
        ):
            continue

        duration = max(
            0.0,
            float(event["end_sec"])
            - float(event["start_sec"])
        )

        if duration <= 0:
            continue

        # Limit the influence of extremely
        # long individual chord segments.

        weight = min(duration, 12.0)

        tones = {
            (root_pc + interval) % 12
            for interval in CHORD_INTERVALS[quality]
        }

        chords.append({
            "root": root_pc,
            "tones": tones,
            "weight": weight,
            "start": float(event["start_sec"]),
            "end": float(event["end_sec"])
        })

    return chords


# -----------------------------------
# 3. Measure diatonic compatibility
# -----------------------------------

def diatonic_compatibility(chords, tonic_pc):

    scale = {
        (tonic_pc + interval) % 12
        for interval in MAJOR_SCALE
    }

    total = 0.0
    matched = 0.0

    for chord in chords:

        weight = chord["weight"]

        # Fraction of chord tones that
        # belong to the candidate scale.

        fraction = (
            len(chord["tones"] & scale)
            / len(chord["tones"])
        )

        matched += weight * fraction
        total += weight

    return matched / total if total else 0.0


# -----------------------------------
# 4. Measure tonic-root emphasis
# -----------------------------------

def tonic_root_share(chords, tonic_pc):

    total = sum(
        chord["weight"]
        for chord in chords
    )

    tonic_total = sum(
        chord["weight"]
        for chord in chords
        if chord["root"] == tonic_pc
    )

    return tonic_total / total if total else 0.0


# -----------------------------------
# 5. Count dominant-root resolutions
# -----------------------------------

def dominant_arrivals(chords, tonic_pc):

    dominant_pc = (tonic_pc + 7) % 12

    count = 0

    for previous, current in zip(
        chords,
        chords[1:]
    ):

        # Avoid counting movements
        # across substantial gaps.

        gap = current["start"] - previous["end"]

        if gap > 0.5:
            continue

        if (
            previous["root"] == dominant_pc
            and current["root"] == tonic_pc
        ):
            count += 1

    return count


# -----------------------------------
# 6. Analyze all 12 pitch collections
# -----------------------------------

def analyze_chord_keys(track_id):

    folder = track_lookup[track_id]

    data = read_json(
        folder / "chords/submission.json"
    )

    chords = extract_chord_evidence(
        data["events"]
    )

    rows = []

    for major_pc in range(12):

        # Relative minor is 9 semitones
        # above the major tonic.

        minor_pc = (major_pc + 9) % 12

        compatibility = (
            diatonic_compatibility(
                chords,
                major_pc
            )
        )

        rows.append({

            "Pitch collection":
                f"{PC_NAMES[major_pc]} major / "
                f"{PC_NAMES[minor_pc]} minor",

            "Scale fit (%)":
                round(compatibility * 100, 2),

            "Major tonic share (%)":
                round(
                    tonic_root_share(
                        chords, major_pc
                    ) * 100, 2
                ),

            "Minor tonic share (%)":
                round(
                    tonic_root_share(
                        chords, minor_pc
                    ) * 100, 2
                ),

            "Major dominant arrivals":
                dominant_arrivals(
                    chords, major_pc
                ),

            "Minor dominant arrivals":
                dominant_arrivals(
                    chords, minor_pc
                )
        })

    return (
        pd.DataFrame(rows)
        .sort_values(
            "Scale fit (%)",
            ascending=False
        )
        .reset_index(drop=True)
    )


# -----------------------------------
# 7. Test our manually inspected songs
# -----------------------------------

TEST_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652",
    "track_1106439",
    "track_0785281"
]

for track_id in TEST_TRACKS:

    print("\n" + "=" * 70)
    print("TRACK:", track_id)
    print("=" * 70)

    results = analyze_chord_keys(
        track_id
    )

    display(results.head(3))


print("\nIndependent chord analysis completed!")


TRACK: track_0571479


,Pitch collection,Scale fit (%),Major tonic share (%),Minor tonic share (%),Major dominant arrivals,Minor dominant arrivals
0,A major / F# minor,100.00,0.00,42.03,0,0
1,E major / C# minor,90.99,30.78,0.00,5,0
2,D major / B minor,89.74,0.00,27.19,0,20



TRACK: track_0603758


,Pitch collection,Scale fit (%),Major tonic share (%),Minor tonic share (%),Major dominant arrivals,Minor dominant arrivals
0,G major / E minor,90.73,20.46,45.34,0,0
1,D major / B minor,90.26,7.20,0.00,2,0
2,C major / A minor,88.33,11.42,15.58,0,5



TRACK: track_0854720


...


TRACK: track_1106439


,Pitch collection,Scale fit (%),Major tonic share (%),Minor tonic share (%),Major dominant arrivals,Minor dominant arrivals
0,C major / A minor,98.89,6.16,36.22,0,20
1,G major / E minor,94.74,10.58,27.56,0,0
2,F major / D minor,86.63,4.86,14.62,0,18



TRACK: track_0785281


,Pitch collection,Scale fit (%),Major tonic share (%),Minor tonic share (%),Major dominant arrivals,Minor dominant arrivals
0,A major / F# minor,81.60,18.33,12.06,0,0
1,D major / B minor,77.62,33.61,0.00,1,0
2,G major / E minor,70.32,9.62,0.00,4,0



Independent chord analysis completed!


### Chord-Based Tonal Center Estimation
Objective: Develop an independent chord-based key estimation method that evaluates all 24 major and minor keys.
The algorithm combines diatonic compatibility, tonic-root emphasis, chord-quality consistency, dominant-to-tonic resolutions, and final-chord information.
Unlike the previous pitch-collection analysis, this method distinguishes relative major and minor keys. Strong dominant-function resolutions are distinguished from weaker minor-dominant movements.
The initial scoring weights are experimental. The resulting rankings will be compared with manually inspected songs before selecting a final estimation method. Existing key annotations are not used.

In [11]:
import pandas as pd

# -----------------------------------
# 1. Experimental scoring weights
# -----------------------------------

KEY_WEIGHTS = {
    "scale": 0.55,
    "tonic": 0.30,
    "cadence": 0.10,
    "ending": 0.05
}

MAJOR_TONIC_QUALITIES = {"maj", "maj7"}
MINOR_TONIC_QUALITIES = {"min", "min7", "min9"}

STRONG_DOMINANT_QUALITIES = {"maj", "7", "9"}


# -----------------------------------
# 2. Prepare absolute chord events
# -----------------------------------

def prepare_key_chords(events):

    chords = []

    for event in events:

        root = root_to_pc(event.get("root"))
        quality = event.get("quality")

        if root is None or quality not in CHORD_INTERVALS:
            continue

        start = float(event["start_sec"])
        end = float(event["end_sec"])

        if end <= start:
            continue

        chords.append({
            "root": root,
            "quality": quality,
            "start": start,
            "end": end,
            "weight": min(end - start, 12.0),
            "tones": {
                (root + interval) % 12
                for interval in CHORD_INTERVALS[quality]
            }
        })

    return sorted(chords, key=lambda x: x["start"])


# -----------------------------------
# 3. Calculate tonic emphasis
# -----------------------------------

def tonic_evidence(chords, tonic_pc, mode):

    total = sum(c["weight"] for c in chords)

    if total == 0:
        return 0.0, 0.0

    expected = (
        MAJOR_TONIC_QUALITIES
        if mode == "major"
        else MINOR_TONIC_QUALITIES
    )

    root_weight = sum(
        c["weight"] for c in chords
        if c["root"] == tonic_pc
    )

    matching_weight = sum(
        c["weight"] for c in chords
        if c["root"] == tonic_pc
        and c["quality"] in expected
    )

    root_share = root_weight / total
    quality_share = matching_weight / total

    # Retain partial credit for tonic-root
    # evidence when chord quality is uncertain.

    tonic_score = (
        0.5 * root_share
        + 0.5 * quality_share
    )

    return tonic_score, root_share


# -----------------------------------
# 4. Detect dominant-to-tonic movements
# -----------------------------------

def cadence_evidence(chords, tonic_pc, mode):

    dominant_pc = (tonic_pc + 7) % 12

    expected = (
        MAJOR_TONIC_QUALITIES
        if mode == "major"
        else MINOR_TONIC_QUALITIES
    )

    strong = 0
    weak = 0
    transitions = 0

    for previous, current in zip(chords, chords[1:]):

        gap = current["start"] - previous["end"]

        if gap > 0.5 or gap < -0.5:
            continue

        if previous["root"] == current["root"]:
            continue

        transitions += 1

        if (
            previous["root"] != dominant_pc
            or current["root"] != tonic_pc
            or current["quality"] not in expected
        ):
            continue

        if previous["quality"] in STRONG_DOMINANT_QUALITIES:
            strong += 1

        elif previous["quality"] in {"min", "min7", "min9"}:
            weak += 1

    # Normalized by the number of chord changes.
    # Weak minor-dominant movements receive
    # less weight than major/dominant sevenths.

    cadence_score = min(
        1.0,
        4.0 * (strong + 0.25 * weak)
        / max(1, transitions)
    )

    return cadence_score, strong, weak


# -----------------------------------
# 5. Calculate ending evidence
# -----------------------------------

def ending_evidence(chords, tonic_pc, mode, duration_sec):

    if not chords:
        return 0.0

    last = chords[-1]

    expected = (
        MAJOR_TONIC_QUALITIES
        if mode == "major"
        else MINOR_TONIC_QUALITIES
    )

    # Do not treat a chord far from the
    # recording's end as a final cadence.

    if last["end"] < duration_sec - 10:
        return 0.0

    return float(
        last["root"] == tonic_pc
        and last["quality"] in expected
    )


# -----------------------------------
# 6. Rank all 24 candidate keys
# -----------------------------------

def estimate_key_from_chords(track_id):

    folder = track_lookup[track_id]

    data = read_json(
        folder / "chords/submission.json"
    )

    chords = prepare_key_chords(
        data["events"]
    )

    rows = []

    for major_pc in range(12):

        relative_minor_pc = (major_pc + 9) % 12

        scale_fit = diatonic_compatibility(
            chords, major_pc
        )

        for mode, tonic_pc in [
            ("major", major_pc),
            ("minor", relative_minor_pc)
        ]:

            tonic_score, root_share = tonic_evidence(
                chords, tonic_pc, mode
            )

            cadence_score, strong, weak = cadence_evidence(
                chords, tonic_pc, mode
            )

            ending_score = ending_evidence(
                chords,
                tonic_pc,
                mode,
                float(data["duration_sec"])
            )

            score = (
                KEY_WEIGHTS["scale"] * scale_fit
                + KEY_WEIGHTS["tonic"] * tonic_score
                + KEY_WEIGHTS["cadence"] * cadence_score
                + KEY_WEIGHTS["ending"] * ending_score
            )

            rows.append({
                "Key": f"{PC_NAMES[tonic_pc]} {mode}",
                "Collection": PC_NAMES[major_pc],
                "Score": score,
                "Scale fit": scale_fit,
                "Tonic root share": root_share,
                "Tonic evidence": tonic_score,
                "Cadence evidence": cadence_score,
                "Strong V-I": strong,
                "Weak v-i": weak,
                "Ending evidence": ending_score
            })

    return (
        pd.DataFrame(rows)
        .sort_values("Score", ascending=False)
        .reset_index(drop=True)
    )


# -----------------------------------
# 7. Evaluate manually inspected songs
# -----------------------------------

TEST_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652",
    "track_1106439",
    "track_0785281"
]

for track_id in TEST_TRACKS:

    results = estimate_key_from_chords(track_id)

    print("\n" + "=" * 65)
    print("TRACK:", track_id)
    print("=" * 65)

    display(
        results.head(5).round(3)
    )

    best = results.iloc[0]

    # Compare the relative major/minor alternative
    # sharing the same pitch collection.

    relative = results[
        (results["Collection"] == best["Collection"])
        & (results["Key"] != best["Key"])
    ].iloc[0]

    print("Top candidate:", best["Key"])
    print("Relative alternative:", relative["Key"])

    print(
        "Relative-key score difference:",
        round(best["Score"] - relative["Score"], 4)
    )


print("\nTonal-center analysis completed!")


TRACK: track_0571479


,Key,Collection,Score,Scale fit,Tonic root share,Tonic evidence,Cadence evidence,Strong V-I,Weak v-i,Ending evidence
0,F# minor,A,0.676,1.000,0.420,0.420,0.000,0,0,0.0
1,B minor,D,0.613,0.897,0.272,0.272,0.377,0,20,0.0
2,E major,E,0.602,0.910,0.308,0.308,0.094,0,5,0.0
3,A major,A,0.550,1.000,0.000,0.000,0.000,0,0,0.0
4,C# minor,E,0.500,0.910,0.000,0.000,0.000,0,0,0.0


Top candidate: F# minor
Relative alternative: A major
Relative-key score difference: 0.1261

TRACK: track_0603758


,Key,Collection,Score,Scale fit,Tonic root share,Tonic evidence,Cadence evidence,Strong V-I,Weak v-i,Ending evidence
0,E minor,G,0.612,0.907,0.453,0.378,0.000,0,0,0.0
1,C major,C,0.570,0.883,0.114,0.114,0.000,0,0,1.0
2,G major,G,0.560,0.907,0.205,0.205,0.000,0,0,0.0
3,D major,D,0.530,0.903,0.072,0.072,0.123,2,0,0.0
4,A minor,C,0.516,0.883,0.156,0.092,0.031,0,2,0.0


Top candidate: E minor
Relative alternative: G major
Relative-key score difference: 0.0521

TRACK: track_0854720


...

Top candidate: E minor
Relative alternative: G major
Relative-key score difference: 0.1972

TRACK: track_1106439


,Key,Collection,Score,Scale fit,Tonic root share,Tonic evidence,Cadence evidence,Strong V-I,Weak v-i,Ending evidence
0,A minor,C,0.723,0.989,0.362,0.362,0.208,0,20,1.0
1,E minor,G,0.604,0.947,0.276,0.276,0.000,0,0,0.0
2,C major,C,0.562,0.989,0.062,0.062,0.000,0,0,0.0
3,G major,G,0.553,0.947,0.106,0.106,0.000,0,0,0.0
4,D minor,F,0.530,0.866,0.146,0.130,0.146,0,14,0.0


Top candidate: A minor
Relative alternative: C major
Relative-key score difference: 0.161

TRACK: track_0785281


,Key,Collection,Score,Scale fit,Tonic root share,Tonic evidence,Cadence evidence,Strong V-I,Weak v-i,Ending evidence
0,D major,D,0.525,0.776,0.336,0.327,0.000,0,0,0.0
1,A major,A,0.502,0.816,0.183,0.178,0.000,0,0,0.0
2,F# minor,A,0.482,0.816,0.121,0.111,0.000,0,0,0.0
3,G major,G,0.451,0.703,0.096,0.096,0.356,4,0,0.0
4,B minor,D,0.427,0.776,0.000,0.000,0.000,0,0,0.0


Top candidate: D major
Relative alternative: B minor
Relative-key score difference: 0.0982

Tonal-center analysis completed!


### Improved Chord-Based Key Estimation
Objective: Improve the independent chord-based key estimator by incorporating musical structure and reducing biases identified in the initial experiments.
The revised method evaluates diatonic compatibility, tonic-chord duration, dominant-to-tonic and subdominant-to-tonic movements, and tonic emphasis on bar downbeats.
Repeated chord transitions receive diminishing influence, preventing highly repetitive progressions from dominating the key estimation. The final-chord heuristic is temporarily removed because song endings do not always provide reliable evidence of the tonal center.
The method ranks all 24 major/minor keys. Score differences are examined to identify ambiguous predictions. All scoring weights remain experimental and require validation against manually verified songs.

In [12]:
from collections import Counter
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Experimental scoring weights
# -----------------------------------

KEY_WEIGHTS_V2 = {
    "scale": 0.50,
    "tonic": 0.23,
    "resolution": 0.17,
    "downbeat": 0.10
}


# -----------------------------------
# 2. Harmonic resolution evidence
# -----------------------------------

def harmonic_resolution_score(chords, tonic_pc, mode):

    dominant = (tonic_pc + 7) % 12
    subdominant = (tonic_pc + 5) % 12

    tonic_qualities = (
        MAJOR_TONIC_QUALITIES
        if mode == "major"
        else MINOR_TONIC_QUALITIES
    )

    transitions = Counter()

    for prev, curr in zip(chords, chords[1:]):

        gap = curr["start"] - prev["end"]

        if abs(gap) > 0.5:
            continue

        if prev["root"] == curr["root"]:
            continue

        transition = (
            prev["root"],
            prev["quality"],
            curr["root"],
            curr["quality"]
        )

        transitions[transition] += 1

    if not transitions:
        return 0.0

    numerator = 0.0
    denominator = 0.0

    for transition, count in transitions.items():

        prev_root, prev_quality, curr_root, curr_quality = transition

        # Diminishing returns for repeated transitions
        frequency_weight = np.sqrt(count)

        denominator += frequency_weight

        if curr_root != tonic_pc:
            continue

        # Stronger evidence when the destination
        # chord has the expected tonic quality.
        destination_strength = (
            1.0 if curr_quality in tonic_qualities
            else 0.35
        )

        strength = 0.0

        # Dominant -> tonic
        if prev_root == dominant:

            if prev_quality in {"7", "9"}:
                strength = 1.0

            elif prev_quality == "maj":
                strength = 0.65

            elif prev_quality in {"min", "min7"}:
                strength = 0.10

        # Subdominant -> tonic (plagal movement)
        elif prev_root == subdominant:

            if prev_quality in {
                "maj", "maj7", "min", "min7"
            }:
                strength = 0.35

        numerator += (
            frequency_weight
            * strength
            * destination_strength
        )

    return min(
        1.0,
        4.0 * numerator / denominator
    )


# -----------------------------------
# 3. Tonic emphasis on bar downbeats
# -----------------------------------

def downbeat_tonic_score(chords, beats, tonic_pc, mode):

    tonic_qualities = (
        MAJOR_TONIC_QUALITIES
        if mode == "major"
        else MINOR_TONIC_QUALITIES
    )

    downbeats = sorted(
        float(b["time_sec"])
        for b in beats
        if b.get("is_downbeat")
    )

    if not downbeats or not chords:
        return 0.0

    total = 0
    score = 0.0
    chord_idx = 0

    for time in downbeats:

        # Find the chord active at this downbeat.
        while (
            chord_idx < len(chords)
            and chords[chord_idx]["end"] <= time
        ):
            chord_idx += 1

        if chord_idx >= len(chords):
            break

        chord = chords[chord_idx]

        if not (
            chord["start"] <= time < chord["end"]
        ):
            continue

        total += 1

        if chord["root"] == tonic_pc:

            score += (
                1.0
                if chord["quality"] in tonic_qualities
                else 0.4
            )

    return score / total if total else 0.0


# -----------------------------------
# 4. Improved independent estimator
# -----------------------------------

def estimate_key_v2(track_id):

    folder = track_lookup[track_id]

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    beat_data = read_json(
        folder / "beats/beats.json"
    )

    chords = prepare_key_chords(
        chord_data["events"]
    )

    beats = beat_data.get("beats", [])

    rows = []

    for major_pc in range(12):

        minor_pc = (major_pc + 9) % 12

        scale_fit = diatonic_compatibility(
            chords,
            major_pc
        )

        for mode, tonic_pc in [
            ("major", major_pc),
            ("minor", minor_pc)
        ]:

            tonic_score, root_share = tonic_evidence(
                chords,
                tonic_pc,
                mode
            )

            resolution = harmonic_resolution_score(
                chords,
                tonic_pc,
                mode
            )

            downbeat = downbeat_tonic_score(
                chords,
                beats,
                tonic_pc,
                mode
            )

            total_score = (
                KEY_WEIGHTS_V2["scale"] * scale_fit
                + KEY_WEIGHTS_V2["tonic"] * tonic_score
                + KEY_WEIGHTS_V2["resolution"] * resolution
                + KEY_WEIGHTS_V2["downbeat"] * downbeat
            )

            rows.append({
                "Key": f"{PC_NAMES[tonic_pc]} {mode}",
                "Collection": PC_NAMES[major_pc],
                "Score": total_score,
                "Scale fit": scale_fit,
                "Tonic": tonic_score,
                "Resolution": resolution,
                "Downbeat": downbeat
            })

    return (
        pd.DataFrame(rows)
        .sort_values(
            "Score",
            ascending=False
        )
        .reset_index(drop=True)
    )


# -----------------------------------
# 5. Compare V1 and V2 predictions
# -----------------------------------

TEST_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652",
    "track_1106439",
    "track_0785281"
]

comparison_rows = []

for track_id in TEST_TRACKS:

    v1 = estimate_key_from_chords(
        track_id
    )

    v2 = estimate_key_v2(
        track_id
    )

    # Difference between best and second-best
    # candidate across ALL 24 keys.
    gap = (
        float(v2.iloc[0]["Score"])
        - float(v2.iloc[1]["Score"])
    )

    comparison_rows.append({
        "Track": track_id,
        "V1 key": v1.iloc[0]["Key"],
        "V2 key": v2.iloc[0]["Key"],
        "V2 second choice": v2.iloc[1]["Key"],
        "V2 score gap": round(gap, 4),
        "Changed": (
            v1.iloc[0]["Key"]
            != v2.iloc[0]["Key"]
        )
    })


print("KEY ESTIMATION: V1 VS V2")
print("--------------------------------")

display(
    pd.DataFrame(comparison_rows)
)


# -----------------------------------
# 6. Inspect detailed V2 rankings
# -----------------------------------

IMPORTANT_TRACKS = [
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_0785281"
]

for track_id in IMPORTANT_TRACKS:

    print("\n" + "=" * 65)
    print("TRACK:", track_id)
    print("=" * 65)

    display(
        estimate_key_v2(track_id)
        .head(5)
        .round(3)
    )


print("\nV2 analysis completed!")

KEY ESTIMATION: V1 VS V2
--------------------------------


,Track,V1 key,V2 key,V2 second choice,V2 score gap,Changed
0,track_0571479,F# minor,F# minor,B minor,0.1046,False
1,track_0603758,E minor,E minor,G major,0.0581,False
2,track_0854720,F# minor,A major,F# minor,0.0335,True
3,track_1049878,G major,G major,D major,0.0143,False
4,track_1096796,C major,C major,G major,0.0095,False
5,track_1129652,E minor,E minor,C major,0.0601,False
6,track_1106439,A minor,A minor,E minor,0.0352,False
7,track_0785281,D major,D major,A major,0.0297,False



TRACK: track_0854720


,Key,Collection,Score,Scale fit,Tonic,Resolution,Downbeat
0,A major,A,0.664,1.000,0.372,0.252,0.354
1,F# minor,A,0.630,1.000,0.391,0.000,0.404
2,D major,D,0.546,1.000,0.137,0.000,0.141
3,B minor,D,0.544,1.000,0.100,0.061,0.101
4,E major,E,0.460,0.921,0.000,0.000,0.000



TRACK: track_1049878


...


TRACK: track_1096796


,Key,Collection,Score,Scale fit,Tonic,Resolution,Downbeat
0,C major,C,0.614,0.897,0.361,0.328,0.263
1,G major,G,0.604,1.000,0.148,0.350,0.105
2,D major,D,0.571,0.880,0.309,0.059,0.500
3,E minor,G,0.555,1.000,0.182,0.000,0.132
4,D minor,F,0.452,0.787,0.154,0.021,0.200



TRACK: track_0785281


,Key,Collection,Score,Scale fit,Tonic,Resolution,Downbeat
0,D major,D,0.513,0.776,0.327,0.105,0.316
1,A major,A,0.483,0.816,0.178,0.079,0.207
2,F# minor,A,0.445,0.816,0.111,0.000,0.109
3,G major,G,0.410,0.703,0.096,0.147,0.109
4,B minor,D,0.388,0.776,0.000,0.000,0.000



V2 analysis completed!


In [13]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1. Run V2 across all candidate tracks
# -----------------------------------

key_v2_records = []
failed_tracks = []

for track_id in sorted(candidate_tracks):

    try:
        results = estimate_key_v2(track_id)

        best = results.iloc[0]
        second = results.iloc[1]

        gap = float(best["Score"] - second["Score"])

        key_v2_records.append({
            "track_id": track_id,
            "best_key": best["Key"],
            "second_key": second["Key"],
            "best_score": float(best["Score"]),
            "second_score": float(second["Score"]),
            "score_gap": gap,
            "scale_fit": float(best["Scale fit"]),
            "same_collection": (
                best["Collection"]
                == second["Collection"]
            )
        })

    except Exception as error:
        failed_tracks.append((track_id, str(error)))

key_v2_df = pd.DataFrame(key_v2_records)

print("DATASET-WIDE KEY ESTIMATION")
print("--------------------------------")
print("Candidate tracks:", len(candidate_tracks))
print("Successfully analyzed:", len(key_v2_df))
print("Failed:", len(failed_tracks))

if failed_tracks:
    print("First errors:", failed_tracks[:5])


# -----------------------------------
# 2. Score gap distribution
# -----------------------------------

print("\nKEY SCORE GAP DISTRIBUTION")
print("--------------------------------")

display(
    key_v2_df["score_gap"]
    .describe()
    .round(4)
)


# -----------------------------------
# 3. Examine several gap thresholds
# -----------------------------------

thresholds = [0.01, 0.02, 0.03, 0.05, 0.10]

rows = []

for threshold in thresholds:

    count = (
        key_v2_df["score_gap"] < threshold
    ).sum()

    rows.append({
        "Score gap below": threshold,
        "Tracks": count,
        "Percentage": (
            100 * count / len(key_v2_df)
        )
    })

print("\nCANDIDATE AMBIGUITY THRESHOLDS")
print("--------------------------------")

display(
    pd.DataFrame(rows).round(2)
)


# -----------------------------------
# 4. Type of disagreement
# -----------------------------------

print("\nTOP TWO KEY RELATIONSHIPS")
print("--------------------------------")

key_v2_df["ambiguity_type"] = np.where(
    key_v2_df["same_collection"],
    "Relative major/minor",
    "Different pitch collections"
)

display(
    key_v2_df["ambiguity_type"]
    .value_counts()
    .rename_axis("Relationship")
    .reset_index(name="Tracks")
)


# -----------------------------------
# 5. Diatonic compatibility
# -----------------------------------

print("\nBEST KEY SCALE FIT")
print("--------------------------------")

display(
    key_v2_df["scale_fit"]
    .describe()
    .round(3)
)

for threshold in [0.70, 0.80, 0.90, 0.95]:

    count = (
        key_v2_df["scale_fit"] < threshold
    ).sum()

    print(
        f"Scale fit below {threshold:.0%}: "
        f"{count} tracks"
    )


# -----------------------------------
# 6. Major/minor distribution
# -----------------------------------

print("\nPREDICTED MODE DISTRIBUTION")
print("--------------------------------")

key_v2_df["predicted_mode"] = (
    key_v2_df["best_key"]
    .str.split()
    .str[-1]
)

display(
    key_v2_df["predicted_mode"]
    .value_counts()
)


# -----------------------------------
# 7. Inspect ambiguous examples
# -----------------------------------

print("\nEXAMPLES WITH CLOSE KEY SCORES")
print("--------------------------------")

display(
    key_v2_df.sort_values("score_gap")
    [[
        "track_id",
        "best_key",
        "second_key",
        "score_gap",
        "scale_fit",
        "ambiguity_type"
    ]]
    .head(15)
    .round(3)
)

print("\nDataset-wide analysis completed!")

DATASET-WIDE KEY ESTIMATION
--------------------------------
Candidate tracks: 1116
Successfully analyzed: 1116
Failed: 0

KEY SCORE GAP DISTRIBUTION
--------------------------------


count    1116.0000
mean        0.0788
std         0.0642
min         0.0002
25%         0.0279
50%         0.0622
75%         0.1138
max         0.3914
Name: score_gap, dtype: float64


CANDIDATE AMBIGUITY THRESHOLDS
--------------------------------


,Score gap below,Tracks,Percentage
0,0.01,98,8.78
1,0.02,201,18.01
2,0.03,308,27.60
3,0.05,462,41.40
4,0.10,766,68.64



TOP TWO KEY RELATIONSHIPS
--------------------------------


...

Scale fit below 70%: 23 tracks
Scale fit below 80%: 90 tracks
Scale fit below 90%: 323 tracks
Scale fit below 95%: 574 tracks

PREDICTED MODE DISTRIBUTION
--------------------------------


predicted_mode
minor    574
major    542
Name: count, dtype: int64


EXAMPLES WITH CLOSE KEY SCORES
--------------------------------


,track_id,best_key,second_key,score_gap,scale_fit,ambiguity_type
681,track_1085332,E minor,A minor,0.000,0.874,Different pitch collections
199,track_0565806,C# minor,F# minor,0.000,0.981,Different pitch collections
683,track_1085352,G major,D major,0.001,0.924,Different pitch collections
149,track_0368770,E minor,G major,0.001,0.820,Relative major/minor
528,track_1051068,E minor,D major,0.001,0.863,Different pitch collections
666,track_1084672,A major,E major,0.001,0.898,Different pitch collections
784,track_1119019,A minor,E minor,0.001,0.914,Different pitch collections
693,track_1087888,A minor,E minor,0.001,0.830,Different pitch collections
689,track_1087538,A major,D major,0.001,0.865,Different pitch collections
60,track_0072838,E minor,B minor,0.001,0.965,Different pitch collections



Dataset-wide analysis completed!


### Major-Scale Normalization Strategy
Objective: Simplify musical key representation by assigning every track to a major-scale pitch collection, independently of its perceived major or minor tonal center.
Relative major and minor keys share the same diatonic pitch collection. Therefore, the proposed approach removes the need to distinguish relative major/minor tonal centers during dataset preparation.
Absolute chord roots and qualities remain unchanged. Only the reference scale used for subsequent transposition and normalization is standardized.
Three approaches are compared: independent chord-based pitch-collection estimation, the previously developed V2 tonal-center estimator, and the original pipeline's chord-based key estimate. All results are converted to their corresponding major-scale collections.
Agreement between estimators is measured, but agreement is not considered proof of correctness. Tracks with similarly ranked pitch collections are flagged for further investigation.

In [14]:
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Convert major/minor key to
#    corresponding major-scale collection
# -----------------------------------

def key_to_major_pc(tonic_pc, mode):

    if tonic_pc is None or mode not in ("major", "minor"):
        return None

    tonic_pc = int(tonic_pc)

    if mode == "minor":
        return (tonic_pc + 3) % 12

    return tonic_pc


def key_name(pc):

    if pc is None:
        return None

    return f"{PC_NAMES[int(pc)]} major"


# -----------------------------------
# 2. Prepare previous V2 predictions
# -----------------------------------

v2_lookup = {}

for _, row in key_v2_df.iterrows():

    tonic_name, mode = row["best_key"].rsplit(" ", 1)

    tonic_pc = PC_NAMES.index(tonic_name)

    v2_lookup[row["track_id"]] = key_to_major_pc(
        tonic_pc,
        mode
    )


# -----------------------------------
# 3. Estimate major collection
#    independently from absolute chords
# -----------------------------------

major_key_records = []

for track_id in sorted(candidate_tracks):

    folder = track_lookup[track_id]

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    comparison = read_json(
        folder / "key_comparison.json"
    )

    chords = prepare_key_chords(
        chord_data["events"]
    )

    # Evaluate all 12 major-scale collections.
    fits = np.array([
        diatonic_compatibility(chords, pc)
        for pc in range(12)
    ])

    ranked = np.argsort(-fits, kind="stable")

    best_pc = int(ranked[0])
    second_pc = int(ranked[1])

    best_fit = float(fits[best_pc])
    second_fit = float(fits[second_pc])

    # Existing chord-derived estimate:
    # Used ONLY for comparison.

    original_info = (
        comparison.get("chord_key") or {}
    )

    original_selected = (
        original_info.get("selected") or {}
    )

    original_pc = key_to_major_pc(
        original_selected.get("tonic_pc"),
        original_selected.get("mode")
    )

    v2_pc = v2_lookup.get(track_id)

    major_key_records.append({

        "track_id": track_id,

        "our_major_key": key_name(best_pc),

        "second_major_key": key_name(second_pc),

        "scale_fit": best_fit,

        "scale_fit_gap": best_fit - second_fit,

        "v2_major_key": key_name(v2_pc),

        "original_chord_major_key": key_name(original_pc),

        "original_confidence":
            original_selected.get("confidence"),

        "matches_original": (
            best_pc == original_pc
            if original_pc is not None
            else pd.NA
        ),

        "matches_v2": (
            best_pc == v2_pc
            if v2_pc is not None
            else pd.NA
        )
    })


major_key_df = pd.DataFrame(major_key_records)


# -----------------------------------
# 4. Overall agreement
# -----------------------------------

print("MAJOR-ONLY KEY ESTIMATION")
print("--------------------------------")

print("Tracks analyzed:", len(major_key_df))

original_available = major_key_df[
    major_key_df["original_chord_major_key"].notna()
]

v2_available = major_key_df[
    major_key_df["v2_major_key"].notna()
]

print(
    "Original chord keys available:",
    len(original_available)
)

print(
    "\nAgreement with original chord estimator:",
    f"{original_available['matches_original'].astype(bool).mean():.2%}"
    if len(original_available) else "N/A"
)

print(
    "Agreement with our V2 estimator:",
    f"{v2_available['matches_v2'].astype(bool).mean():.2%}"
    if len(v2_available) else "N/A"
)


# -----------------------------------
# 5. Scale-fit statistics
# -----------------------------------

print("\nSCALE COMPATIBILITY")
print("--------------------------------")

display(
    major_key_df[
        ["scale_fit", "scale_fit_gap"]
    ].describe().round(4)
)


# -----------------------------------
# 6. Ambiguous pitch collections
# -----------------------------------

print("\nPITCH-COLLECTION AMBIGUITY")
print("--------------------------------")

for threshold in [0.005, 0.01, 0.02, 0.05]:

    count = (
        major_key_df["scale_fit_gap"] < threshold
    ).sum()

    print(
        f"Gap below {threshold:.3f}: "
        f"{count} tracks "
        f"({100 * count / len(major_key_df):.2f}%)"
    )


# -----------------------------------
# 7. Compare our manually inspected songs
# -----------------------------------

MANUAL_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0659892",
    "track_0785281",
    "track_0854720",
    "track_0920193",
    "track_1049878",
    "track_1075195",
    "track_1096796",
    "track_1098835",
    "track_1129652",
    "track_1106439"
]

print("\nMANUALLY INSPECTED SONGS")
print("--------------------------------")

display(
    major_key_df[
        major_key_df["track_id"].isin(MANUAL_TRACKS)
    ][[
        "track_id",
        "our_major_key",
        "second_major_key",
        "scale_fit",
        "scale_fit_gap",
        "v2_major_key",
        "original_chord_major_key",
        "original_confidence"
    ]].round(3)
)


# -----------------------------------
# 8. Show estimator disagreements
# -----------------------------------

print("\nEXAMPLE DISAGREEMENTS")
print("--------------------------------")

disagreements = major_key_df[
    major_key_df["matches_original"] == False
]

display(
    disagreements[
        [
            "track_id",
            "our_major_key",
            "second_major_key",
            "scale_fit",
            "scale_fit_gap",
            "original_chord_major_key",
            "original_confidence"
        ]
    ].sample(
        n=min(15, len(disagreements)),
        random_state=42
    ).round(3)
)


print("\nMajor-only comparison completed!")

MAJOR-ONLY KEY ESTIMATION
--------------------------------
Tracks analyzed: 1116
Original chord keys available: 1116

Agreement with original chord estimator: 69.44%
Agreement with our V2 estimator: 66.85%

SCALE COMPATIBILITY
--------------------------------


,scale_fit,scale_fit_gap
count,1116.0000,1116.0000
mean,0.9401,0.0437
std,0.0671,0.0299
min,0.6414,0.0000
25%,0.9078,0.0185
50%,0.9619,0.0400
75%,0.9921,0.0649
max,1.0000,0.1557



PITCH-COLLECTION AMBIGUITY
--------------------------------
Gap below 0.005: 85 tracks (7.62%)
Gap below 0.010: 156 tracks (13.98%)
Gap below 0.020: 305 tracks (27.33%)
Gap below 0.050: 676 tracks (60.57%)

MANUALLY INSPECTED SONGS
--------------------------------


,track_id,our_major_key,second_major_key,scale_fit,scale_fit_gap,v2_major_key,original_chord_major_key,original_confidence
222,track_0603758,G major,D major,0.907,0.005,G major,G major,high
279,track_0785281,A major,D major,0.816,0.040,D major,A major,medium
299,track_0854720,D major,A major,1.000,0.000,A major,A major,medium
524,track_1049878,G major,D major,0.911,0.004,G major,D major,medium
713,track_1096796,G major,C major,1.000,0.103,C major,G major,medium
725,track_1098835,E major,B major,1.000,0.079,E major,E major,medium
735,track_1106439,C major,G major,0.989,0.041,C major,C major,medium
835,track_1129652,G major,C major,1.000,0.044,G major,G major,high



EXAMPLE DISAGREEMENTS
--------------------------------


,track_id,our_major_key,second_major_key,scale_fit,scale_fit_gap,original_chord_major_key,original_confidence
1056,track_1189646,D major,A major,0.963,0.025,A major,high
307,track_0865722,B major,F# major,0.899,0.034,F# major,high
301,track_0855373,C major,G major,0.951,0.009,G major,low
114,track_0243758,A major,E major,0.936,0.047,E major,low
333,track_0915579,Eb major,Ab major,0.843,0.009,C major,high
715,track_1096808,Eb major,Bb major,0.938,0.110,Ab major,low
555,track_1056816,C major,F major,1.000,0.097,F major,high
528,track_1051068,C major,G major,0.876,0.014,D major,low
1111,track_1400609,B major,E major,0.901,0.017,E major,high
91,track_0210969,C major,G major,0.957,0.023,G major,low



Major-only comparison completed!


### Findings from Step 3.12
The independent major-scale estimator achieved 69.44% agreement with the pipeline's original chord-based key estimator across 1,116 candidate tracks. The average best-fit diatonic compatibility was 94.01%.
However, 305 tracks exhibited a difference below 0.02 between their two highest-ranked scale collections. Manual examples demonstrated that different scales can explain the same chord collection equally well.
Therefore, scale compatibility alone is insufficient for reliable key assignment in all cases. The next experiment investigates additional harmonic evidence to distinguish competing major-scale collections without reintroducing major/minor mode classification.

### Harmonic Support for Competing Pitch Collections
Objective: Examine the harmonic evidence supporting the two highest-ranked major-scale collections.
For each candidate collection, the analysis measures the duration-weighted presence of its major tonic root and relative minor tonic root. These measurements are compared with diatonic compatibility and the original chord-based key estimate.
No final key-selection rules are applied in this experiment. The objective is to determine whether harmonic root prominence can help resolve otherwise ambiguous pitch collections.

In [16]:
import pandas as pd

# -----------------------------------
# 1. Calculate root emphasis
# -----------------------------------

def collection_root_support(chords, major_pc):

    minor_pc = (major_pc + 9) % 12

    total = sum(
        chord["weight"]
        for chord in chords
    )

    if total == 0:
        return 0.0, 0.0, 0.0

    major_share = sum(
        chord["weight"]
        for chord in chords
        if chord["root"] == major_pc
    ) / total

    minor_share = sum(
        chord["weight"]
        for chord in chords
        if chord["root"] == minor_pc
    ) / total

    return (
        major_share,
        minor_share,
        major_share + minor_share
    )


# -----------------------------------
# 2. Inspect important tracks
# -----------------------------------

TEST_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652",
    "track_1106439",
    "track_0785281"
]

for track_id in TEST_TRACKS:

    folder = track_lookup[track_id]

    data = read_json(
        folder / "chords/submission.json"
    )

    chords = prepare_key_chords(
        data["events"]
    )

    rows = []

    for major_pc in range(12):

        fit = diatonic_compatibility(
            chords,
            major_pc
        )

        major_share, minor_share, combined = (
            collection_root_support(
                chords,
                major_pc
            )
        )

        rows.append({
            "Major collection": key_name(major_pc),
            "Scale fit": fit,
            "Major root share": major_share,
            "Relative minor root share": minor_share,
            "Combined root support": combined
        })

    results = (
        pd.DataFrame(rows)
        .sort_values(
            "Scale fit",
            ascending=False,
            kind="stable"
        )
        .reset_index(drop=True)
    )

# Read the original chord-based key directly.
# This works even for tracks excluded by our diversity filter.

    comparison = read_json(
        folder / "key_comparison.json"
    )
    
    original_selected = (
        (comparison.get("chord_key") or {})
        .get("selected") or {}
    )
    
    original_pc = key_to_major_pc(
        original_selected.get("tonic_pc"),
        original_selected.get("mode")
    )
    
    original = key_name(original_pc)

    print("\n" + "=" * 65)
    print("TRACK:", track_id)
    print("Original chord estimate:", original)
    print("=" * 65)

    display(
        results.head(3).round(4)
    )


print("\nHarmonic support analysis completed!")


TRACK: track_0571479
Original chord estimate: A major


,Major collection,Scale fit,Major root share,Relative minor root share,Combined root support
0,A major,1.0000,0.0000,0.4203,0.4203
1,E major,0.9099,0.3078,0.0000,0.3078
2,D major,0.8974,0.0000,0.2719,0.2719



TRACK: track_0603758
Original chord estimate: G major


,Major collection,Scale fit,Major root share,Relative minor root share,Combined root support
0,G major,0.9073,0.2046,0.4534,0.658
1,D major,0.9026,0.0720,0.0000,0.072
2,C major,0.8833,0.1142,0.1558,0.270



TRACK: track_0854720
Original chord estimate: A major


...


TRACK: track_1106439
Original chord estimate: C major


,Major collection,Scale fit,Major root share,Relative minor root share,Combined root support
0,C major,0.9889,0.0616,0.3622,0.4238
1,G major,0.9474,0.1058,0.2756,0.3814
2,F major,0.8663,0.0486,0.1462,0.1948



TRACK: track_0785281
Original chord estimate: A major


,Major collection,Scale fit,Major root share,Relative minor root share,Combined root support
0,A major,0.8160,0.1833,0.1206,0.3039
1,D major,0.7762,0.3361,0.0000,0.3361
2,G major,0.7032,0.0962,0.0000,0.0962



Harmonic support analysis completed!


### Two-Stage Major-Scale Estimation
Objective: Develop and evaluate a two-stage major-scale estimator using absolute chord annotations.
The first stage ranks all 12 major-scale collections according to duration-weighted chord-tone compatibility. When several collections have similar compatibility, the second stage selects the collection with the greatest combined root support from its major tonic and relative minor tonic.
Different compatibility margins are evaluated to examine the sensitivity of the method. Agreement with the original chord-based estimator and predictions for manually inspected tracks are reported.
The compatibility margins are experimental and do not represent calibrated confidence thresholds. No existing key estimates are used to generate predictions.

In [17]:
import pandas as pd
import numpy as np


# -----------------------------------
# 1. Calculate evidence for all
#    12 major-scale collections
# -----------------------------------

def collection_evidence(track_id):

    folder = track_lookup[track_id]

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    chords = prepare_key_chords(
        chord_data["events"]
    )

    rows = []

    for major_pc in range(12):

        fit = diatonic_compatibility(
            chords,
            major_pc
        )

        major_share, minor_share, combined = (
            collection_root_support(
                chords,
                major_pc
            )
        )

        rows.append({
            "pc": major_pc,
            "Key": key_name(major_pc),
            "Scale fit": fit,
            "Root support": combined
        })

    return pd.DataFrame(rows)


# -----------------------------------
# 2. Two-stage selection
# -----------------------------------

def select_major_key(evidence, margin=0.02):

    best_fit = evidence["Scale fit"].max()

    # Stage 1:
    # Keep scales with similar compatibility.

    candidates = evidence[
        evidence["Scale fit"] >= best_fit - margin - 1e-10
    ].copy()

    # Stage 2:
    # Select using combined root emphasis.
    # Break remaining ties by scale fit.

    candidates = candidates.sort_values(
        ["Root support", "Scale fit", "pc"],
        ascending=[False, False, True]
    )

    return candidates.iloc[0]["Key"]


# -----------------------------------
# 3. Evaluate several margins
# -----------------------------------

MARGINS = [0.0, 0.01, 0.02, 0.03]

evaluation_rows = []

for track_id in sorted(candidate_tracks):

    evidence = collection_evidence(track_id)

    original_row = major_key_df.loc[
        major_key_df["track_id"] == track_id
    ].iloc[0]

    record = {
        "track_id": track_id,
        "Original": original_row["original_chord_major_key"]
    }

    for margin in MARGINS:

        record[f"margin_{margin}"] = (
            select_major_key(evidence, margin)
        )

    evaluation_rows.append(record)


key_margin_df = pd.DataFrame(evaluation_rows)


# -----------------------------------
# 4. Compare predictions
# -----------------------------------

print("TWO-STAGE KEY ESTIMATION")
print("--------------------------------")

summary_rows = []

baseline = key_margin_df["margin_0.0"]

for margin in MARGINS:

    predictions = key_margin_df[
        f"margin_{margin}"
    ]

    agreement = (
        predictions == key_margin_df["Original"]
    ).mean()

    changed = (
        predictions != baseline
    ).sum()

    summary_rows.append({
        "Compatibility margin": margin,
        "Agreement with original (%)":
            100 * agreement,
        "Changed from exact-fit selection":
            changed
    })

display(
    pd.DataFrame(summary_rows).round(2)
)


# -----------------------------------
# 5. Manually inspected tracks
# -----------------------------------

MANUAL_TRACKS = [
    "track_0571479",
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652",
    "track_1106439",
    "track_0785281"
]

manual_rows = []

for track_id in MANUAL_TRACKS:

    evidence = collection_evidence(track_id)

    comparison = read_json(
        track_lookup[track_id] / "key_comparison.json"
    )

    original = (
        (comparison.get("chord_key") or {})
        .get("selected") or {}
    )

    original_pc = key_to_major_pc(
        original.get("tonic_pc"),
        original.get("mode")
    )

    manual_rows.append({
        "Track": track_id,
        "Scale-only": select_major_key(
            evidence, 0.0
        ),
        "Margin 0.01": select_major_key(
            evidence, 0.01
        ),
        "Margin 0.02": select_major_key(
            evidence, 0.02
        ),
        "Margin 0.03": select_major_key(
            evidence, 0.03
        ),
        "Original chord estimate":
            key_name(original_pc)
    })


print("\nMANUALLY INSPECTED TRACKS")
print("--------------------------------")

display(
    pd.DataFrame(manual_rows)
)


# -----------------------------------
# 6. Inspect changed predictions
# -----------------------------------

print("\nEXAMPLES OF CHANGED ESTIMATES")
print("--------------------------------")

changed_tracks = key_margin_df[
    key_margin_df["margin_0.02"]
    != key_margin_df["margin_0.0"]
]

print(
    "Tracks changed at margin 0.02:",
    len(changed_tracks)
)

display(
    changed_tracks.sample(
        n=min(12, len(changed_tracks)),
        random_state=42
    )[
        [
            "track_id",
            "margin_0.0",
            "margin_0.02",
            "Original"
        ]
    ]
)

print("\nAnalysis completed!")

TWO-STAGE KEY ESTIMATION
--------------------------------


,Compatibility margin,Agreement with original (%),Changed from exact-fit selection
0,0.00,70.34,0
1,0.01,72.85,61
2,0.02,75.45,126
3,0.03,77.15,168



MANUALLY INSPECTED TRACKS
--------------------------------


,Track,Scale-only,Margin 0.01,Margin 0.02,Margin 0.03,Original chord estimate
0,track_0571479,A major,A major,A major,A major,A major
1,track_0603758,G major,G major,G major,G major,G major
2,track_0854720,A major,A major,A major,A major,A major
3,track_1049878,G major,D major,D major,D major,D major
4,track_1096796,G major,G major,G major,G major,G major
5,track_1129652,G major,G major,G major,G major,G major
6,track_1106439,C major,C major,C major,C major,C major
7,track_0785281,A major,A major,A major,A major,A major



EXAMPLES OF CHANGED ESTIMATES
--------------------------------
Tracks changed at margin 0.02: 126


,track_id,margin_0.0,margin_0.02,Original
556,track_1056817,D major,A major,A major
149,track_0368770,G major,C major,G major
1034,track_1169060,A major,D major,D major
516,track_1049827,B major,F# major,F# major
801,track_1124726,F# major,B major,F# major
616,track_1075425,A major,D major,D major
234,track_0618822,E major,A major,A major
403,track_0976868,C major,Bb major,F major
1043,track_1169399,B major,G major,G major
316,track_0881095,C# major,Ab major,Eb major



Analysis completed!


### Findings from Step 3.14
The two-stage major-scale estimator combines duration-weighted chord-tone compatibility with harmonic root prominence. Using a compatibility margin of 0.02, the method achieved 75.45% agreement with the original chord-based estimator across 1,116 tracks.
The method also resolved several ambiguities observed during manual inspection, including the distinction between A major and D major for track 0854720.
Since agreement with another estimator does not establish accuracy, the method will be treated as provisional. Alternative scale candidates and uncertainty indicators will be retained for further evaluation.

### Provisional Key Assignment
Objective: Generate provisional major-scale reference keys for all candidate tracks using the independently developed two-stage chord-based estimator.
For each track, the algorithm records the selected major-scale collection, scale compatibility, alternative candidates, and harmonic root support.
Predictions sensitive to the compatibility margin, supported by closely competing candidates, or exhibiting low diatonic compatibility are flagged for further review.
These review criteria are exploratory and are not calibrated confidence estimates. No tracks are removed, and existing source annotations remain unchanged.

In [18]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1. Provisional estimator settings
# -----------------------------------

KEY_MARGIN = 0.02

# Diagnostic thresholds only.
LOW_SCALE_FIT = 0.80
CLOSE_ROOT_SUPPORT = 0.05

phase3_key_records = []


# -----------------------------------
# 2. Estimate keys for all candidates
# -----------------------------------

for track_id in sorted(candidate_tracks):

    evidence = collection_evidence(track_id)

    best_fit = evidence["Scale fit"].max()

    # Keep collections close to the best fit.
    plausible = evidence[
        evidence["Scale fit"]
        >= best_fit - KEY_MARGIN - 1e-10
    ].copy()

    # Select the strongest root-supported scale.
    plausible = plausible.sort_values(
        ["Root support", "Scale fit", "pc"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    selected = plausible.iloc[0]

    # Best alternative within the plausible set.
    if len(plausible) > 1:

        alternative = plausible.iloc[1]

        support_gap = (
            selected["Root support"]
            - alternative["Root support"]
        )

    else:

        other = evidence[
            evidence["pc"] != selected["pc"]
        ].sort_values(
            ["Scale fit", "Root support"],
            ascending=False
        )

        alternative = other.iloc[0]
        support_gap = np.nan

    # Check sensitivity to the chosen margin.
    prediction_001 = select_major_key(
        evidence, 0.01
    )

    prediction_003 = select_major_key(
        evidence, 0.03
    )

    margin_stable = (
        prediction_001 == selected["Key"]
        and prediction_003 == selected["Key"]
    )

    # Flags for investigation, not rejection.
    low_fit = (
        selected["Scale fit"] < LOW_SCALE_FIT
    )

    close_support = (
        len(plausible) > 1
        and support_gap < CLOSE_ROOT_SUPPORT
    )

    review_flag = (
        low_fit
        or close_support
        or not margin_stable
    )

    phase3_key_records.append({

        "track_id": track_id,

        "estimated_key": selected["Key"],

        "alternative_key": alternative["Key"],

        "scale_fit": float(
            selected["Scale fit"]
        ),

        "best_possible_fit": float(best_fit),

        "root_support": float(
            selected["Root support"]
        ),

        "root_support_gap": float(support_gap),

        "plausible_collections": len(plausible),

        "margin_stable": margin_stable,

        "low_scale_fit": bool(low_fit),

        "close_root_support": bool(close_support),

        "review_flag": bool(review_flag)
    })


phase3_key_df = pd.DataFrame(
    phase3_key_records
)

assert len(phase3_key_df) == len(candidate_tracks)


# -----------------------------------
# 3. Dataset summary
# -----------------------------------

print("PHASE 3 — PROVISIONAL KEY ASSIGNMENT")
print("--------------------------------")

print("Tracks analyzed:", len(phase3_key_df))

print(
    "Tracks flagged for review:",
    int(phase3_key_df["review_flag"].sum())
)

print(
    "Tracks not flagged:",
    int((~phase3_key_df["review_flag"]).sum())
)


# -----------------------------------
# 4. Reasons for review
# -----------------------------------

print("\nREVIEW INDICATORS")
print("--------------------------------")

review_summary = pd.DataFrame({

    "Indicator": [
        "Low scale compatibility",
        "Close harmonic root support",
        "Sensitive to compatibility margin",
        "Any review indicator"
    ],

    "Tracks": [
        int(phase3_key_df["low_scale_fit"].sum()),

        int(phase3_key_df["close_root_support"].sum()),

        int((~phase3_key_df["margin_stable"]).sum()),

        int(phase3_key_df["review_flag"].sum())
    ]
})

display(review_summary)


# -----------------------------------
# 5. Distribution of reference keys
# -----------------------------------

print("\nESTIMATED MAJOR KEYS")
print("--------------------------------")

display(
    phase3_key_df["estimated_key"]
    .value_counts()
    .rename_axis("Key")
    .reset_index(name="Tracks")
)


# -----------------------------------
# 6. Inspect uncertain examples
# -----------------------------------

print("\nEXAMPLES FLAGGED FOR REVIEW")
print("--------------------------------")

review_examples = phase3_key_df[
    phase3_key_df["review_flag"]
]

display(
    review_examples.sample(
        n=min(12, len(review_examples)),
        random_state=42
    ).round(3)
)


# -----------------------------------
# 7. Inspect our familiar tracks
# -----------------------------------

print("\nMANUALLY INSPECTED TRACKS")
print("--------------------------------")

display(
    phase3_key_df[
        phase3_key_df["track_id"].isin(MANUAL_TRACKS)
    ][[
        "track_id",
        "estimated_key",
        "alternative_key",
        "scale_fit",
        "root_support",
        "margin_stable",
        "review_flag"
    ]].round(3)
)


print("\nProvisional key assignment completed!")

PHASE 3 — PROVISIONAL KEY ASSIGNMENT
--------------------------------
Tracks analyzed: 1116
Tracks flagged for review: 180
Tracks not flagged: 936

REVIEW INDICATORS
--------------------------------


,Indicator,Tracks
0,Low scale compatibility,61
1,Close harmonic root support,46
2,Sensitive to compatibility margin,110
3,Any review indicator,180



ESTIMATED MAJOR KEYS
--------------------------------


,Key,Tracks
0,C major,230
1,G major,194
2,F major,118
3,D major,117
4,A major,87
5,Bb major,76
6,Eb major,75
7,E major,65
8,B major,43
9,F# major,43



EXAMPLES FLAGGED FOR REVIEW
--------------------------------


,track_id,estimated_key,alternative_key,scale_fit,best_possible_fit,root_support,root_support_gap,plausible_collections,margin_stable,low_scale_fit,close_root_support,review_flag
97,track_0225366,Bb major,Eb major,0.851,0.851,0.249,0.007,2,True,False,True,True
195,track_0565795,G major,C major,0.984,1.000,0.860,0.768,2,False,False,False,True
1025,track_1168477,F major,C# major,0.693,0.706,0.382,0.250,3,False,True,False,True
444,track_1028815,D major,A major,0.952,0.967,0.577,0.396,2,False,False,False,True
966,track_1159696,F major,C major,0.971,0.983,0.668,0.396,2,False,False,False,True
77,track_0156888,Ab major,Eb major,0.912,0.924,0.718,0.531,2,False,False,False,True
115,track_0243794,G major,D major,0.742,0.755,0.295,0.021,2,False,True,True,True
390,track_0971191,Eb major,Bb major,0.724,0.739,0.288,0.023,2,False,True,True,True
698,track_1090470,A major,D major,0.891,0.891,0.085,NaN,1,False,False,False,True
741,track_1107314,F major,C major,0.897,0.897,0.088,NaN,1,False,False,False,True



MANUALLY INSPECTED TRACKS
--------------------------------


,track_id,estimated_key,alternative_key,scale_fit,root_support,margin_stable,review_flag
222,track_0603758,G major,D major,0.907,0.658,True,False
279,track_0785281,A major,D major,0.816,0.304,True,False
299,track_0854720,A major,D major,1.000,0.763,True,False
524,track_1049878,D major,G major,0.907,0.441,True,False
713,track_1096796,G major,C major,1.000,0.330,True,False
735,track_1106439,C major,G major,0.989,0.424,True,False
835,track_1129652,G major,C major,1.000,0.524,True,False



Provisional key assignment completed!


### Findings from Step 3.15
The provisional chord-based major-scale estimator successfully analyzed all 1,116 candidate tracks. A total of 936 tracks showed no predefined uncertainty indicators, while 180 were flagged for review due to low scale compatibility, competing harmonic interpretations, or sensitivity to the selection margin.
The estimator uses only absolute chord annotations and does not rely on previously generated key labels. All predictions remain provisional, as the absence of uncertainty flags does not guarantee transcription or key-estimation accuracy.
The next experiment evaluates whether extracted melody notes are consistent with the independently estimated major-scale collections.

### Melody–Scale Consistency
Objective: Evaluate the consistency between extracted melody pitch classes and independently estimated major-scale collections.
The analysis calculates the duration-weighted proportion of melody notes belonging to each song's estimated seven-note diatonic collection. Individual note durations are capped at two seconds to limit the influence of unusually long detected notes.
Compatibility with an alternative estimated scale is also measured. Additional tests transpose melody pitch classes by ±1 semitone to identify potential systematic pitch-offset problems.
Chromatic notes are not automatically classified as errors because non-diatonic melodic notes are common in rock, jazz, blues, and other musical styles. All measurements are diagnostic; no songs are removed during this experiment.

In [19]:
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Configuration
# -----------------------------------

MAX_NOTE_DURATION = 2.0

MAJOR_INTERVALS = {0, 2, 4, 5, 7, 9, 11}


# -----------------------------------
# 2. Helper: major-key pitch classes
# -----------------------------------

def get_major_pc(key_name_str):

    if not isinstance(key_name_str, str):
        return None

    tonic = key_name_str.replace(
        " major", ""
    ).strip()

    if tonic not in PC_NAMES:
        return None

    return PC_NAMES.index(tonic)


# -----------------------------------
# 3. Build duration-weighted
#    melody pitch-class distribution
# -----------------------------------

def melody_pitch_distribution(notes):

    distribution = np.zeros(12, dtype=float)

    valid_notes = 0

    for note in notes:

        pitch = note.get("midi_pitch_rounded")

        if pitch is None:
            continue

        try:
            pitch = float(pitch)

            onset = float(note["onset_sec"])
            offset = float(note["offset_sec"])

        except (TypeError, ValueError, KeyError):
            continue

        if not np.isfinite(
            [pitch, onset, offset]
        ).all():
            continue

        duration = offset - onset

        if duration <= 0:
            continue

        duration = min(
            duration,
            MAX_NOTE_DURATION
        )

        pitch_class = int(round(pitch)) % 12

        distribution[pitch_class] += duration

        valid_notes += 1

    return distribution, valid_notes


# -----------------------------------
# 4. Calculate scale compatibility
# -----------------------------------

def melody_scale_fit(
    distribution,
    major_pc,
    semitone_shift=0
):

    total = distribution.sum()

    if total <= 0 or major_pc is None:
        return np.nan

    scale = {
        (major_pc + interval) % 12
        for interval in MAJOR_INTERVALS
    }

    # Check where each original pitch class
    # would fall after the specified shift.

    matching_duration = sum(
        duration
        for pc, duration in enumerate(distribution)
        if (pc + semitone_shift) % 12 in scale
    )

    return matching_duration / total


# -----------------------------------
# 5. Analyze all candidate songs
# -----------------------------------

melody_scale_records = []

for _, row in phase3_key_df.iterrows():

    track_id = row["track_id"]

    folder = track_lookup[track_id]

    melody_data = read_json(
        folder / "melody/combined.json"
    )

    notes = melody_data.get("notes", [])

    distribution, valid_notes = (
        melody_pitch_distribution(notes)
    )

    estimated_pc = get_major_pc(
        row["estimated_key"]
    )

    alternative_pc = get_major_pc(
        row["alternative_key"]
    )

    # Original melody compatibility
    original_fit = melody_scale_fit(
        distribution,
        estimated_pc
    )

    # Compatibility with alternative scale
    alternative_fit = melody_scale_fit(
        distribution,
        alternative_pc
    )

    # Test possible semitone offsets
    minus_one_fit = melody_scale_fit(
        distribution,
        estimated_pc,
        semitone_shift=-1
    )

    plus_one_fit = melody_scale_fit(
        distribution,
        estimated_pc,
        semitone_shift=1
    )

    shift_scores = {
        -1: minus_one_fit,
         0: original_fit,
         1: plus_one_fit
    }

    if valid_notes > 0:

        best_shift = max(
            shift_scores,
            key=shift_scores.get
        )

        best_shift_fit = shift_scores[
            best_shift
        ]

        shift_improvement = (
            best_shift_fit - original_fit
        )

    else:

        best_shift = np.nan
        best_shift_fit = np.nan
        shift_improvement = np.nan

    melody_scale_records.append({

        "track_id": track_id,

        "estimated_key":
            row["estimated_key"],

        "alternative_key":
            row["alternative_key"],

        "key_review_flag":
            row["review_flag"],

        "melody_notes": valid_notes,

        "melody_duration_sec":
            distribution.sum(),

        "melody_scale_fit":
            original_fit,

        "out_of_scale_fraction":
            1 - original_fit
            if np.isfinite(original_fit)
            else np.nan,

        "alternative_scale_fit":
            alternative_fit,

        "shift_minus_1_fit":
            minus_one_fit,

        "shift_plus_1_fit":
            plus_one_fit,

        "best_semitone_shift":
            best_shift,

        "shift_improvement":
            shift_improvement
    })


melody_scale_df = pd.DataFrame(
    melody_scale_records
)


# -----------------------------------
# 6. Dataset overview
# -----------------------------------

print("MELODY–SCALE CONSISTENCY")
print("--------------------------------")

print(
    "Tracks analyzed:",
    len(melody_scale_df)
)

print(
    "Tracks with no valid melody notes:",
    int(
        (melody_scale_df["melody_notes"] == 0).sum()
    )
)


# -----------------------------------
# 7. Melody compatibility statistics
# -----------------------------------

print("\nMELODY–SCALE FIT STATISTICS")
print("--------------------------------")

display(
    melody_scale_df[
        [
            "melody_notes",
            "melody_scale_fit",
            "out_of_scale_fraction"
        ]
    ].describe().round(3)
)


# -----------------------------------
# 8. Distribution of melody compatibility
# -----------------------------------

print("\nMELODY COMPATIBILITY THRESHOLDS")
print("--------------------------------")

valid_df = melody_scale_df[
    melody_scale_df["melody_scale_fit"].notna()
]

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:

    count = (
        valid_df["melody_scale_fit"] < threshold
    ).sum()

    print(
        f"Below {threshold:.0%}: "
        f"{count} tracks"
    )


# -----------------------------------
# 9. Possible semitone-offset problems
# -----------------------------------

print("\nPOSSIBLE SEMITONE-OFFSET CASES")
print("--------------------------------")

# Improvement of at least 10 percentage points
# after shifting melody by ±1 semitone.

shift_candidates = melody_scale_df[
    (melody_scale_df["shift_improvement"] >= 0.10)
    & (melody_scale_df["melody_notes"] >= 30)
].copy()

print(
    "Tracks with improvement >= 10 percentage points:",
    len(shift_candidates)
)

display(
    shift_candidates[
        [
            "track_id",
            "estimated_key",
            "melody_notes",
            "melody_scale_fit",
            "best_semitone_shift",
            "shift_improvement"
        ]
    ].sort_values(
        "shift_improvement",
        ascending=False
    ).head(15).round(3)
)


# -----------------------------------
# 10. Inspect problematic melodies
# -----------------------------------

print("\nLOWEST MELODY–SCALE COMPATIBILITY")
print("--------------------------------")

display(
    melody_scale_df[
        melody_scale_df["melody_notes"] >= 30
    ].sort_values(
        "melody_scale_fit"
    )[
        [
            "track_id",
            "estimated_key",
            "melody_notes",
            "melody_scale_fit",
            "alternative_scale_fit",
            "key_review_flag"
        ]
    ].head(15).round(3)
)


# -----------------------------------
# 11. Previously inspected tracks
# -----------------------------------

MANUAL_CHECK_TRACKS = [
    "track_0603758",
    "track_0785281",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652",
    "track_1106439"
]

print("\nMANUALLY INSPECTED TRACKS")
print("--------------------------------")

display(
    melody_scale_df[
        melody_scale_df["track_id"].isin(
            MANUAL_CHECK_TRACKS
        )
    ][
        [
            "track_id",
            "estimated_key",
            "melody_notes",
            "melody_scale_fit",
            "alternative_scale_fit",
            "best_semitone_shift",
            "shift_improvement"
        ]
    ].round(3)
)


print("\nMelody–scale analysis completed!")

MELODY–SCALE CONSISTENCY
--------------------------------
Tracks analyzed: 1116
Tracks with no valid melody notes: 0

MELODY–SCALE FIT STATISTICS
--------------------------------


,melody_notes,melody_scale_fit,out_of_scale_fraction
count,1116.000,1116.000,1116.000
mean,505.019,0.873,0.127
std,278.302,0.103,0.103
min,3.000,0.098,-0.000
25%,338.000,0.819,0.047
50%,467.500,0.897,0.103
75%,614.000,0.953,0.181
max,2463.000,1.000,0.902



MELODY COMPATIBILITY THRESHOLDS
--------------------------------
Below 50%: 2 tracks
Below 60%: 17 tracks
Below 70%: 88 tracks
Below 80%: 241 tracks
Below 90%: 570 tracks

POSSIBLE SEMITONE-OFFSET CASES
--------------------------------
Tracks with improvement >= 10 percentage points: 8


,track_id,estimated_key,melody_notes,melody_scale_fit,best_semitone_shift,shift_improvement
325,track_0897311,E major,327,0.098,1,0.895
1115,track_1419770,C# major,292,0.331,1,0.513
279,track_0785281,A major,176,0.534,1,0.306
613,track_1075180,B major,98,0.583,-1,0.279
12,track_0007392,C major,76,0.591,1,0.194
808,track_1126090,E major,648,0.622,-1,0.155
395,track_0973387,E major,377,0.660,-1,0.154
125,track_0345140,A major,484,0.600,-1,0.147



LOWEST MELODY–SCALE COMPATIBILITY
--------------------------------


,track_id,estimated_key,melody_notes,melody_scale_fit,alternative_scale_fit,key_review_flag
325,track_0897311,E major,327,0.098,0.072,True
1115,track_1419770,C# major,292,0.331,0.447,True
76,track_0156887,A major,387,0.530,0.691,True
105,track_0230871,G major,397,0.533,0.758,True
279,track_0785281,A major,176,0.534,0.292,False
344,track_0920434,Bb major,480,0.565,0.566,False
799,track_1124712,A major,150,0.573,0.516,True
75,track_0156873,Bb major,627,0.575,0.599,False
1082,track_1300047,C major,205,0.580,0.662,True
925,track_1140605,F major,700,0.581,0.716,True



MANUALLY INSPECTED TRACKS
--------------------------------


,track_id,estimated_key,melody_notes,melody_scale_fit,alternative_scale_fit,best_semitone_shift,shift_improvement
222,track_0603758,G major,490,0.961,0.919,0,0.000
279,track_0785281,A major,176,0.534,0.292,1,0.306
299,track_0854720,A major,397,0.937,0.941,0,0.000
524,track_1049878,D major,591,0.721,0.741,0,0.000
713,track_1096796,G major,484,0.785,0.592,0,0.000
735,track_1106439,C major,490,0.970,0.957,0,0.000
835,track_1129652,G major,484,0.957,0.935,0,0.000



Melody–scale analysis completed!


### Findings from Step 3.16
The duration-weighted melody–scale analysis achieved an average compatibility of 87.3% across 1,116 candidate tracks. Only 17 tracks exhibited compatibility below 60%, suggesting that most extracted melodies broadly agree with their independently estimated diatonic pitch collections.
Eight tracks exhibited improvements of at least ten percentage points when their melodies were shifted by one semitone. These tracks were flagged for further investigation rather than automatically corrected, because the improvement could indicate pitch-extraction errors, tuning differences, or incorrect reference-scale estimates.
The next experiment evaluates direct melody–chord compatibility using temporal alignment between melody notes and absolute chord annotations.

### Chord-Tone Compatibility
Objective: Evaluate the temporal and harmonic consistency between extracted melody notes and absolute chord annotations.
Each melody note is aligned with the chord events overlapping its duration. The analysis measures the proportion of aligned melody-note duration corresponding to chord tones, using note durations capped at two seconds.
Chord-tone compatibility is also calculated after hypothetical shifts of ±1 semitone to investigate systematic pitch mismatches.
Non-chord tones are musically valid, including passing tones, suspensions, and extensions. Consequently, low chord-tone compatibility is treated as a diagnostic indicator rather than an automatic rejection criterion.

In [21]:
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Analyze melody–chord alignment
# -----------------------------------

def analyze_melody_chord_fit(track_id):

    folder = track_lookup[track_id]

    melody_data = read_json(
        folder / "melody/combined.json"
    )

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    notes = sorted(
        melody_data.get("notes", []),
        key=lambda n: n["onset_sec"]
    )

    # Reuse the absolute-chord parser
    # developed in Step 3.9.

    chords = prepare_key_chords(
        chord_data["events"]
    )

    total_note_duration = 0.0
    aligned_duration = 0.0

    matched = {
        -1: 0.0,
         0: 0.0,
         1: 0.0
    }

    chord_idx = 0

    for note in notes:

        pitch = note.get("midi_pitch_rounded")

        if pitch is None:
            continue

        try:
            pitch = float(pitch)
            start = float(note["onset_sec"])
            end = float(note["offset_sec"])
        except (ValueError, TypeError, KeyError):
            continue

        if not np.isfinite(
            [pitch, start, end]
        ).all():
            continue

        duration = end - start

        if duration <= 0:
            continue

        pc = int(round(pitch)) % 12

        # Apply the same 2-second duration cap
        # used in the melody–scale experiment.

        note_weight = min(duration, 2.0) / duration

        total_note_duration += min(duration, 2.0)

        # Skip chords ending before this note.

        while (
            chord_idx < len(chords)
            and chords[chord_idx]["end"] <= start
        ):
            chord_idx += 1

        j = chord_idx

        # Find all chords overlapping the note.

        while (
            j < len(chords)
            and chords[j]["start"] < end
        ):

            chord = chords[j]

            overlap = max(
                0.0,
                min(end, chord["end"])
                - max(start, chord["start"])
            )

            if overlap > 0:

                weight = overlap * note_weight

                aligned_duration += weight

                for shift in [-1, 0, 1]:

                    shifted_pc = (pc + shift) % 12

                    if shifted_pc in chord["tones"]:
                        matched[shift] += weight

            j += 1

    if aligned_duration == 0:

        return {
            "chord_coverage": np.nan,
            "chord_tone_fit": np.nan,
            "shift_minus_1": np.nan,
            "shift_plus_1": np.nan,
            "best_shift": np.nan,
            "shift_improvement": np.nan
        }

    fits = {
        shift: matched[shift] / aligned_duration
        for shift in [-1, 0, 1]
    }

    # Prefer no shift when scores are tied.
    best_shift = max(
        [0, -1, 1],
        key=lambda shift: fits[shift]
    )

    return {
        "chord_coverage":
            aligned_duration / total_note_duration,

        "chord_tone_fit":
            fits[0],

        "shift_minus_1":
            fits[-1],

        "shift_plus_1":
            fits[1],

        "best_shift":
            best_shift,

        "shift_improvement":
            fits[best_shift] - fits[0]
    }


# -----------------------------------
# 2. Analyze all candidate tracks
# -----------------------------------

chord_fit_records = []

for track_id in sorted(candidate_tracks):

    result = analyze_melody_chord_fit(
        track_id
    )

    chord_fit_records.append({
        "track_id": track_id,
        **result
    })

chord_fit_df = pd.DataFrame(
    chord_fit_records
)

assert len(chord_fit_df) == len(candidate_tracks)


# -----------------------------------
# 3. Combine with scale-fit results
# -----------------------------------

# Give the scale-based shift measurement a distinct name.
# Keep "shift_improvement" for chord-tone compatibility.

consistency_df = melody_scale_df.rename(
    columns={
        "shift_improvement": "scale_shift_improvement"
    }
).merge(
    chord_fit_df,
    on="track_id",
    how="inner",
    validate="one_to_one"
)

print("Merged tracks:", len(consistency_df))


# -----------------------------------
# 4. Overall statistics
# -----------------------------------

print("MELODY–CHORD COMPATIBILITY")
print("--------------------------------")

print("Tracks analyzed:", len(consistency_df))

display(
    consistency_df[
        [
            "melody_scale_fit",
            "chord_tone_fit",
            "chord_coverage",
            "shift_improvement"
        ]
    ].describe().round(3)
)


# -----------------------------------
# 5. Chord-tone compatibility
# -----------------------------------

print("\nCHORD-TONE COMPATIBILITY")
print("--------------------------------")

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:

    count = (
        consistency_df["chord_tone_fit"] < threshold
    ).sum()

    print(
        f"Below {threshold:.0%}: "
        f"{count} tracks"
    )


# -----------------------------------
# 6. Possible systematic pitch offsets
# -----------------------------------

print("\nPOSSIBLE PITCH-OFFSET CASES")
print("--------------------------------")

shift_cases = consistency_df[
    (consistency_df["shift_improvement"] >= 0.10)
    & (consistency_df["melody_notes"] >= 30)
    & (consistency_df["chord_coverage"] >= 0.60)
]

print(
    "Tracks improving >= 10 percentage points:",
    len(shift_cases)
)

display(
    shift_cases.sort_values(
        "shift_improvement",
        ascending=False
    )[
        [
            "track_id",
            "estimated_key",
            "melody_scale_fit",
            "chord_tone_fit",
            "best_shift",
            "shift_improvement"
        ]
    ].head(15).round(3)
)


# -----------------------------------
# 7. Previously identified problem songs
# -----------------------------------

IMPORTANT_TRACKS = [
    "track_0785281",
    "track_0897311",
    "track_1419770",
    "track_0603758",
    "track_0854720",
    "track_1049878",
    "track_1096796",
    "track_1129652"
]

print("\nIMPORTANT TRACKS")
print("--------------------------------")

display(
    consistency_df[
        consistency_df["track_id"].isin(
            IMPORTANT_TRACKS
        )
    ][
        [
            "track_id",
            "estimated_key",
            "melody_scale_fit",
            "chord_tone_fit",
            "chord_coverage",
            "best_shift",
            "shift_improvement"
        ]
    ].round(3)
)


print("\nMelody–chord analysis completed!")

Merged tracks: 1116
MELODY–CHORD COMPATIBILITY
--------------------------------
Tracks analyzed: 1116


,melody_scale_fit,chord_tone_fit,chord_coverage,shift_improvement
count,1116.000,1114.000,1114.000,1114.000
mean,0.873,0.602,0.960,0.002
std,0.103,0.109,0.084,0.029
min,0.098,0.059,0.225,0.000
25%,0.819,0.536,0.961,0.000
50%,0.897,0.606,0.991,0.000
75%,0.953,0.674,1.000,0.000
max,1.000,0.903,1.000,0.796



CHORD-TONE COMPATIBILITY
--------------------------------
Below 30%: 8 tracks
Below 40%: 52 tracks
Below 50%: 165 tracks
Below 60%: 520 tracks
Below 70%: 917 tracks

POSSIBLE PITCH-OFFSET CASES
--------------------------------
Tracks improving >= 10 percentage points: 6


,track_id,estimated_key,melody_scale_fit,chord_tone_fit,best_shift,shift_improvement
325,track_0897311,E major,0.098,0.059,-1.0,0.796
613,track_1075180,B major,0.583,0.145,-1.0,0.331
253,track_0659900,C# major,0.980,0.242,1.0,0.304
1115,track_1419770,C# major,0.331,0.187,1.0,0.150
250,track_0659893,C major,1.000,0.458,1.0,0.116
764,track_1114501,C major,0.979,0.411,1.0,0.112



IMPORTANT TRACKS
--------------------------------


,track_id,estimated_key,melody_scale_fit,chord_tone_fit,chord_coverage,best_shift,shift_improvement
222,track_0603758,G major,0.961,0.545,0.990,0.0,0.000
279,track_0785281,A major,0.534,0.330,1.000,1.0,0.098
299,track_0854720,A major,0.937,0.713,1.000,0.0,0.000
325,track_0897311,E major,0.098,0.059,0.973,-1.0,0.796
524,track_1049878,D major,0.721,0.588,0.990,0.0,0.000
713,track_1096796,G major,0.785,0.464,0.917,0.0,0.000
835,track_1129652,G major,0.957,0.640,0.999,0.0,0.000
1115,track_1419770,C# major,0.331,0.187,0.994,1.0,0.150



Melody–chord analysis completed!


### Findings from Step 3.17
Melody–chord compatibility analysis was completed for 1,116 candidate tracks. Average melody–scale compatibility was 87.3%, while average direct chord-tone compatibility was 60.2%. Approximately 96% of extracted melody duration overlapped recognized chord annotations.
These results suggest that most tracks contain substantial usable musical information. However, non-chord tones are common in legitimate melodies, so chord-tone compatibility will not be used as a strict rejection criterion.
Only a small number of tracks exhibited substantial semitone-shift improvements. Automatic pitch correction was therefore deferred.
The next stage will evaluate complete musical bars and identify suitable training windows.

### Training Window Availability
Objective: Evaluate the availability of complete 8-bar and 16-bar training windows across the 1,116 candidate tracks.
Windows are constructed using the beat and downbeat annotations. Only complete bars containing the expected number of beats are considered.
The experiment compares non-overlapping and overlapping windows to determine how window length affects training-data availability.
No quality filters are applied during this experiment.

In [22]:
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Extract consecutive complete bars
# -----------------------------------

def extract_complete_bar_runs(beat_events):

    beats = sorted(
        beat_events,
        key=lambda b: float(b["time_sec"])
    )

    # Locate bar beginnings.
    downbeat_indices = [
        i for i, beat in enumerate(beats)
        if beat.get("beat_in_bar") == 1
    ]

    runs = []
    current_run = []

    for start_idx, end_idx in zip(
        downbeat_indices,
        downbeat_indices[1:]
    ):

        bar_beats = beats[start_idx:end_idx]

        try:
            meter = int(
                bar_beats[0]["meter_numerator"]
            )

            positions = [
                int(b["beat_in_bar"])
                for b in bar_beats
            ]

            start_time = float(
                beats[start_idx]["time_sec"]
            )

            end_time = float(
                beats[end_idx]["time_sec"]
            )

            valid = (
                2 <= meter <= 12
                and positions == list(range(1, meter + 1))
                and end_time > start_time
                and all(
                    int(b["meter_numerator"]) == meter
                    for b in bar_beats
                )
            )

        except (KeyError, ValueError, TypeError):
            valid = False

        if valid:

            bar = {
                "start": start_time,
                "end": end_time,
                "meter": meter
            }

            current_run.append(bar)

        else:

            if current_run:
                runs.append(current_run)

            current_run = []

    if current_run:
        runs.append(current_run)

    return runs


# -----------------------------------
# 2. Count windows in valid bar runs
# -----------------------------------

def count_windows(runs, window_bars, stride):

    total = 0

    for run in runs:

        n_bars = len(run)

        if n_bars >= window_bars:

            total += (
                (n_bars - window_bars) // stride
            ) + 1

    return total


# -----------------------------------
# 3. Analyze candidate tracks
# -----------------------------------

window_inventory = []

for track_id in sorted(candidate_tracks):

    folder = track_lookup[track_id]

    beat_data = read_json(
        folder / "beats/beats.json"
    )

    runs = extract_complete_bar_runs(
        beat_data.get("beats", [])
    )

    complete_bars = sum(
        len(run) for run in runs
    )

    windows_8 = count_windows(
        runs, window_bars=8, stride=4
    )

    windows_16 = count_windows(
        runs, window_bars=16, stride=8
    )

    windows_16_no_overlap = count_windows(
        runs, window_bars=16, stride=16
    )

    window_inventory.append({

        "track_id": track_id,

        "complete_bars": complete_bars,

        "longest_valid_run": max(
            (len(run) for run in runs),
            default=0
        ),

        "windows_8bar": windows_8,

        "windows_16bar": windows_16,

        "windows_16bar_no_overlap":
            windows_16_no_overlap
    })


window_inventory_df = pd.DataFrame(
    window_inventory
)


# -----------------------------------
# 4. Dataset statistics
# -----------------------------------

print("PHASE 3 — WINDOW AVAILABILITY")
print("--------------------------------")

print(
    "Candidate tracks:",
    len(window_inventory_df)
)

display(
    window_inventory_df[
        [
            "complete_bars",
            "longest_valid_run",
            "windows_8bar",
            "windows_16bar"
        ]
    ].describe().round(2)
)


# -----------------------------------
# 5. Total available windows
# -----------------------------------

print("\nAVAILABLE TRAINING WINDOWS")
print("--------------------------------")

summary = pd.DataFrame([
    {
        "Configuration": "8 bars, stride 4",
        "Total windows":
            window_inventory_df["windows_8bar"].sum(),
        "Songs represented":
            (window_inventory_df["windows_8bar"] > 0).sum()
    },
    {
        "Configuration": "16 bars, stride 8",
        "Total windows":
            window_inventory_df["windows_16bar"].sum(),
        "Songs represented":
            (window_inventory_df["windows_16bar"] > 0).sum()
    },
    {
        "Configuration": "16 bars, stride 16",
        "Total windows":
            window_inventory_df[
                "windows_16bar_no_overlap"
            ].sum(),
        "Songs represented":
            (
                window_inventory_df[
                    "windows_16bar_no_overlap"
                ] > 0
            ).sum()
    }
])

display(summary)


# -----------------------------------
# 6. Songs without long enough sections
# -----------------------------------

print("\nSONGS WITHOUT COMPLETE WINDOWS")
print("--------------------------------")

print(
    "No 8-bar windows:",
    (window_inventory_df["windows_8bar"] == 0).sum()
)

print(
    "No 16-bar windows:",
    (window_inventory_df["windows_16bar"] == 0).sum()
)


# -----------------------------------
# 7. Example tracks
# -----------------------------------

print("\nEXAMPLE TRACKS")
print("--------------------------------")

display(
    window_inventory_df.sample(
        n=min(10, len(window_inventory_df)),
        random_state=42
    )
)

print("\nWindow availability analysis completed!")

PHASE 3 — WINDOW AVAILABILITY
--------------------------------
Candidate tracks: 1116


,complete_bars,longest_valid_run,windows_8bar,windows_16bar
count,1116.00,1116.00,1116.00,1116.00
mean,119.26,81.14,25.78,11.06
std,63.11,61.73,15.77,7.94
min,15.00,15.00,2.00,0.00
25%,82.00,43.00,16.00,6.00
50%,109.00,65.00,23.00,10.00
75%,143.00,98.00,31.00,13.25
max,757.00,757.00,188.00,93.00



AVAILABLE TRAINING WINDOWS
--------------------------------


,Configuration,Total windows,Songs represented
0,"8 bars, stride 4",28773,1116
1,"16 bars, stride 8",12346,1114
2,"16 bars, stride 16",6783,1114



SONGS WITHOUT COMPLETE WINDOWS
--------------------------------
No 8-bar windows: 0
No 16-bar windows: 2

EXAMPLE TRACKS
--------------------------------


,track_id,complete_bars,longest_valid_run,windows_8bar,windows_16bar,windows_16bar_no_overlap
993,track_1160059,169,130,38,17,9
101,track_0230823,109,88,25,11,6
486,track_1037231,120,71,22,8,5
862,track_1132141,239,144,55,25,13
1075,track_1294933,68,68,16,7,4
792,track_1119519,113,72,23,9,5
525,track_1050671,117,85,27,12,7
997,track_1160153,106,76,23,10,5
128,track_0345145,100,41,18,6,3
1071,track_1283374,112,112,27,13,7



Window availability analysis completed!


### Window-Level Quality Assessment
Objective: Evaluate the quality and harmonic diversity of all available 16-bar windows before defining final dataset filtering criteria.
Each window is assessed using melody-note density, temporal melody coverage, maximum melody silence, recognized-chord coverage, beat-level chord-label reliability, and harmonic root diversity.
Beat targets are considered reliable when a single recognized chord occupies at least 80% of the beat duration. Meaningful chord roots are distinguished from brief incidental detections using a minimum cumulative-duration requirement.
This analysis determines the distributions of quality indicators and estimates potential dataset retention under different filtering configurations. No windows are permanently excluded.

In [24]:
from collections import defaultdict
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Configuration
# -----------------------------------

WINDOW_BARS = 16
WINDOW_STRIDE = 8

BEAT_CHORD_THRESHOLD = 0.80


# -----------------------------------
# 2. Measure temporal coverage
# -----------------------------------

def interval_statistics(intervals, start, end):

    clipped = sorted(
        (
            max(a, start),
            min(b, end)
        )
        for a, b in intervals
        if a < end and b > start
    )

    if not clipped:
        return 0.0, end - start

    merged = []

    for a, b in clipped:

        if b <= a:
            continue

        if merged and a <= merged[-1][1]:
            merged[-1] = (
                merged[-1][0],
                max(merged[-1][1], b)
            )
        else:
            merged.append((a, b))

    if not merged:
        return 0.0, end - start

    covered = sum(
        b - a for a, b in merged
    )

    gaps = [merged[0][0] - start]

    gaps += [
        merged[i + 1][0] - merged[i][1]
        for i in range(len(merged) - 1)
    ]

    gaps.append(end - merged[-1][1])

    return (
        covered / (end - start),
        max(gaps)
    )


# -----------------------------------
# 3. Prepare recognized chord events
# -----------------------------------

def prepare_window_chords(events):

    chords = []

    for event in events:

        pc = root_to_pc(event.get("root"))

        if pc is None:
            continue

        start = float(event["start_sec"])
        end = float(event["end_sec"])

        if end <= start:
            continue

        chords.append({
            "start": start,
            "end": end,
            "root": pc,
            "quality": event.get("quality")
        })

    chords.sort(key=lambda c: c["start"])

    # Merge adjacent annotations representing
    # the same chord root and quality.

    merged = []

    for chord in chords:

        if (
            merged
            and merged[-1]["root"] == chord["root"]
            and merged[-1]["quality"] == chord["quality"]
            and 0 <= chord["start"] - merged[-1]["end"] <= 0.05
        ):
            merged[-1]["end"] = chord["end"]
        else:
            merged.append(chord.copy())

    return merged


# -----------------------------------
# 4. Analyze one window
# -----------------------------------

def analyze_window(
    start,
    end,
    notes,
    chords,
    beats
):

    duration = end - start

    # Melody notes overlapping this window.
    window_notes = [
        n for n in notes
        if n["start"] < end
        and n["end"] > start
    ]

    note_intervals = [
        (n["start"], n["end"])
        for n in window_notes
    ]

    melody_coverage, max_gap = (
        interval_statistics(
            note_intervals, start, end
        )
    )

    # Recognized chords overlapping this window.
    window_chords = [
        c for c in chords
        if c["start"] < end
        and c["end"] > start
    ]

    chord_intervals = [
        (c["start"], c["end"])
        for c in window_chords
    ]

    chord_coverage, _ = interval_statistics(
        chord_intervals, start, end
    )

    # Duration accumulated by each chord root.
    root_durations = defaultdict(float)

    for chord in window_chords:

        overlap = max(
            0.0,
            min(chord["end"], end)
            - max(chord["start"], start)
        )

        root_durations[chord["root"]] += overlap

    total_chord_duration = sum(
        root_durations.values()
    )

    meaningful_threshold = max(
        1.0,
        0.02 * total_chord_duration
    )

    meaningful_roots = sum(
        duration >= meaningful_threshold
        for duration in root_durations.values()
    )

    # Count changes between consecutive roots.
    root_transitions = 0
    previous = None

    for chord in window_chords:

        clipped_start = max(chord["start"], start)
        clipped_end = min(chord["end"], end)

        if clipped_end <= clipped_start:
            continue

        if previous is not None:

            gap = clipped_start - previous["end"]

            if (
                chord["root"] != previous["root"]
                and -0.05 <= gap <= 0.5
            ):
                root_transitions += 1

        previous = {
            "root": chord["root"],
            "end": clipped_end
        }

    # Beat-level chord target reliability.
    window_beats = sorted(
        float(b["time_sec"])
        for b in beats
        if start <= float(b["time_sec"]) < end
    )

    valid_beats = 0

    for i, beat_start in enumerate(window_beats):

        beat_end = (
            window_beats[i + 1]
            if i + 1 < len(window_beats)
            else end
        )

        beat_duration = beat_end - beat_start

        if beat_duration <= 0:
            continue

        best_overlap = 0.0

        for chord in window_chords:

            overlap = max(
                0.0,
                min(beat_end, chord["end"])
                - max(beat_start, chord["start"])
            )

            best_overlap = max(
                best_overlap,
                overlap
            )

        if (
            best_overlap / beat_duration
            >= BEAT_CHORD_THRESHOLD
        ):
            valid_beats += 1

    beat_target_coverage = (
        valid_beats / len(window_beats)
        if window_beats else 0.0
    )

    return {
        "note_count": len(window_notes),
        "melody_coverage": melody_coverage,
        "max_melody_gap_sec": max_gap,
        "chord_coverage": chord_coverage,
        "beat_target_coverage": beat_target_coverage,
        "unique_roots": len(root_durations),
        "meaningful_roots": meaningful_roots,
        "root_transitions": root_transitions
    }


# -----------------------------------
# 5. Generate and analyze all windows
# -----------------------------------

window_quality_records = []

for track_id in sorted(candidate_tracks):

    folder = track_lookup[track_id]

    melody_data = read_json(
        folder / "melody/combined.json"
    )

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    beat_data = read_json(
        folder / "beats/beats.json"
    )

    # Prepare melody notes once per track.
    notes = []

    for note in melody_data.get("notes", []):

        try:
            start = float(note["onset_sec"])
            end = float(note["offset_sec"])
            pitch = float(note["midi_pitch_rounded"])

            if (
                np.isfinite([start, end, pitch]).all()
                and end > start
            ):
                notes.append({
                    "start": start,
                    "end": end
                })

        except (KeyError, TypeError, ValueError):
            continue

    chords = prepare_window_chords(
        chord_data.get("events", [])
    )

    beats = beat_data.get("beats", [])

    runs = extract_complete_bar_runs(beats)

    window_number = 0

    for run in runs:

        if len(run) < WINDOW_BARS:
            continue

        for i in range(
            0,
            len(run) - WINDOW_BARS + 1,
            WINDOW_STRIDE
        ):

            start = run[i]["start"]

            end = run[
                i + WINDOW_BARS - 1
            ]["end"]

            quality = analyze_window(
                start,
                end,
                notes,
                chords,
                beats
            )

            window_quality_records.append({
                "track_id": track_id,
                "window_index": window_number,
                "start_sec": start,
                "end_sec": end,
                "duration_sec": end - start,
                **quality
            })

            window_number += 1


window_quality_df = pd.DataFrame(
    window_quality_records
)


# -----------------------------------
# 6. Overall statistics
# -----------------------------------

print("PHASE 3 — WINDOW QUALITY ANALYSIS")
print("--------------------------------")

print(
    "Windows analyzed:",
    len(window_quality_df)
)

print(
    "Songs represented:",
    window_quality_df["track_id"].nunique()
)

assert len(window_quality_df) == 12346, (
    "Window count differs from the previous "
    "inventory. Check bar-generation logic."
)

metrics = [
    "duration_sec",
    "note_count",
    "melody_coverage",
    "max_melody_gap_sec",
    "chord_coverage",
    "beat_target_coverage",
    "unique_roots",
    "meaningful_roots",
    "root_transitions"
]

display(
    window_quality_df[metrics]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .T
    .round(2)
)


# -----------------------------------
# 7. Examine potential filter thresholds
# -----------------------------------

print("\nPROVISIONAL FILTER ANALYSIS")
print("--------------------------------")

conditions = {
    "At least 20 melody notes":
        window_quality_df["note_count"] >= 20,

    "Melody coverage >= 50%":
        window_quality_df["melody_coverage"] >= 0.50,

    "Maximum melody gap <= 8 seconds":
        window_quality_df["max_melody_gap_sec"] <= 8,

    "Valid chord targets >= 80%":
        window_quality_df["beat_target_coverage"] >= 0.80,

    "At least 2 meaningful roots":
        window_quality_df["meaningful_roots"] >= 2,

    "At least 2 root transitions":
        window_quality_df["root_transitions"] >= 2
}

total = len(window_quality_df)

filter_rows = []

for name, condition in conditions.items():

    retained = int(condition.sum())

    filter_rows.append({
        "Condition": name,
        "Windows meeting condition": retained,
        "Percentage": 100 * retained / total
    })

display(
    pd.DataFrame(filter_rows).round(2)
)


# -----------------------------------
# 8. Combined provisional retention
# -----------------------------------

combined = pd.Series(
    True,
    index=window_quality_df.index
)

for condition in conditions.values():
    combined &= condition

retained_windows = window_quality_df[combined]

print("\nALL PROVISIONAL CONDITIONS COMBINED")
print("--------------------------------")

print(
    "Windows:",
    len(retained_windows)
)

print(
    "Percentage:",
    f"{100 * len(retained_windows) / total:.2f}%"
)

print(
    "Songs represented:",
    retained_windows["track_id"].nunique()
)


# -----------------------------------
# 9. Harmonic diversity distribution
# -----------------------------------

print("\nWINDOW HARMONIC DIVERSITY")
print("--------------------------------")

diversity_counts = pd.DataFrame({
    "Meaningful roots": [
        "0–1",
        "2",
        "3",
        "4+"
    ],
    "Windows": [
        int((window_quality_df["meaningful_roots"] <= 1).sum()),
        int((window_quality_df["meaningful_roots"] == 2).sum()),
        int((window_quality_df["meaningful_roots"] == 3).sum()),
        int((window_quality_df["meaningful_roots"] >= 4).sum())
    ]
})

display(diversity_counts)


print("\nWindow quality analysis completed!")

PHASE 3 — WINDOW QUALITY ANALYSIS
--------------------------------
Windows analyzed: 12346
Songs represented: 1114


,count,mean,std,min,10%,25%,50%,75%,90%,max
duration_sec,12346.0,30.76,9.09,15.32,20.48,24.85,30.32,35.84,42.24,66.60
note_count,12346.0,68.12,33.94,0.00,31.00,46.00,64.00,87.00,111.00,258.00
melody_coverage,12346.0,0.79,0.20,0.00,0.59,0.73,0.84,0.91,0.97,1.00
max_melody_gap_sec,12346.0,2.78,5.24,0.00,0.36,0.73,1.36,2.45,4.94,56.32
chord_coverage,12346.0,0.95,0.17,0.00,0.87,1.00,1.00,1.00,1.00,1.00
beat_target_coverage,12346.0,0.85,0.16,0.00,0.72,0.81,0.89,0.94,0.97,1.00
unique_roots,12346.0,4.02,1.56,0.00,2.00,3.00,4.00,5.00,6.00,12.00
meaningful_roots,12346.0,3.81,1.50,0.00,2.00,3.00,4.00,5.00,6.00,12.00
root_transitions,12346.0,10.36,6.34,0.00,2.00,6.00,10.00,14.00,18.00,41.00



PROVISIONAL FILTER ANALYSIS
--------------------------------


,Condition,Windows meeting condition,Percentage
0,At least 20 melody notes,11654,94.39
1,Melody coverage >= 50%,11491,93.07
2,Maximum melody gap <= 8 seconds,11546,93.52
3,Valid chord targets >= 80%,9690,78.49
4,At least 2 meaningful roots,11441,92.67
5,At least 2 root transitions,11318,91.67



ALL PROVISIONAL CONDITIONS COMBINED
--------------------------------
Windows: 8483
Percentage: 68.71%
Songs represented: 1055

WINDOW HARMONIC DIVERSITY
--------------------------------


,Meaningful roots,Windows
0,0–1,905
1,2,1250
2,3,2567
3,4+,7624



Window quality analysis completed!


### Findings from Step 3.19
Window-level analysis identified 12,346 complete 16-bar windows across 1,114 tracks. Applying the six provisional quality requirements retained 8,483 windows from 1,055 tracks.
Average melody coverage was 79%, and average valid beat-level chord-target coverage was 85%. Windows contained an average of 3.81 meaningful chord roots and 10.36 root transitions, indicating substantial harmonic variation.
Beat-level chord-target coverage was the most restrictive individual condition. Further analysis is required before finalizing its threshold. The current results demonstrate that 16-bar windows can provide a substantially larger training dataset than Phase 2.

### Window-Level Harmonic Consistency
Objective: Evaluate melody–scale and melody–chord compatibility within individual 16-bar windows.
Whole-song consistency measurements can conceal variations in extraction quality between musical sections. This experiment calculates duration-weighted melodic compatibility for each window using its independently estimated major-scale collection and temporally aligned absolute chord annotations.
Compatibility measurements will be compared against the six previously evaluated window-quality requirements. The objective is to identify unreliable sections without unnecessarily rejecting complete songs.
Non-chord tones and chromatic notes are not automatically considered extraction errors. No final compatibility thresholds are imposed during this experiment.

In [25]:
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Prepare reference key lookup
# -----------------------------------

key_lookup = phase3_key_df.set_index(
    "track_id"
)["estimated_key"].to_dict()


# -----------------------------------
# 2. Analyze harmonic compatibility
#    inside an individual window
# -----------------------------------

def window_harmonic_fit(
    start, end, notes, chords, major_pc
):

    scale = {
        (major_pc + interval) % 12
        for interval in MAJOR_INTERVALS
    }

    total_weight = 0.0
    scale_match = 0.0

    aligned_weight = 0.0
    chord_match = 0.0

    window_chords = [
        c for c in chords
        if c["start"] < end
        and c["end"] > start
    ]

    for note in notes:

        if note["start"] >= end:
            break

        if note["end"] <= start:
            continue

        a = max(start, note["start"])
        b = min(end, note["end"])

        duration = b - a

        if duration <= 0:
            continue

        weight = min(duration, 2.0)
        weight_factor = weight / duration

        total_weight += weight

        pc = note["pc"]

        if pc in scale:
            scale_match += weight

        # Compare note against the chords
        # active during its duration.

        for chord in window_chords:

            if chord["start"] >= b:
                break

            if chord["end"] <= a:
                continue

            overlap = max(
                0.0,
                min(b, chord["end"])
                - max(a, chord["start"])
            )

            if overlap <= 0:
                continue

            overlap_weight = overlap * weight_factor

            aligned_weight += overlap_weight

            if pc in chord["tones"]:
                chord_match += overlap_weight

    return {
        "window_scale_fit": (
            scale_match / total_weight
            if total_weight > 0 else np.nan
        ),

        "window_chord_fit": (
            chord_match / aligned_weight
            if aligned_weight > 0 else np.nan
        ),

        "melody_chord_alignment": (
            aligned_weight / total_weight
            if total_weight > 0 else np.nan
        )
    }


# -----------------------------------
# 3. Analyze all windows
# -----------------------------------

harmonic_records = []

for track_id, group in window_quality_df.groupby(
    "track_id"
):

    folder = track_lookup[track_id]

    melody_data = read_json(
        folder / "melody/combined.json"
    )

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    major_pc = get_major_pc(
        key_lookup[track_id]
    )

    # Prepare valid melody notes.
    notes = []

    for note in melody_data.get("notes", []):

        try:
            start = float(note["onset_sec"])
            end = float(note["offset_sec"])
            pitch = float(note["midi_pitch_rounded"])

            if not np.isfinite(
                [start, end, pitch]
            ).all():
                continue

            if end <= start:
                continue

            notes.append({
                "start": start,
                "end": end,
                "pc": int(round(pitch)) % 12
            })

        except (TypeError, ValueError, KeyError):
            continue

    notes.sort(key=lambda n: n["start"])

    # Prepare absolute chord tones.
    chords = prepare_key_chords(
        chord_data["events"]
    )

    for row in group.itertuples():

        result = window_harmonic_fit(
            row.start_sec,
            row.end_sec,
            notes,
            chords,
            major_pc
        )

        harmonic_records.append({
            "track_id": track_id,
            "window_index": row.window_index,
            **result
        })


window_harmonic_df = pd.DataFrame(
    harmonic_records
)


# -----------------------------------
# 4. Merge with window-quality data
# -----------------------------------

window_analysis_df = window_quality_df.merge(
    window_harmonic_df,
    on=["track_id", "window_index"],
    how="left",
    validate="one_to_one"
)

assert len(window_analysis_df) == len(window_quality_df)


# -----------------------------------
# 5. Overall compatibility statistics
# -----------------------------------

print("WINDOW-LEVEL HARMONIC COMPATIBILITY")
print("--------------------------------")

print(
    "Windows analyzed:",
    len(window_analysis_df)
)

display(
    window_analysis_df[
        [
            "window_scale_fit",
            "window_chord_fit",
            "melody_chord_alignment"
        ]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).round(3)
)


# -----------------------------------
# 6. Compatibility thresholds
# -----------------------------------

print("\nWINDOW COMPATIBILITY DISTRIBUTION")
print("--------------------------------")

for metric, thresholds in [
    (
        "window_scale_fit",
        [0.50, 0.60, 0.70, 0.80]
    ),
    (
        "window_chord_fit",
        [0.30, 0.40, 0.50, 0.60]
    )
]:

    print(f"\n{metric}")

    for threshold in thresholds:

        count = (
            window_analysis_df[metric] < threshold
        ).sum()

        print(
            f"Below {threshold:.0%}: "
            f"{count} windows"
        )


# -----------------------------------
# 7. Compare against our provisional
#    six quality filters
# -----------------------------------

base_filter = (
    (window_analysis_df["note_count"] >= 20)
    & (window_analysis_df["melody_coverage"] >= 0.50)
    & (window_analysis_df["max_melody_gap_sec"] <= 8)
    & (window_analysis_df["beat_target_coverage"] >= 0.80)
    & (window_analysis_df["meaningful_roots"] >= 2)
    & (window_analysis_df["root_transitions"] >= 2)
)

base_windows = window_analysis_df[
    base_filter
].copy()

print("\nWINDOWS PASSING PREVIOUS QUALITY FILTERS")
print("--------------------------------")

print("Windows:", len(base_windows))

print(
    "Songs:",
    base_windows["track_id"].nunique()
)

display(
    base_windows[
        [
            "window_scale_fit",
            "window_chord_fit",
            "melody_chord_alignment"
        ]
    ].describe().round(3)
)


# -----------------------------------
# 8. Explore possible combined filters
#    (diagnostic only)
# -----------------------------------

print("\nEXPLORATORY COMPATIBILITY FILTERS")
print("--------------------------------")

filter_results = []

for scale_threshold in [0.60, 0.70, 0.80]:

    for chord_threshold in [0.30, 0.40, 0.50]:

        selected = base_windows[
            (base_windows["window_scale_fit"]
             >= scale_threshold)
            &
            (base_windows["window_chord_fit"]
             >= chord_threshold)
        ]

        filter_results.append({
            "Scale fit minimum": scale_threshold,
            "Chord fit minimum": chord_threshold,
            "Windows": len(selected),
            "Songs": selected["track_id"].nunique()
        })

display(
    pd.DataFrame(filter_results)
)


print("\nWindow-level compatibility analysis completed!")

WINDOW-LEVEL HARMONIC COMPATIBILITY
--------------------------------
Windows analyzed: 12346


,window_scale_fit,window_chord_fit,melody_chord_alignment
count,12165.000,12026.000,12165.000
mean,0.866,0.593,0.961
std,0.139,0.159,0.150
min,0.000,0.000,0.000
10%,0.680,0.393,0.941
25%,0.804,0.498,1.000
50%,0.907,0.600,1.000
75%,0.971,0.699,1.000
90%,1.000,0.785,1.000
max,1.000,1.000,1.000



WINDOW COMPATIBILITY DISTRIBUTION
--------------------------------

window_scale_fit
Below 50%: 255 windows
Below 60%: 639 windows
Below 70%: 1441 windows
Below 80%: 2959 windows

window_chord_fit
Below 30%: 482 windows
Below 40%: 1278 windows
Below 50%: 3050 windows
Below 60%: 6017 windows

WINDOWS PASSING PREVIOUS QUALITY FILTERS
--------------------------------
Windows: 8483
Songs: 1055


,window_scale_fit,window_chord_fit,melody_chord_alignment
count,8483.000,8483.000,8483.000
mean,0.873,0.598,0.998
std,0.126,0.143,0.014
min,0.032,0.000,0.756
25%,0.815,0.506,1.000
50%,0.909,0.602,1.000
75%,0.968,0.698,1.000
max,1.000,1.000,1.000



EXPLORATORY COMPATIBILITY FILTERS
--------------------------------


,Scale fit minimum,Chord fit minimum,Windows,Songs
0,0.6,0.3,8002,1047
1,0.6,0.4,7535,1032
2,0.6,0.5,6354,991
3,0.7,0.3,7524,1029
4,0.7,0.4,7181,1017
5,0.7,0.5,6155,976
6,0.8,0.3,6522,981
7,0.8,0.4,6312,974
8,0.8,0.5,5532,943



Window-level compatibility analysis completed!


### Findings from Step 3.20
Window-level harmonic analysis demonstrated that the majority of provisionally selected windows exhibit reasonable melody–scale and melody–chord compatibility. Applying minimum thresholds of 60% melody–scale compatibility and 30% chord-tone compatibility retained 8,002 windows from 1,047 songs.
These thresholds were selected provisionally to identify extreme harmonic inconsistencies without unnecessarily excluding valid non-chord tones.
Since beat-level chord targets can be masked during Transformer training, the next experiment investigates whether relaxing the minimum proportion of reliable beats per window can increase dataset retention.

### Beat-Target Coverage and Dataset Retention
Objective: Compare alternative minimum beat-target coverage requirements for 16-bar training windows.
The experiment preserves the previous melody-quality, harmonic-diversity, and compatibility requirements while varying the minimum proportion of reliable beat-level chord targets.
Unreliable beats will be masked during model training. The objective is to retain more useful melody–chord examples without excessively reducing the number of supervised chord targets.
No final dataset is generated during this experiment.

In [26]:
import pandas as pd

# -----------------------------------
# 1. Existing quality conditions
#    excluding beat-target coverage
# -----------------------------------

base_conditions = (
    (window_analysis_df["note_count"] >= 20)
    & (window_analysis_df["melody_coverage"] >= 0.50)
    & (window_analysis_df["max_melody_gap_sec"] <= 8)
    & (window_analysis_df["meaningful_roots"] >= 2)
    & (window_analysis_df["root_transitions"] >= 2)
)


# -----------------------------------
# 2. Compatibility configurations
# -----------------------------------

compatibility_options = {
    "No compatibility filtering": None,
    "Light: scale >= 0.60, chord >= 0.30": (0.60, 0.30),
    "Moderate: scale >= 0.70, chord >= 0.40": (0.70, 0.40)
}


# -----------------------------------
# 3. Compare beat coverage thresholds
# -----------------------------------

beat_thresholds = [
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]

results = []

for beat_threshold in beat_thresholds:

    for option_name, compatibility in compatibility_options.items():

        selected = base_conditions.copy()

        selected &= (
            window_analysis_df["beat_target_coverage"]
            >= beat_threshold
        )

        if compatibility is not None:

            scale_min, chord_min = compatibility

            selected &= (
                window_analysis_df["window_scale_fit"]
                >= scale_min
            )

            selected &= (
                window_analysis_df["window_chord_fit"]
                >= chord_min
            )

        windows = window_analysis_df[selected]

        results.append({
            "Minimum valid beats": beat_threshold,
            "Compatibility": option_name,
            "Windows": len(windows),
            "Songs": windows["track_id"].nunique(),
            "Retention (%)": (
                100 * len(windows)
                / len(window_analysis_df)
            ),
            "Mean valid-beat coverage (%)": (
                100 * windows["beat_target_coverage"].mean()
                if len(windows) else 0
            )
        })


comparison_df = pd.DataFrame(results)


# -----------------------------------
# 4. Display results
# -----------------------------------

print("BEAT-TARGET COVERAGE COMPARISON")
print("--------------------------------")

display(
    comparison_df.round(2)
)


# -----------------------------------
# 5. Highlight our main candidates
# -----------------------------------

print("\nMAIN CONFIGURATIONS")
print("--------------------------------")

recommended_comparison = comparison_df[
    (
        comparison_df["Minimum valid beats"]
        .isin([0.70, 0.75, 0.80])
    )
    &
    (
        comparison_df["Compatibility"]
        == "Light: scale >= 0.60, chord >= 0.30"
    )
]

display(
    recommended_comparison.round(2)
)


print("\nFilter comparison completed!")

BEAT-TARGET COVERAGE COMPARISON
--------------------------------


,Minimum valid beats,Compatibility,Windows,Songs,Retention (%),Mean valid-beat coverage (%)
0,0.65,No compatibility filtering,10125,1088,82.01,87.77
1,0.65,"Light: scale >= 0.60, chord >= 0.30",9531,1082,77.20,87.79
2,0.65,"Moderate: scale >= 0.70, chord >= 0.40",8529,1056,69.08,87.83
3,0.70,No compatibility filtering,9874,1087,79.98,88.29
4,0.70,"Light: scale >= 0.60, chord >= 0.30",9293,1081,75.27,88.31
5,0.70,"Moderate: scale >= 0.70, chord >= 0.40",8316,1053,67.36,88.35
6,0.75,No compatibility filtering,9502,1082,76.96,88.92
7,0.75,"Light: scale >= 0.60, chord >= 0.30",8952,1076,72.51,88.93
8,0.75,"Moderate: scale >= 0.70, chord >= 0.40",8014,1045,64.91,88.96
9,0.80,No compatibility filtering,8483,1055,68.71,90.30



MAIN CONFIGURATIONS
--------------------------------


,Minimum valid beats,Compatibility,Windows,Songs,Retention (%),Mean valid-beat coverage (%)
4,0.70,"Light: scale >= 0.60, chord >= 0.30",9293,1081,75.27,88.31
7,0.75,"Light: scale >= 0.60, chord >= 0.30",8952,1076,72.51,88.93
10,0.80,"Light: scale >= 0.60, chord >= 0.30",8002,1047,64.81,90.29



Filter comparison completed!


Comparing beat-target coverage thresholds demonstrated that reducing the minimum reliable-beat requirement from 80% to 75% increases dataset retention from 8,002 to 8,952 windows. The selected windows represent 1,076 songs and maintain an average reliable-beat coverage of 88.93%. A 75% threshold was provisionally selected because unreliable individual beat targets can be masked during training.
The next stage will investigate genre-label availability, genre imbalance, and the distribution of training windows across genres before finalizing the Phase 3 dataset.

### Genre Metadata Analysis
Objective: Analyze genre-label availability and distribution in the provisionally selected Phase 3 dataset.
Genre metadata is obtained from the official MTG-Jamendo annotations and matched using track identifiers. Multiple genre labels per song are preserved.
The analysis measures songs, windows, and unique artists per genre to identify class imbalance and determine which genres have sufficient representation for genre-conditioned chord generation.
No genre filtering is applied at this stage.

In [27]:
import csv
from pathlib import Path
from collections import defaultdict

import pandas as pd


# -----------------------------------
# 1. Find MTG-Jamendo genre metadata
# -----------------------------------

GENRE_TSV = None

search_dirs = [
    PROJECT_ROOT / "datasets" / "mtg-jamendo",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT
]

for directory in search_dirs:

    if not directory.exists():
        continue

    matches = sorted(
        directory.rglob("autotagging_genre.tsv")
    )

    if matches:
        GENRE_TSV = matches[0]
        break

if GENRE_TSV is None:

    raise FileNotFoundError(
        "Could not find autotagging_genre.tsv. "
        "Download it from the official "
        "MTG-Jamendo metadata repository."
    )

print("Genre metadata:", GENRE_TSV)


# -----------------------------------
# 2. Read metadata
# -----------------------------------

metadata_records = []

with open(
    GENRE_TSV,
    "r",
    encoding="utf-8"
) as file:

    reader = csv.reader(
        file,
        delimiter="\t"
    )

    next(reader, None)

    for row in reader:

        if len(row) < 5:
            continue

        track_id = row[0].strip()
        artist_id = row[1].strip()

        genres = set()

        for tag in row[5:]:

            if tag.startswith("genre---"):

                names = tag.replace(
                    "genre---", "", 1
                ).split(",")

                genres.update(
                    name.strip().lower()
                    for name in names
                    if name.strip()
                )

        metadata_records.append({
            "track_id": track_id,
            "artist_id": artist_id,
            "genres": sorted(genres)
        })


genre_metadata_df = pd.DataFrame(
    metadata_records
).drop_duplicates("track_id")

print(
    "\nMetadata tracks:",
    len(genre_metadata_df)
)


# -----------------------------------
# 3. Recreate provisional selection
# -----------------------------------

selected_mask = (
    (window_analysis_df["note_count"] >= 20)
    &
    (window_analysis_df["melody_coverage"] >= 0.50)
    &
    (window_analysis_df["max_melody_gap_sec"] <= 8)
    &
    (window_analysis_df["beat_target_coverage"] >= 0.75)
    &
    (window_analysis_df["meaningful_roots"] >= 2)
    &
    (window_analysis_df["root_transitions"] >= 2)
    &
    (window_analysis_df["window_scale_fit"] >= 0.60)
    &
    (window_analysis_df["window_chord_fit"] >= 0.30)
)

selected_windows_df = window_analysis_df[
    selected_mask
].copy()

print("\nSELECTED DATASET")
print("--------------------------------")

print(
    "Windows:",
    len(selected_windows_df)
)

print(
    "Songs:",
    selected_windows_df["track_id"].nunique()
)


# -----------------------------------
# 4. Match genre metadata
# -----------------------------------

genre_windows_df = selected_windows_df.merge(
    genre_metadata_df,
    on="track_id",
    how="left",
    validate="many_to_one"
)

song_genres_df = genre_windows_df.drop_duplicates(
    "track_id"
).copy()

song_genres_df["has_genre"] = (
    song_genres_df["genres"].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
)

print("\nGENRE AVAILABILITY")
print("--------------------------------")

print(
    "Songs with genre:",
    song_genres_df["has_genre"].sum()
)

print(
    "Songs without genre:",
    (~song_genres_df["has_genre"]).sum()
)

print(
    "Songs with multiple genres:",
    song_genres_df["genres"].apply(
        lambda x:
            isinstance(x, list) and len(x) > 1
    ).sum()
)

print(
    "Unique artists:",
    song_genres_df["artist_id"].nunique()
)


# -----------------------------------
# 5. Genre distribution
# -----------------------------------

window_counts = (
    selected_windows_df["track_id"]
    .value_counts()
    .to_dict()
)

genre_stats = defaultdict(
    lambda: {
        "songs": 0,
        "windows": 0,
        "artists": set()
    }
)

for row in song_genres_df.itertuples():

    genres = (
        row.genres
        if isinstance(row.genres, list)
        else []
    )

    for genre in genres:

        genre_stats[genre]["songs"] += 1

        genre_stats[genre]["windows"] += (
            window_counts[row.track_id]
        )

        if pd.notna(row.artist_id):

            genre_stats[genre]["artists"].add(
                row.artist_id
            )


genre_summary_df = pd.DataFrame([
    {
        "Genre": genre,
        "Songs": info["songs"],
        "Windows": info["windows"],
        "Artists": len(info["artists"])
    }

    for genre, info in genre_stats.items()
]).sort_values(
    "Songs",
    ascending=False
).reset_index(drop=True)


# -----------------------------------
# 6. Display distribution
# -----------------------------------

print("\nGENRE DISTRIBUTION")
print("--------------------------------")

print(
    "Distinct genre tags:",
    len(genre_summary_df)
)

display(
    genre_summary_df.head(30)
)


# -----------------------------------
# 7. Genre representation
# -----------------------------------

print("\nGENRE REPRESENTATION")
print("--------------------------------")

for minimum in [10, 25, 50, 100]:

    count = (
        genre_summary_df["Songs"] >= minimum
    ).sum()

    print(
        f"Genres with >= {minimum} songs: {count}"
    )


print("\nGenre analysis completed!")

Genre metadata: /Users/praise/Developer/vocal-harmonization/datasets/mtg-jamendo/annotations/autotagging_genre.tsv

Metadata tracks: 55215

SELECTED DATASET
--------------------------------
Windows: 8952
Songs: 1076

GENRE AVAILABILITY
--------------------------------
Songs with genre: 1069
Songs without genre: 7
Songs with multiple genres: 946
Unique artists: 218

GENRE DISTRIBUTION
--------------------------------
Distinct genre tags: 90


,Genre,Songs,Windows,Artists
0,rock,441,3952,95
1,indie,279,2457,38
2,pop,265,2130,78
3,alternative,244,2389,40
4,folk,145,1295,33
5,poprock,116,898,36
6,electronic,112,906,50
7,popfolk,105,821,22
8,punkrock,71,807,10
9,soundtrack,70,471,28



GENRE REPRESENTATION
--------------------------------
Genres with >= 10 songs: 46
Genres with >= 25 songs: 24
Genres with >= 50 songs: 17
Genres with >= 100 songs: 8

Genre analysis completed!


### Findings from Step 3.22
Genre metadata analysis identified 1,069 genre-annotated songs among the 1,076 provisionally selected tracks, representing 99.35% metadata coverage.
The dataset contains 90 distinct genre tags, with 946 songs having multiple labels. Rock and pop are the most strongly represented genres, while jazz, blues, and several other styles contain fewer songs and artists.
These findings indicate that genre-conditioned learning is feasible, but genre imbalance, overlapping labels, and limited artist diversity must be considered before defining the final conditioning vocabulary.
The next experiment examines genre co-occurrence and artist concentration to identify suitable genre groupings.

### Genre Overlap and Artist Diversity
Objective: Investigate relationships between MTG-Jamendo genre tags and assess whether the available examples represent sufficiently diverse artists.
The experiment identifies frequently co-occurring genre labels, examines the number of labels assigned to each song, and measures artist concentration within major genres.
These findings will inform the selection of a smaller, musically meaningful set of genre conditions while preserving the original multi-label annotations.

In [28]:
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Songs with valid genre labels
# -----------------------------------

valid_genre_songs = song_genres_df[
    song_genres_df["has_genre"]
].copy()


# -----------------------------------
# 2. Number of genre labels per song
# -----------------------------------

valid_genre_songs["genre_count"] = (
    valid_genre_songs["genres"].apply(len)
)

print("GENRE LABELS PER SONG")
print("--------------------------------")

display(
    valid_genre_songs["genre_count"]
    .describe()
    .round(2)
)

print("\nDistribution:")

display(
    valid_genre_songs["genre_count"]
    .value_counts()
    .sort_index()
    .head(15)
)


# -----------------------------------
# 3. Find common genre combinations
# -----------------------------------

genre_pairs = Counter()

for genres in valid_genre_songs["genres"]:

    for pair in combinations(
        sorted(set(genres)), 2
    ):

        genre_pairs[pair] += 1


pair_records = []

for (genre_a, genre_b), count in genre_pairs.items():

    pair_records.append({
        "Genre A": genre_a,
        "Genre B": genre_b,
        "Songs together": count
    })


genre_pairs_df = pd.DataFrame(
    pair_records
).sort_values(
    "Songs together",
    ascending=False
)


print("\nMOST COMMON GENRE PAIRS")
print("--------------------------------")

display(
    genre_pairs_df.head(25)
)


# -----------------------------------
# 4. Calculate artist concentration
# -----------------------------------

artist_records = []

top_genres = genre_summary_df[
    "Genre"
].head(20).tolist()

for genre in top_genres:

    genre_songs = valid_genre_songs[
        valid_genre_songs["genres"].apply(
            lambda genres: genre in genres
        )
    ]

    artist_counts = (
        genre_songs["artist_id"]
        .value_counts()
    )

    total_songs = len(genre_songs)

    top_artist_share = (
        artist_counts.iloc[0] / total_songs
        if len(artist_counts)
        else np.nan
    )

    top_three_share = (
        artist_counts.head(3).sum()
        / total_songs
        if len(artist_counts)
        else np.nan
    )

    artist_records.append({
        "Genre": genre,
        "Songs": total_songs,
        "Artists": genre_songs["artist_id"].nunique(),
        "Largest artist share (%)":
            100 * top_artist_share,
        "Top 3 artists share (%)":
            100 * top_three_share
    })


artist_diversity_df = pd.DataFrame(
    artist_records
)

print("\nARTIST DIVERSITY BY GENRE")
print("--------------------------------")

display(
    artist_diversity_df.round(2)
)


# -----------------------------------
# 5. Examine overlap for key genres
# -----------------------------------

print("\nOVERLAP OF IMPORTANT GENRES")
print("--------------------------------")

focus_genres = [
    "pop",
    "rock",
    "jazz",
    "blues",
    "electronic",
    "folk",
    "classical"
]

for focus_genre in focus_genres:

    matching = valid_genre_songs[
        valid_genre_songs["genres"].apply(
            lambda genres: focus_genre in genres
        )
    ]

    other_genres = Counter()

    for genres in matching["genres"]:

        other_genres.update(
            genre
            for genre in genres
            if genre != focus_genre
        )

    print(
        f"\n{focus_genre.upper()} "
        f"({len(matching)} songs)"
    )

    print(
        "Common accompanying labels:",
        other_genres.most_common(5)
    )


# -----------------------------------
# 6. Songs missing genre metadata
# -----------------------------------

print("\nSONGS WITHOUT GENRE LABELS")
print("--------------------------------")

display(
    song_genres_df[
        ~song_genres_df["has_genre"]
    ][
        ["track_id", "artist_id"]
    ]
)


print("\nGenre overlap analysis completed!")

GENRE LABELS PER SONG
--------------------------------


count    1069.00
mean        2.87
std         1.16
min         1.00
25%         2.00
50%         3.00
75%         4.00
max         6.00
Name: genre_count, dtype: float64


Distribution:


genre_count
1    123
2    316
3    315
4    220
5     86
6      9
Name: count, dtype: int64


MOST COMMON GENRE PAIRS
--------------------------------


...


ARTIST DIVERSITY BY GENRE
--------------------------------


,Genre,Songs,Artists,Largest artist share (%),Top 3 artists share (%)
0,rock,441,95,8.39,18.37
1,indie,279,38,10.75,27.60
2,pop,265,78,8.30,19.25
3,alternative,244,40,15.16,27.05
4,folk,145,33,13.79,32.41
5,poprock,116,36,18.97,37.93
6,electronic,112,50,9.82,20.54
7,popfolk,105,22,19.05,50.48
8,punkrock,71,10,52.11,74.65
9,soundtrack,70,28,17.14,37.14



OVERLAP OF IMPORTANT GENRES
--------------------------------

POP (265 songs)
Common accompanying labels: [('rock', 139), ('indie', 66), ('alternative', 65), ('poprock', 44), ('popfolk', 27)]

ROCK (441 songs)
Common accompanying labels: [('indie', 192), ('alternative', 170), ('pop', 139), ('poprock', 70), ('punkrock', 65)]

JAZZ (70 songs)
Common accompanying labels: [('hiphop', 33), ('funk', 32), ('soul', 27), ('pop', 13), ('fusion', 10)]

BLUES (63 songs)
Common accompanying labels: [('rock', 32), ('pop', 24), ('alternative', 15), ('indie', 12), ('soul', 11)]

ELECTRONIC (112 songs)
Common accompanying labels: [('rock', 20), ('pop', 18), ('ambient', 18), ('experimental', 18), ('house', 17)]

FOLK (145 songs)
Common accompanying labels: [('popfolk', 73), ('pop', 25), ('rock', 25), ('indie', 19), ('poprock', 17)]

CLASSICAL (67 songs)
Common accompanying labels: [('soundtrack', 12), ('orchestral', 9), ('ambient', 8), ('popfolk', 6), ('poprock', 4)]

SONGS WITHOUT GENRE LABELS
-------

,track_id,artist_id
540,track_0079690,NaN
571,track_0156873,NaN
574,track_0156888,NaN
1213,track_0480340,NaN
3930,track_1045346,NaN
5252,track_1082101,NaN
5254,track_1082102,NaN



Genre overlap analysis completed!


### Findings from Step 3.23
Genre analysis revealed substantial overlap between several MTG-Jamendo labels, particularly rock, indie, and alternative. Of the 265 pop songs, 139 are also labeled rock, demonstrating that genre annotations are frequently non-exclusive.
Five provisional genre conditions were selected: Rock, Pop, Electronic, Folk, and Jazz. The selection emphasizes musical diversity, dataset availability, and research relevance.
A multi-label genre representation is proposed to preserve overlapping genre annotations without duplicating identical melody–chord examples under potentially conflicting individual labels.
Further analysis will determine whether the five selected genres provide sufficient independent examples and whether their harmonic characteristics differ meaningfully.

### Five-Genre Overlap Analysis
Objective: Evaluate the coverage, independence, and overlap of five provisional genre conditions.
Songs are represented using a five-dimensional multi-hot genre vector. Genre-specific song and window counts, exclusive examples, artist diversity, and pairwise co-occurrence percentages are measured.
This analysis will identify highly overlapping genre pairs and determine how much of the selected dataset remains available when restricting genre conditioning to five categories.

In [29]:
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Five provisional genres
# -----------------------------------

FIVE_GENRES = [
    "rock",
    "pop",
    "electronic",
    "folk",
    "jazz"
]


# -----------------------------------
# 2. Prepare five-genre labels
# -----------------------------------

five_genre_df = song_genres_df.copy()

five_genre_df["selected_genres"] = (
    five_genre_df["genres"].apply(
        lambda tags: [
            genre
            for genre in FIVE_GENRES
            if isinstance(tags, list)
            and genre in tags
        ]
    )
)

five_genre_df["selected_genre_count"] = (
    five_genre_df["selected_genres"].apply(len)
)

five_genre_df["window_count"] = (
    five_genre_df["track_id"]
    .map(window_counts)
    .fillna(0)
    .astype(int)
)


# -----------------------------------
# 3. Coverage of selected genres
# -----------------------------------

selected_songs = five_genre_df[
    five_genre_df["selected_genre_count"] > 0
].copy()

excluded_songs = five_genre_df[
    five_genre_df["selected_genre_count"] == 0
].copy()

print("FIVE-GENRE DATASET COVERAGE")
print("--------------------------------")

print("Original songs:", len(five_genre_df))

print("Songs with selected genres:", len(selected_songs))

print("Songs without selected genres:", len(excluded_songs))

print(
    "Windows with selected genres:",
    selected_songs["window_count"].sum()
)

print(
    "Windows without selected genres:",
    excluded_songs["window_count"].sum()
)

print(
    "Unique artists:",
    selected_songs["artist_id"].nunique()
)


# -----------------------------------
# 4. Number of selected genres per song
# -----------------------------------

print("\nSELECTED GENRES PER SONG")
print("--------------------------------")

display(
    five_genre_df["selected_genre_count"]
    .value_counts()
    .sort_index()
    .rename_axis("Selected genres")
    .reset_index(name="Songs")
)


# -----------------------------------
# 5. Individual genre statistics
# -----------------------------------

genre_records = []

for genre in FIVE_GENRES:

    matching = selected_songs[
        selected_songs["selected_genres"].apply(
            lambda labels: genre in labels
        )
    ]

    exclusive = matching[
        matching["selected_genre_count"] == 1
    ]

    artist_counts = matching["artist_id"].value_counts()

    genre_records.append({

        "Genre": genre,

        "Songs": len(matching),

        "Windows": matching["window_count"].sum(),

        "Artists": matching["artist_id"].nunique(),

        "Exclusive songs": len(exclusive),

        "Exclusive windows":
            exclusive["window_count"].sum(),

        "Exclusive (%)":
            100 * len(exclusive) / len(matching)
            if len(matching) else 0,

        "Largest artist (%)":
            100 * artist_counts.iloc[0] / len(matching)
            if len(artist_counts) else 0
    })


five_genre_summary_df = pd.DataFrame(
    genre_records
)

print("\nINDIVIDUAL GENRE STATISTICS")
print("--------------------------------")

display(
    five_genre_summary_df.round(2)
)


# -----------------------------------
# 6. Pairwise genre overlap
# -----------------------------------

overlap_records = []

for genre_a, genre_b in combinations(
    FIVE_GENRES, 2
):

    songs_a = set(
        selected_songs.loc[
            selected_songs["selected_genres"].apply(
                lambda labels: genre_a in labels
            ),
            "track_id"
        ]
    )

    songs_b = set(
        selected_songs.loc[
            selected_songs["selected_genres"].apply(
                lambda labels: genre_b in labels
            ),
            "track_id"
        ]
    )

    shared = songs_a & songs_b
    union = songs_a | songs_b

    overlap_records.append({

        "Genre A": genre_a,
        "Genre B": genre_b,

        "Shared songs": len(shared),

        "% of Genre A":
            100 * len(shared) / len(songs_a)
            if songs_a else 0,

        "% of Genre B":
            100 * len(shared) / len(songs_b)
            if songs_b else 0,

        "Jaccard (%)":
            100 * len(shared) / len(union)
            if union else 0
    })


genre_overlap_df = pd.DataFrame(
    overlap_records
).sort_values(
    "Jaccard (%)",
    ascending=False
)

print("\nPAIRWISE GENRE OVERLAP")
print("--------------------------------")

display(
    genre_overlap_df.round(2)
)


# -----------------------------------
# 7. Most common genre combinations
# -----------------------------------

combination_counts = Counter(
    tuple(labels)
    for labels in selected_songs["selected_genres"]
)

combination_df = pd.DataFrame([
    {
        "Genre combination": " + ".join(labels),
        "Songs": count
    }
    for labels, count in combination_counts.items()
]).sort_values(
    "Songs",
    ascending=False
)

print("\nMOST COMMON GENRE COMBINATIONS")
print("--------------------------------")

display(
    combination_df.head(15)
)


# -----------------------------------
# 8. Examine excluded songs
# -----------------------------------

other_genres = Counter()

for labels in excluded_songs["genres"]:

    if isinstance(labels, list):
        other_genres.update(labels)

print("\nGENRES OUTSIDE OUR FIVE")
print("--------------------------------")

print(
    "Most common excluded-song labels:"
)

display(
    pd.DataFrame(
        other_genres.most_common(15),
        columns=["Genre", "Songs"]
    )
)


print("\nFive-genre overlap analysis completed!")

FIVE-GENRE DATASET COVERAGE
--------------------------------
Original songs: 1076
Songs with selected genres: 808
Songs without selected genres: 268
Windows with selected genres: 6896
Windows without selected genres: 2056
Unique artists: 172

SELECTED GENRES PER SONG
--------------------------------


,Selected genres,Songs
0,0,268
1,1,600
2,2,191
3,3,17



INDIVIDUAL GENRE STATISTICS
--------------------------------


,Genre,Songs,Windows,Artists,Exclusive songs,Exclusive windows,Exclusive (%),Largest artist (%)
0,rock,441,3952,95,274,2584,62.13,8.39
1,pop,265,2130,78,87,679,32.83,8.30
2,electronic,112,906,50,84,704,75.00,9.82
3,folk,145,1295,33,100,923,68.97,13.79
4,jazz,70,414,14,55,315,78.57,44.29



PAIRWISE GENRE OVERLAP
--------------------------------


...


MOST COMMON GENRE COMBINATIONS
--------------------------------


,Genre combination,Songs
5,rock,274
2,rock + pop,122
8,folk,100
3,pop,87
4,electronic,84
7,jazz,55
0,pop + folk,20
10,rock + folk,20
9,pop + jazz,13
1,rock + pop + electronic,12



GENRES OUTSIDE OUR FIVE
--------------------------------
Most common excluded-song labels:


,Genre,Songs
0,classical,60
1,indie,52
2,alternative,48
3,soundtrack,39
4,experimental,33
5,metal,27
6,hiphop,22
7,popfolk,20
8,ambient,19
9,blues,19



Five-genre overlap analysis completed!


### Findings from Step 3.24
Five-genre overlap analysis identified 808 songs belonging to at least one selected genre, including 600 songs with exactly one selected genre label. Rock and pop exhibited moderate overlap, whereas electronic, folk, and jazz were relatively distinct.
All 1,076 songs and 8,952 windows remain available for training. Songs outside the selected genre vocabulary will be retained for unconditional training and represented using an explicit unknown-genre condition when appropriate.
The five provisional genres are Rock, Pop, Electronic, Folk, and Jazz. Jazz requires particular attention because of its limited artist diversity.
The next experiment will compare chord-quality distributions across these genres to evaluate whether the labels correspond to observable differences in harmony.

### Genre-Specific Chord Distributions
Objective: Compare the distribution of chord qualities across the five selected genres.
The experiment examines major, minor, dominant-seventh, major-seventh, minor-seventh, and other chord qualities. Chord durations are measured only within the provisionally selected training windows, without double-counting overlapping portions.
Each song's chord-quality distribution is normalized before calculating genre averages, preventing longer songs from automatically receiving greater weight.
Results are reported for both all genre-matched songs and songs exclusive to one of the five selected genres.

In [30]:
from collections import defaultdict

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Chord quality categories
# -----------------------------------

QUALITY_MAP = {
    "maj": "Major",
    "min": "Minor",
    "7": "Dominant 7",
    "maj7": "Major 7",
    "min7": "Minor 7"
}

QUALITY_CLASSES = [
    "Major",
    "Minor",
    "Dominant 7",
    "Major 7",
    "Minor 7",
    "Other"
]


# -----------------------------------
# 2. Merge overlapping window intervals
# -----------------------------------

def merge_time_intervals(intervals):

    intervals = sorted(intervals)

    merged = []

    for start, end in intervals:

        if end <= start:
            continue

        if merged and start <= merged[-1][1]:

            merged[-1] = (
                merged[-1][0],
                max(merged[-1][1], end)
            )

        else:
            merged.append((start, end))

    return merged


def duration_inside_intervals(
    start, end, intervals
):

    return sum(
        max(
            0.0,
            min(end, b) - max(start, a)
        )
        for a, b in intervals
        if a < end and b > start
    )


# -----------------------------------
# 3. Calculate each song's
#    chord-quality distribution
# -----------------------------------

song_quality_records = []

genre_song_lookup = five_genre_df.set_index(
    "track_id"
)

for track_id, group in selected_windows_df.groupby(
    "track_id"
):

    folder = track_lookup[track_id]

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    intervals = merge_time_intervals(
        list(
            zip(
                group["start_sec"],
                group["end_sec"]
            )
        )
    )

    durations = defaultdict(float)

    for event in chord_data.get("events", []):

        if root_to_pc(event.get("root")) is None:
            continue

        try:
            start = float(event["start_sec"])
            end = float(event["end_sec"])
        except (KeyError, TypeError, ValueError):
            continue

        if not np.isfinite([start, end]).all():
            continue

        if end <= start:
            continue

        overlap = duration_inside_intervals(
            start, end, intervals
        )

        if overlap <= 0:
            continue

        quality = QUALITY_MAP.get(
            event.get("quality"),
            "Other"
        )

        durations[quality] += overlap

    total_duration = sum(durations.values())

    if total_duration <= 0:
        continue

    metadata = genre_song_lookup.loc[track_id]

    song_quality_records.append({

        "track_id": track_id,

        "artist_id": metadata["artist_id"],

        "selected_genres":
            metadata["selected_genres"],

        "selected_genre_count":
            metadata["selected_genre_count"],

        "annotated_duration_sec":
            total_duration,

        **{
            quality:
                durations[quality] / total_duration
            for quality in QUALITY_CLASSES
        }
    })


song_quality_df = pd.DataFrame(
    song_quality_records
)


print("SONG-LEVEL CHORD QUALITY ANALYSIS")
print("--------------------------------")

print(
    "Songs analyzed:",
    len(song_quality_df)
)

print(
    "Songs without usable chord duration:",
    len(selected_windows_df["track_id"].unique())
    - len(song_quality_df)
)


# -----------------------------------
# 4. Calculate genre profiles
# -----------------------------------

def calculate_genre_profiles(
    exclusive_only=False,
    artist_balanced=False
):

    records = []

    for genre in FIVE_GENRES:

        matching = song_quality_df[
            song_quality_df["selected_genres"].apply(
                lambda labels: genre in labels
            )
        ]

        if exclusive_only:

            matching = matching[
                matching["selected_genre_count"] == 1
            ]

        if matching.empty:
            continue

        # Every song contributes equally initially.
        distributions = matching[
            QUALITY_CLASSES
        ]

        # Optional: average within each artist first,
        # then average the artists equally.
        if artist_balanced:

            distributions = (
                matching.dropna(
                    subset=["artist_id"]
                )
                .groupby("artist_id")[
                    QUALITY_CLASSES
                ]
                .mean()
            )

        averages = distributions.mean() * 100

        records.append({

            "Genre": genre,

            "Songs": len(matching),

            "Artists":
                matching["artist_id"].nunique(),

            **{
                quality: averages[quality]
                for quality in QUALITY_CLASSES
            }
        })

    return pd.DataFrame(records)


# -----------------------------------
# 5. All songs matching each genre
# -----------------------------------

print("\nALL GENRE-MATCHED SONGS")
print("--------------------------------")

genre_chord_all_df = calculate_genre_profiles()

display(
    genre_chord_all_df.round(2)
)


# -----------------------------------
# 6. Exclusive genre songs
# -----------------------------------

print("\nEXCLUSIVE FIVE-GENRE SONGS")
print("--------------------------------")

genre_chord_exclusive_df = (
    calculate_genre_profiles(
        exclusive_only=True
    )
)

display(
    genre_chord_exclusive_df.round(2)
)


# -----------------------------------
# 7. Artist-balanced distributions
# -----------------------------------

print("\nARTIST-BALANCED GENRE PROFILES")
print("--------------------------------")

genre_chord_artist_df = (
    calculate_genre_profiles(
        artist_balanced=True
    )
)

display(
    genre_chord_artist_df.round(2)
)


print("\nGenre chord analysis completed!")

SONG-LEVEL CHORD QUALITY ANALYSIS
--------------------------------
Songs analyzed: 1076
Songs without usable chord duration: 0

ALL GENRE-MATCHED SONGS
--------------------------------


,Genre,Songs,Artists,Major,Minor,Dominant 7,Major 7,Minor 7,Other
0,rock,441,95,61.17,32.97,2.29,0.97,2.20,0.40
1,pop,265,78,61.12,27.70,3.01,1.82,5.82,0.52
2,electronic,112,50,44.63,39.22,2.48,4.16,8.09,1.41
3,folk,145,33,58.71,33.65,3.43,1.07,2.80,0.34
4,jazz,70,14,25.13,32.01,12.96,5.34,22.81,1.76



EXCLUSIVE FIVE-GENRE SONGS
--------------------------------


,Genre,Songs,Artists,Major,Minor,Dominant 7,Major 7,Minor 7,Other
0,rock,274,59,59.34,35.98,1.60,0.93,1.70,0.44
1,pop,87,37,55.18,25.87,3.46,3.30,11.45,0.75
2,electronic,84,36,40.55,40.11,3.10,5.06,9.46,1.71
3,folk,100,21,56.85,37.06,2.50,1.15,2.06,0.37
4,jazz,55,11,21.78,30.72,14.31,5.53,25.84,1.82



ARTIST-BALANCED GENRE PROFILES
--------------------------------


,Genre,Songs,Artists,Major,Minor,Dominant 7,Major 7,Minor 7,Other
0,rock,441,95,59.51,33.13,2.83,1.07,3.02,0.44
1,pop,265,78,58.77,29.26,3.11,2.71,5.76,0.39
2,electronic,112,50,46.27,37.02,2.70,4.24,8.63,1.14
3,folk,145,33,61.73,29.88,3.22,1.49,3.46,0.22
4,jazz,70,14,26.06,26.62,22.29,4.11,19.90,1.03



Genre chord analysis completed!


### Findings from Step 3.25
Genre-specific chord analysis revealed differences in harmonic vocabulary across the five provisional genres. Jazz exhibited substantially higher proportions of dominant-seventh and minor-seventh chords than rock, pop, and folk. Electronic music also showed increased use of seventh chords relative to rock.
Artist-balanced analysis confirmed that these differences were not solely caused by songs from the most represented artists. However, rock and folk displayed similar overall chord-quality distributions.
These results provide preliminary evidence supporting genre-conditioned harmonization. Since the proposed Transformer predicts coarse harmonic progressions rather than detailed chord realizations, the next experiment examines normalized chord-root transitions to determine whether genre differences also exist in progression structure.

### Genre-Specific Root Transitions
Objective: Investigate whether the five selected genres exhibit different chord-root progression patterns.
Chord roots are normalized relative to each song's estimated major-scale reference. Consecutive chord-root changes are extracted from the selected musical sections.
The analysis compares root movements and common consecutive root pairs across genres. Each song contributes a normalized transition distribution to reduce bias from differences in song length.
Artist-balanced distributions are also calculated to investigate whether observed differences are dominated by individual artists.

In [31]:
from collections import Counter
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Configuration
# -----------------------------------

MAX_CHORD_GAP = 0.5

ROOT_INTERVALS = list(range(1, 12))


# -----------------------------------
# 2. Extract chord-root transitions
# -----------------------------------

def get_song_root_transitions(track_id, intervals):

    folder = track_lookup[track_id]

    chord_data = read_json(
        folder / "chords/submission.json"
    )

    key_pc = get_major_pc(
        key_lookup[track_id]
    )

    events = []

    for event in chord_data.get("events", []):

        root = root_to_pc(
            event.get("root")
        )

        if root is None:
            continue

        try:
            start = float(event["start_sec"])
            end = float(event["end_sec"])
        except (KeyError, ValueError, TypeError):
            continue

        if not np.isfinite([start, end]).all():
            continue

        if end <= start:
            continue

        events.append({
            "start": start,
            "end": end,
            "root": root
        })

    events.sort(key=lambda x: x["start"])

    interval_counts = Counter()
    pair_counts = Counter()

    # Process separate selected regions independently.
    # This prevents artificial transitions across
    # discarded portions of a song.

    for region_start, region_end in intervals:

        previous_root = None
        previous_end = None

        for chord in events:

            if chord["start"] >= region_end:
                break

            if chord["end"] <= region_start:
                continue

            start = max(
                region_start,
                chord["start"]
            )

            end = min(
                region_end,
                chord["end"]
            )

            if end <= start:
                continue

            root = chord["root"]

            if previous_root is not None:

                gap = start - previous_end

                if gap <= MAX_CHORD_GAP:

                    movement = (
                        root - previous_root
                    ) % 12

                    # Count changes in root only.
                    if movement != 0:

                        interval_counts[movement] += 1

                        relative_previous = (
                            previous_root - key_pc
                        ) % 12

                        relative_current = (
                            root - key_pc
                        ) % 12

                        pair_counts[
                            (
                                relative_previous,
                                relative_current
                            )
                        ] += 1

            previous_root = root
            previous_end = end

    return interval_counts, pair_counts


# -----------------------------------
# 3. Analyze all selected songs
# -----------------------------------

transition_records = []

for track_id, group in selected_windows_df.groupby(
    "track_id"
):

    selected_intervals = merge_time_intervals(
        list(
            zip(
                group["start_sec"],
                group["end_sec"]
            )
        )
    )

    interval_counts, pair_counts = (
        get_song_root_transitions(
            track_id,
            selected_intervals
        )
    )

    total = sum(interval_counts.values())

    metadata = genre_song_lookup.loc[track_id]

    transition_records.append({

        "track_id": track_id,

        "artist_id": metadata["artist_id"],

        "selected_genres":
            metadata["selected_genres"],

        "selected_genre_count":
            metadata["selected_genre_count"],

        "total_transitions": total,

        "interval_distribution": {
            interval:
                interval_counts[interval] / total
            for interval in ROOT_INTERVALS
        } if total > 0 else {},

        "pair_distribution": {
            pair: count / total
            for pair, count in pair_counts.items()
        } if total > 0 else {}
    })


transition_df = pd.DataFrame(
    transition_records
)


# -----------------------------------
# 4. Overview
# -----------------------------------

print("CHORD-ROOT TRANSITION ANALYSIS")
print("--------------------------------")

print(
    "Songs analyzed:",
    len(transition_df)
)

print(
    "Songs with no root transitions:",
    (transition_df["total_transitions"] == 0).sum()
)

display(
    transition_df["total_transitions"]
    .describe()
    .round(2)
)


# -----------------------------------
# 5. Calculate genre-level distributions
# -----------------------------------

def genre_interval_profile(
    genre,
    artist_balanced=False
):

    matching = transition_df[
        transition_df["selected_genres"].apply(
            lambda labels: genre in labels
        )
        &
        (transition_df["total_transitions"] > 0)
    ].copy()

    if matching.empty:
        return np.zeros(11)

    vectors = np.array([
        [
            distribution.get(interval, 0.0)
            for interval in ROOT_INTERVALS
        ]
        for distribution
        in matching["interval_distribution"]
    ])

    if not artist_balanced:
        return vectors.mean(axis=0)

    # Average songs within each artist first,
    # then give every artist equal weight.

    artist_vectors = []

    for artist_id in matching["artist_id"].dropna().unique():

        indices = (
            matching["artist_id"] == artist_id
        ).to_numpy()

        artist_vectors.append(
            vectors[indices].mean(axis=0)
        )

    if not artist_vectors:
        return vectors.mean(axis=0)

    return np.mean(artist_vectors, axis=0)


# -----------------------------------
# 6. Root-movement distribution
# -----------------------------------

profiles = {}

for genre in FIVE_GENRES:

    profiles[genre] = genre_interval_profile(
        genre,
        artist_balanced=True
    )


movement_df = pd.DataFrame(
    {
        genre: profile * 100
        for genre, profile in profiles.items()
    },
    index=[
        f"{interval} semitones"
        for interval in ROOT_INTERVALS
    ]
).T


print("\nARTIST-BALANCED ROOT MOVEMENTS (%)")
print("--------------------------------")

display(
    movement_df.round(2)
)


# -----------------------------------
# 7. Most common root movements
# -----------------------------------

print("\nTOP ROOT MOVEMENTS BY GENRE")
print("--------------------------------")

for genre in FIVE_GENRES:

    profile = profiles[genre]

    top_indices = np.argsort(
        profile
    )[::-1][:5]

    print(f"\n{genre.upper()}")

    for index in top_indices:

        interval = ROOT_INTERVALS[index]

        print(
            f"  {interval} semitones: "
            f"{100 * profile[index]:.2f}%"
        )


# -----------------------------------
# 8. Compare actual normalized
#    root-to-root pairs
# -----------------------------------

def average_pair_distribution(genre):

    matching = transition_df[
        transition_df["selected_genres"].apply(
            lambda labels: genre in labels
        )
        &
        (transition_df["total_transitions"] > 0)
    ]

    if matching.empty:
        return Counter()

    combined = Counter()

    for distribution in matching["pair_distribution"]:

        combined.update(distribution)

    # Equal weight per song.
    for pair in combined:
        combined[pair] /= len(matching)

    return combined


print("\nCOMMON KEY-NORMALIZED ROOT PAIRS")
print("--------------------------------")

for genre in FIVE_GENRES:

    pairs = average_pair_distribution(
        genre
    )

    print(f"\n{genre.upper()}")

    for (root_a, root_b), frequency in (
        pairs.most_common(8)
    ):

        print(
            f"  {PC_NAMES[root_a]} → "
            f"{PC_NAMES[root_b]}: "
            f"{100 * frequency:.2f}%"
        )


# -----------------------------------
# 9. Quantify distribution differences
# -----------------------------------

def total_variation_distance(p, q):

    return 0.5 * np.abs(p - q).sum()


distance_records = []

for genre_a, genre_b in combinations(
    FIVE_GENRES, 2
):

    distance = total_variation_distance(
        profiles[genre_a],
        profiles[genre_b]
    )

    distance_records.append({

        "Genre A": genre_a,
        "Genre B": genre_b,

        "Root-movement distance":
            distance
    })


genre_movement_distance_df = pd.DataFrame(
    distance_records
).sort_values(
    "Root-movement distance",
    ascending=False
)


print("\nGENRE ROOT-MOVEMENT DIFFERENCES")
print("--------------------------------")

display(
    genre_movement_distance_df.round(3)
)


print("\nGenre progression analysis completed!")

CHORD-ROOT TRANSITION ANALYSIS
--------------------------------
Songs analyzed: 1076
Songs with no root transitions: 0


count    1076.00
mean       59.53
std        34.76
min         2.00
25%        33.00
50%        55.00
75%        80.00
max       210.00
Name: total_transitions, dtype: float64


ARTIST-BALANCED ROOT MOVEMENTS (%)
--------------------------------


,1 semitones,2 semitones,3 semitones,4 semitones,5 semitones,6 semitones,7 semitones,8 semitones,9 semitones,10 semitones,11 semitones
rock,2.35,16.20,7.80,6.14,16.25,0.83,18.78,7.96,7.55,13.46,2.69
pop,3.20,17.31,6.86,5.57,19.07,0.57,18.54,7.70,7.04,11.31,2.82
electronic,2.53,12.25,8.04,6.70,18.46,0.37,22.24,7.50,6.68,12.02,3.21
folk,1.64,15.89,6.41,3.29,21.64,0.25,22.88,8.22,7.34,10.41,2.05
jazz,5.48,10.55,9.79,3.04,27.00,1.61,14.59,3.98,8.32,7.05,8.58



TOP ROOT MOVEMENTS BY GENRE
--------------------------------

ROCK
  7 semitones: 18.78%
  5 semitones: 16.25%
  2 semitones: 16.20%
  10 semitones: 13.46%
  8 semitones: 7.96%

POP
  5 semitones: 19.07%
  7 semitones: 18.54%
  2 semitones: 17.31%
  10 semitones: 11.31%
  8 semitones: 7.70%

ELECTRONIC
  7 semitones: 22.24%
  5 semitones: 18.46%
  2 semitones: 12.25%
  10 semitones: 12.02%
  3 semitones: 8.04%

FOLK
  7 semitones: 22.88%
  5 semitones: 21.64%
  2 semitones: 15.89%
  10 semitones: 10.41%
  8 semitones: 8.22%

JAZZ
  5 semitones: 27.00%
  7 semitones: 14.59%
  2 semitones: 10.55%
  3 semitones: 9.79%
  11 semitones: 8.58%

COMMON KEY-NORMALIZED ROOT PAIRS
--------------------------------

ROCK
  A → F: 5.93%
  F → G: 5.57%
  F → C: 5.56%
  C → G: 5.31%
  G → A: 5.17%
  G → C: 4.59%
  G → F: 3.97%
  A → G: 3.83%

POP
  F → C: 6.10%
  G → C: 5.59%
  F → G: 5.30%
  C → G: 5.14%
  A → F: 4.87%
  C → F: 4.73%
  G → A: 4.71%
  G → F: 4.00%

ELECTRONIC
  A → F: 6.80%
  D → A: 

,Genre A,Genre B,Root-movement distance
3,rock,jazz,0.233
8,electronic,jazz,0.215
9,folk,jazz,0.215
6,pop,jazz,0.212
2,rock,folk,0.097
7,electronic,folk,0.088
5,pop,folk,0.077
1,rock,electronic,0.072
4,pop,electronic,0.071
0,rock,pop,0.049



Genre progression analysis completed!


### Artist-Disjoint Dataset Splitting
Objective: Construct reproducible, artist-disjoint training, validation, and test partitions for the selected 16-bar windows.
An approximate 70:15:15 split is targeted. All songs and windows belonging to the same identified artist are assigned to the same partition.
Candidate splits are compared using genre representation and overall window counts. The split is selected using metadata only, without considering model performance.
Songs without verified artist identifiers are temporarily restricted to the training partition, pending further identity and duplicate-recording checks.

In [32]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit


# -----------------------------------
# 1. Prepare song-level metadata
# -----------------------------------

split_metadata_df = five_genre_df[
    [
        "track_id",
        "artist_id",
        "selected_genres",
        "window_count"
    ]
].copy()

split_metadata_df["artist_id"] = (
    split_metadata_df["artist_id"].replace(
        "", np.nan
    )
)

known = split_metadata_df[
    split_metadata_df["artist_id"].notna()
].reset_index(drop=True)

unknown = split_metadata_df[
    split_metadata_df["artist_id"].isna()
].copy()

print("ARTIST METADATA")
print("--------------------------------")

print("Known-artist songs:", len(known))
print("Unknown-artist songs:", len(unknown))

print(
    "Unique identified artists:",
    known["artist_id"].nunique()
)


# -----------------------------------
# 2. Generate candidate grouped splits
# -----------------------------------

SPLITS = ["train", "val", "test"]

TARGET_RATIOS = np.array([
    0.70, 0.15, 0.15
])

genre_masks = {
    genre: known["selected_genres"].apply(
        lambda labels: genre in labels
    ).to_numpy()
    for genre in FIVE_GENRES
}

best_score = float("inf")
best_assignments = None
best_seed = None

unknown_windows = unknown["window_count"].sum()

for seed in range(200):

    outer = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=seed
    )

    trainval_idx, test_idx = next(
        outer.split(
            known,
            groups=known["artist_id"]
        )
    )

    inner_data = known.iloc[trainval_idx]

    inner = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15 / 0.85,
        random_state=1000 + seed
    )

    train_local, val_local = next(
        inner.split(
            inner_data,
            groups=inner_data["artist_id"]
        )
    )

    assignments = np.full(
        len(known),
        "train",
        dtype=object
    )

    assignments[test_idx] = "test"

    assignments[
        trainval_idx[val_local]
    ] = "val"

    # -----------------------------------
    # 3. Evaluate split balance
    # -----------------------------------

    window_totals = np.array([
        known.loc[
            assignments == split,
            "window_count"
        ].sum()
        +
        (unknown_windows if split == "train" else 0)
        for split in SPLITS
    ])

    window_ratios = (
        window_totals / window_totals.sum()
    )

    score = 3 * np.abs(
        window_ratios - TARGET_RATIOS
    ).sum()

    for genre, mask in genre_masks.items():

        genre_counts = np.array([
            np.sum(
                (assignments == split) & mask
            )
            for split in SPLITS
        ])

        if genre_counts.sum() == 0:
            continue

        genre_ratios = (
            genre_counts / genre_counts.sum()
        )

        score += np.abs(
            genre_ratios - TARGET_RATIOS
        ).sum()

        # Encourage at least two represented
        # artists per genre in validation and test.

        for split in ["val", "test"]:

            n_artists = known.loc[
                (assignments == split) & mask,
                "artist_id"
            ].nunique()

            if n_artists < 2:
                score += 2

    if score < best_score:

        best_score = score
        best_assignments = assignments.copy()
        best_seed = seed


# -----------------------------------
# 4. Construct provisional assignments
# -----------------------------------

known["split"] = best_assignments

# Unknown-artist songs are training-only
# candidates until their identities are checked.
unknown["split"] = "train"

song_split_df = pd.concat(
    [known, unknown],
    ignore_index=True
)

assert song_split_df["track_id"].is_unique

assert len(song_split_df) == len(
    five_genre_df
)


# -----------------------------------
# 5. Validate identified-artist separation
# -----------------------------------

artist_sets = {
    split: set(
        song_split_df.loc[
            (song_split_df["split"] == split)
            &
            (song_split_df["artist_id"].notna()),
            "artist_id"
        ]
    )
    for split in SPLITS
}

for i, split_a in enumerate(SPLITS):

    for split_b in SPLITS[i + 1:]:

        assert artist_sets[split_a].isdisjoint(
            artist_sets[split_b]
        ), (
            f"Artist leakage: "
            f"{split_a} vs {split_b}"
        )


print("\nARTIST-DISJOINT SPLIT")
print("--------------------------------")

print("Selected candidate seed:", best_seed)

print("Identified artists do not overlap.")


# -----------------------------------
# 6. Assign windows to partitions
# -----------------------------------

window_split_df = selected_windows_df.merge(
    song_split_df[
        ["track_id", "split"]
    ],
    on="track_id",
    how="left",
    validate="many_to_one"
)

assert window_split_df["split"].notna().all()

assert len(window_split_df) == 8952


# -----------------------------------
# 7. Overall split statistics
# -----------------------------------

summary_records = []

for split in SPLITS:

    songs = song_split_df[
        song_split_df["split"] == split
    ]

    windows = window_split_df[
        window_split_df["split"] == split
    ]

    summary_records.append({
        "Split": split,
        "Songs": len(songs),
        "Windows": len(windows),
        "Artists": songs["artist_id"].nunique(),
        "Window percentage":
            100 * len(windows)
            / len(window_split_df)
    })

split_summary_df = pd.DataFrame(
    summary_records
)

print("\nDATASET SPLIT SUMMARY")
print("--------------------------------")

display(
    split_summary_df.round(2)
)


# -----------------------------------
# 8. Genre distribution in each split
# -----------------------------------

genre_split_records = []

for genre in FIVE_GENRES:

    for split in SPLITS:

        matching = song_split_df[
            (song_split_df["split"] == split)
            &
            song_split_df["selected_genres"].apply(
                lambda labels: genre in labels
            )
        ]

        genre_split_records.append({
            "Genre": genre,
            "Split": split,
            "Songs": len(matching),
            "Artists":
                matching["artist_id"].nunique(),
            "Windows":
                matching["window_count"].sum()
        })

genre_split_df = pd.DataFrame(
    genre_split_records
)

print("\nGENRE DISTRIBUTION BY SPLIT")
print("--------------------------------")

display(genre_split_df)


# -----------------------------------
# 9. Check songs outside selected genres
# -----------------------------------

print("\nUNCONDITIONED / UNKNOWN-GENRE SONGS")
print("--------------------------------")

for split in SPLITS:

    matching = song_split_df[
        (song_split_df["split"] == split)
        &
        (song_split_df["selected_genres"].apply(
            len
        ) == 0)
    ]

    print(
        f"{split}: {len(matching)} songs, "
        f"{matching['window_count'].sum()} windows"
    )


print("\nProvisional split analysis completed!")

ARTIST METADATA
--------------------------------
Known-artist songs: 1069
Unknown-artist songs: 7
Unique identified artists: 218

ARTIST-DISJOINT SPLIT
--------------------------------
Selected candidate seed: 91
Identified artists do not overlap.

DATASET SPLIT SUMMARY
--------------------------------


,Split,Songs,Windows,Artists,Window percentage
0,train,759,6324,152,70.64
1,val,176,1405,33,15.69
2,test,141,1223,33,13.66



GENRE DISTRIBUTION BY SPLIT
--------------------------------


,Genre,Split,Songs,Artists,Windows
0,rock,train,311,68,2740
1,rock,val,83,14,717
2,rock,test,47,13,495
3,pop,train,170,53,1339
4,pop,val,37,11,315
5,pop,test,58,14,476
6,electronic,train,77,33,617
7,electronic,val,18,8,142
8,electronic,test,17,9,147
9,folk,train,116,24,1058



UNCONDITIONED / UNKNOWN-GENRE SONGS
--------------------------------
train: 192 songs, 1508 windows
val: 46 songs, 320 windows
test: 30 songs, 228 windows

Provisional split analysis completed!


### Normalized Training Example
Objective: Construct and inspect one 16-bar training example using the selected dataset.
The melody pitches and chord roots are transposed consistently according to the estimated major-scale reference, with C as the common reference. Chord qualities are preserved.
Melody events retain their relative timing and duration. Chord annotations are aligned with individual beats, and uncertain beat targets are masked.
This experiment verifies the input/output preparation logic before applying it to the complete dataset. Roman-numeral conversion will be implemented after the final chord vocabulary is confirmed.

In [33]:
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Select one example window
# -----------------------------------

PREFERRED_TRACK = "track_0854720"

examples = window_split_df[
    window_split_df["track_id"] == PREFERRED_TRACK
]

if examples.empty:

    examples = window_split_df.merge(
        phase3_key_df[
            ["track_id", "estimated_key"]
        ],
        on="track_id"
    )

    examples = examples[
        examples["estimated_key"] != "C major"
    ]

example = examples.sort_values(
    "start_sec"
).iloc[0]

track_id = example["track_id"]

window_start = float(example["start_sec"])
window_end = float(example["end_sec"])

folder = track_lookup[track_id]


# -----------------------------------
# 2. Load melody, chords, beats
# -----------------------------------

melody_data = read_json(
    folder / "melody/combined.json"
)

chord_data = read_json(
    folder / "chords/submission.json"
)

beat_data = read_json(
    folder / "beats/beats.json"
)


# -----------------------------------
# 3. Calculate reference transposition
# -----------------------------------

key_name = key_lookup[track_id]

key_pc = get_major_pc(key_name)

assert key_pc is not None

# Choose the smallest semitone transposition
# that maps the reference tonic to C.

semitone_shift = ((-key_pc + 6) % 12) - 6

print("NORMALIZED TRAINING EXAMPLE")
print("--------------------------------")

print("Track:", track_id)
print("Split:", example["split"])
print("Original reference:", key_name)

print(
    "Transposition:",
    f"{semitone_shift:+d} semitones"
)

print(
    "Window:",
    f"{window_start:.2f}s – {window_end:.2f}s"
)


# -----------------------------------
# 4. Normalize melody notes
# -----------------------------------

normalized_notes = []

for note in melody_data.get("notes", []):

    try:

        start = float(note["onset_sec"])
        end = float(note["offset_sec"])

        pitch = float(
            note["midi_pitch_rounded"]
        )

    except (KeyError, TypeError, ValueError):
        continue

    if not np.isfinite(
        [start, end, pitch]
    ).all():
        continue

    if end <= start:
        continue

    if start >= window_end or end <= window_start:
        continue

    # Clip notes at window boundaries.

    clipped_start = max(start, window_start)
    clipped_end = min(end, window_end)

    original_pitch = int(round(pitch))

    normalized_pitch = (
        original_pitch + semitone_shift
    )

    normalized_notes.append({

        "onset_sec":
            clipped_start - window_start,

        "duration_sec":
            clipped_end - clipped_start,

        "original_midi":
            original_pitch,

        "normalized_midi":
            normalized_pitch,

        "normalized_pc":
            normalized_pitch % 12
    })


normalized_notes.sort(
    key=lambda n: n["onset_sec"]
)


# -----------------------------------
# 5. Prepare beat-level chord targets
# -----------------------------------

chords = prepare_window_chords(
    chord_data.get("events", [])
)

beat_times = sorted(
    float(beat["time_sec"])
    for beat in beat_data.get("beats", [])
    if window_start
    <= float(beat["time_sec"])
    < window_end
)

beat_targets = []

for i, beat_start in enumerate(beat_times):

    beat_end = (
        beat_times[i + 1]
        if i + 1 < len(beat_times)
        else window_end
    )

    beat_duration = beat_end - beat_start

    if beat_duration <= 0:
        continue

    overlaps = []

    for chord in chords:

        overlap = max(
            0.0,
            min(beat_end, chord["end"])
            - max(beat_start, chord["start"])
        )

        if overlap > 0:
            overlaps.append((overlap, chord))

    best_chord = None
    best_fraction = 0.0

    if overlaps:

        best_overlap, candidate = max(
            overlaps,
            key=lambda item: item[0]
        )

        best_fraction = (
            best_overlap / beat_duration
        )

        # Reject contradictory chord overlaps.

        conflicting = any(
            overlap / beat_duration >= 0.20
            and (
                chord["root"],
                chord["quality"]
            ) != (
                candidate["root"],
                candidate["quality"]
            )
            for overlap, chord in overlaps
            if chord is not candidate
        )

        if (
            best_fraction >= 0.80
            and not conflicting
            and candidate["quality"] is not None
        ):
            best_chord = candidate

    if best_chord is not None:

        relative_root = (
            best_chord["root"] - key_pc
        ) % 12

        chord_quality = best_chord["quality"]

        valid_mask = 1

    else:

        relative_root = None
        chord_quality = None
        valid_mask = 0

    beat_targets.append({

        "beat_index": i,

        "relative_time_sec":
            beat_start - window_start,

        "relative_root_pc":
            relative_root,

        "normalized_root":
            (
                PC_NAMES[relative_root]
                if relative_root is not None
                else None
            ),

        "chord_quality":
            chord_quality,

        "valid_mask":
            valid_mask
    })


# -----------------------------------
# 6. Inspect normalized melody
# -----------------------------------

print("\nNORMALIZED MELODY")
print("--------------------------------")

print(
    "Melody notes:",
    len(normalized_notes)
)

display(
    pd.DataFrame(
        normalized_notes
    ).head(15).round(3)
)


# -----------------------------------
# 7. Inspect beat-level targets
# -----------------------------------

print("\nBEAT-LEVEL CHORD TARGETS")
print("--------------------------------")

print(
    "Total beats:",
    len(beat_targets)
)

valid_targets = sum(
    target["valid_mask"]
    for target in beat_targets
)

print(
    "Valid targets:",
    valid_targets
)

print(
    "Masked targets:",
    len(beat_targets) - valid_targets
)

print(
    "Valid target percentage:",
    f"{100 * valid_targets / len(beat_targets):.2f}%"
    if beat_targets else "N/A"
)

display(
    pd.DataFrame(
        beat_targets
    ).head(20).round(3)
)


print("\nTraining example preview completed!")

NORMALIZED TRAINING EXAMPLE
--------------------------------
Track: track_0854720
Split: val
Original reference: A major
Transposition: +3 semitones
Window: 0.16s – 42.74s

NORMALIZED MELODY
--------------------------------
Melody notes: 67


,onset_sec,duration_sec,original_midi,normalized_midi,normalized_pc
0,0.000,0.502,76,79,7
1,0.502,0.497,76,79,7
2,1.000,0.680,76,79,7
3,1.680,0.350,76,79,7
4,2.030,0.330,76,79,7
5,2.360,0.330,76,79,7
6,2.690,0.510,76,79,7
7,3.200,0.510,76,79,7
8,3.710,0.675,76,79,7
9,4.385,0.670,76,79,7



BEAT-LEVEL CHORD TARGETS
--------------------------------
Total beats: 64
Valid targets: 60
Masked targets: 4
Valid target percentage: 93.75%


,beat_index,relative_time_sec,relative_root_pc,normalized_root,chord_quality,valid_mask
0,0,0.00,0.0,C,maj,1
1,1,0.67,0.0,C,maj,1
2,2,1.33,0.0,C,maj,1
3,3,2.03,0.0,C,maj,1
4,4,2.69,0.0,C,maj,1
5,5,3.37,0.0,C,maj,1
6,6,4.05,0.0,C,maj,1
7,7,4.72,0.0,C,maj,1
8,8,5.39,9.0,A,min,1
9,9,6.06,9.0,A,min,1



Training example preview completed!


### Dataset Construction
Objective: Convert all selected 16-bar windows into normalized melody–chord training examples.
Melody pitches and chord roots are transposed relative to each song's estimated major-scale reference. Original chord qualities are preserved. Chord annotations are mapped to beat-level targets, with unreliable targets masked.
Each example retains its track identifier, artist identifier, dataset split, reference key, and five-dimensional genre representation.
The resulting dataset is saved as JSONL for subsequent validation and Roman-numeral label encoding.

In [34]:
import json
from bisect import bisect_left
from pathlib import Path

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Output location
# -----------------------------------

DRAFT_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "mtg-jamendo"
    / "phase3-prepared"
)

DRAFT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRAFT_FILE = (
    DRAFT_DIR / "windows_v0_pre_roman.jsonl"
)


# -----------------------------------
# 2. Build a single normalized window
# -----------------------------------

def build_normalized_example(
    row,
    notes,
    chords,
    beats,
    major_pc,
    metadata
):

    start = float(row.start_sec)
    end = float(row.end_sec)

    shift = ((-major_pc + 6) % 12) - 6

    # Normalize melody.
    melody = []

    for note in notes:

        if note["start"] >= end:
            break

        if note["end"] <= start:
            continue

        onset = max(note["start"], start)
        offset = min(note["end"], end)

        pitch = note["pitch"] + shift

        melody.append({
            "onset_sec": round(onset - start, 6),
            "duration_sec": round(offset - onset, 6),
            "midi_pitch": pitch,
            "pitch_class": pitch % 12
        })

    # Prepare beats inside the window.
    times = [b["time"] for b in beats]

    left = bisect_left(times, start)
    right = bisect_left(times, end)

    targets = []
    chord_pointer = 0

    for local_index, beat_index in enumerate(
        range(left, right)
    ):

        beat_start = times[beat_index]

        beat_end = (
            times[beat_index + 1]
            if beat_index + 1 < right
            else end
        )

        beat_duration = beat_end - beat_start

        if beat_duration <= 0:
            continue

        # Skip chords already finished.
        while (
            chord_pointer < len(chords)
            and chords[chord_pointer]["end"]
            <= beat_start
        ):
            chord_pointer += 1

        overlaps = []
        j = chord_pointer

        while (
            j < len(chords)
            and chords[j]["start"] < beat_end
        ):

            chord = chords[j]

            overlap = max(
                0.0,
                min(beat_end, chord["end"])
                - max(beat_start, chord["start"])
            )

            if overlap > 0:
                overlaps.append((overlap, chord))

            j += 1

        best_chord = None

        if overlaps:

            best_overlap, candidate = max(
                overlaps,
                key=lambda x: x[0]
            )

            best_fraction = (
                best_overlap / beat_duration
            )

            conflicting = any(
                overlap / beat_duration >= 0.20
                and (
                    chord["root"],
                    chord["quality"]
                ) != (
                    candidate["root"],
                    candidate["quality"]
                )
                for overlap, chord in overlaps
                if chord is not candidate
            )

            if (
                best_fraction >= 0.80
                and not conflicting
                and candidate["quality"] is not None
            ):
                best_chord = candidate

        valid = int(best_chord is not None)

        relative_root = (
            (best_chord["root"] - major_pc) % 12
            if valid else None
        )

        targets.append({
            "beat_index": local_index,
            "time_sec": round(
                beat_start - start, 6
            ),
            "duration_sec": round(
                beat_duration, 6
            ),
            "root_pc": relative_root,
            "quality": (
                best_chord["quality"]
                if valid else None
            ),
            "valid_mask": valid
        })

    genres = metadata["selected_genres"]

    artist_id = metadata["artist_id"]

    return {
        "track_id": row.track_id,
        "window_index": int(row.window_index),
        "split": row.split,

        "artist_id": (
            str(artist_id)
            if pd.notna(artist_id)
            else None
        ),

        "reference_key": key_lookup[row.track_id],
        "semitone_shift": shift,

        "duration_sec": round(end - start, 6),
        "window_start_sec": round(start, 6),

        "genre_vector": [
            int(g in genres)
            for g in FIVE_GENRES
        ],
        "has_selected_genre": bool(genres),

        "melody": melody,
        "beat_targets": targets
    }


# -----------------------------------
# 3. Process all songs and windows
# -----------------------------------

metadata_lookup = five_genre_df.set_index(
    "track_id"
).to_dict("index")

construction_stats = []

with open(
    DRAFT_FILE,
    "w",
    encoding="utf-8"
) as output_file:

    for track_id, group in window_split_df.groupby(
        "track_id"
    ):

        folder = track_lookup[track_id]

        melody_data = read_json(
            folder / "melody/combined.json"
        )

        chord_data = read_json(
            folder / "chords/submission.json"
        )

        beat_data = read_json(
            folder / "beats/beats.json"
        )

        major_pc = get_major_pc(
            key_lookup[track_id]
        )

        assert major_pc is not None

        # Prepare melody once per song.
        notes = []

        for note in melody_data.get("notes", []):

            try:
                a = float(note["onset_sec"])
                b = float(note["offset_sec"])
                p = float(note["midi_pitch_rounded"])

                if (
                    np.isfinite([a, b, p]).all()
                    and b > a
                ):
                    notes.append({
                        "start": a,
                        "end": b,
                        "pitch": int(round(p))
                    })

            except (KeyError, TypeError, ValueError):
                continue

        notes.sort(key=lambda n: n["start"])

        chords = prepare_window_chords(
            chord_data.get("events", [])
        )

        beats = sorted(
            [
                {"time": float(b["time_sec"])}
                for b in beat_data.get("beats", [])
            ],
            key=lambda b: b["time"]
        )

        metadata = metadata_lookup[track_id]

        for row in group.itertuples():

            example = build_normalized_example(
                row,
                notes,
                chords,
                beats,
                major_pc,
                metadata
            )

            output_file.write(
                json.dumps(
                    example,
                    allow_nan=False
                ) + "\n"
            )

            targets = example["beat_targets"]

            valid_count = sum(
                t["valid_mask"]
                for t in targets
            )

            construction_stats.append({
                "track_id": track_id,
                "split": row.split,
                "melody_notes": len(
                    example["melody"]
                ),
                "beats": len(targets),
                "valid_targets": valid_count,
                "actual_valid_fraction": (
                    valid_count / len(targets)
                    if targets else 0
                )
            })


# -----------------------------------
# 4. Validate construction
# -----------------------------------

construction_df = pd.DataFrame(
    construction_stats
)

assert len(construction_df) == len(
    window_split_df
)

print("PHASE 3 — DATASET CONSTRUCTION")
print("--------------------------------")

print("Saved:", DRAFT_FILE)

print(
    "Examples:",
    len(construction_df)
)

print(
    "Songs:",
    construction_df["track_id"].nunique()
)

print(
    "Total melody notes:",
    construction_df["melody_notes"].sum()
)

print(
    "Total beat targets:",
    construction_df["beats"].sum()
)

print(
    "Valid beat targets:",
    construction_df["valid_targets"].sum()
)


# -----------------------------------
# 5. Inspect distributions
# -----------------------------------

print("\nEXAMPLE STATISTICS")
print("--------------------------------")

display(
    construction_df[
        [
            "melody_notes",
            "beats",
            "valid_targets",
            "actual_valid_fraction"
        ]
    ].describe().round(3)
)


# -----------------------------------
# 6. Check target-mask consistency
# -----------------------------------

print("\nTARGET COVERAGE VALIDATION")
print("--------------------------------")

below_75 = construction_df[
    construction_df["actual_valid_fraction"] < 0.75
]

print(
    "Windows below 75% after final alignment:",
    len(below_75)
)

display(
    below_75.sort_values(
        "actual_valid_fraction"
    ).head(10)
)


# -----------------------------------
# 7. Split distribution
# -----------------------------------

print("\nEXAMPLES PER SPLIT")
print("--------------------------------")

display(
    construction_df["split"].value_counts()
)


print("\nDraft dataset construction completed!")

PHASE 3 — DATASET CONSTRUCTION
--------------------------------
Saved: /Users/praise/Developer/vocal-harmonization/datasets/mtg-jamendo/phase3-prepared/windows_v0_pre_roman.jsonl
Examples: 8952
Songs: 1076
Total melody notes: 637182
Total beat targets: 535806
Valid beat targets: 478153

EXAMPLE STATISTICS
--------------------------------


,melody_notes,beats,valid_targets,actual_valid_fraction
count,8952.000,8952.000,8952.000,8952.000
mean,71.178,59.853,53.413,0.889
std,29.267,15.271,14.813,0.061
min,20.000,32.000,23.000,0.719
25%,50.000,64.000,49.000,0.844
50%,66.000,64.000,56.000,0.891
75%,87.000,64.000,60.000,0.938
max,258.000,96.000,96.000,1.000



TARGET COVERAGE VALIDATION
--------------------------------
Windows below 75% after final alignment: 3


,track_id,split,melody_notes,beats,valid_targets,actual_valid_fraction
5498,track_1087536,train,66,32,23,0.718750
2770,track_0961261,test,38,48,35,0.729167
3616,track_1037054,train,87,64,47,0.734375



EXAMPLES PER SPLIT
--------------------------------


split
train    6324
val      1405
test     1223
Name: count, dtype: int64


Draft dataset construction completed!


### Chord Vocabulary Audit
Objective: Examine the distribution of chord qualities in the constructed Phase 3 dataset before finalizing the coarse chord-family representation.
The experiment counts valid beat-level chord labels across training, validation, and test partitions. It also measures the number of training songs supporting each quality and identifies rare or unseen labels.
The proposed representation consists of twelve relative chord roots and nine chord families, with an additional no-chord category. Mapping rules will be finalized after examining the observed chord vocabulary.
This step does not modify the existing dataset.

In [35]:
import json
from collections import Counter, defaultdict

import pandas as pd


# -----------------------------------
# 1. Read constructed dataset
# -----------------------------------

quality_counts = {
    split: Counter()
    for split in ["train", "val", "test"]
}

quality_songs = {
    split: defaultdict(set)
    for split in ["train", "val", "test"]
}

total_beats = Counter()
masked_beats = Counter()

example_labels = {}

window_count = 0


with open(
    DRAFT_FILE,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        example = json.loads(line)

        window_count += 1

        split = example["split"]
        track_id = example["track_id"]

        for target in example["beat_targets"]:

            total_beats[split] += 1

            if target["valid_mask"] == 0:

                masked_beats[split] += 1
                continue

            quality = target["quality"]
            root = target["root_pc"]

            assert quality is not None
            assert root is not None
            assert 0 <= root <= 11

            quality_counts[split][quality] += 1

            quality_songs[split][quality].add(
                track_id
            )

            if quality not in example_labels:

                example_labels[quality] = {
                    "root_pc": root,
                    "track_id": track_id
                }


# -----------------------------------
# 2. Summary
# -----------------------------------

print("CHORD VOCABULARY AUDIT")
print("--------------------------------")

print("Windows:", window_count)

print(
    "Unique chord qualities:",
    len(set().union(
        *[
            set(counter)
            for counter in quality_counts.values()
        ]
    ))
)


# -----------------------------------
# 3. Quality frequency by split
# -----------------------------------

all_qualities = sorted(
    set().union(
        *[
            set(counter)
            for counter in quality_counts.values()
        ]
    )
)

records = []

train_total = sum(
    quality_counts["train"].values()
)

for quality in all_qualities:

    train_count = quality_counts["train"][quality]

    records.append({

        "Quality": quality,

        "Train beats": train_count,

        "Train (%)": (
            100 * train_count / train_total
            if train_total else 0
        ),

        "Train songs": len(
            quality_songs["train"][quality]
        ),

        "Val beats":
            quality_counts["val"][quality],

        "Test beats":
            quality_counts["test"][quality]
    })


chord_vocabulary_df = pd.DataFrame(
    records
).sort_values(
    "Train beats",
    ascending=False
).reset_index(drop=True)


print("\nCHORD QUALITY DISTRIBUTION")
print("--------------------------------")

display(
    chord_vocabulary_df.round(3)
)


# -----------------------------------
# 4. Rare and unseen labels
# -----------------------------------

print("\nRARE TRAINING QUALITIES")
print("--------------------------------")

rare_qualities = chord_vocabulary_df[
    chord_vocabulary_df["Train beats"] < 20
]

display(rare_qualities)


print("\nQUALITIES UNSEEN IN TRAINING")
print("--------------------------------")

unseen = chord_vocabulary_df[
    (chord_vocabulary_df["Train beats"] == 0)
    &
    (
        (chord_vocabulary_df["Val beats"] > 0)
        |
        (chord_vocabulary_df["Test beats"] > 0)
    )
]

display(unseen)


# -----------------------------------
# 5. Masked beat statistics
# -----------------------------------

print("\nBEAT TARGET SUMMARY")
print("--------------------------------")

for split in ["train", "val", "test"]:

    total = total_beats[split]
    masked = masked_beats[split]

    print(
        f"{split}: "
        f"{total:,} beats, "
        f"{total - masked:,} valid, "
        f"{masked:,} masked"
    )


# -----------------------------------
# 6. Example chord labels
# -----------------------------------

print("\nEXAMPLE LABELS")
print("--------------------------------")

for quality, information in sorted(
    example_labels.items()
):

    print(
        f"{quality:12s} "
        f"root={information['root_pc']:2d} "
        f"track={information['track_id']}"
    )


print("\nChord vocabulary audit completed!")

CHORD VOCABULARY AUDIT
--------------------------------
Windows: 8952
Unique chord qualities: 14

CHORD QUALITY DISTRIBUTION
--------------------------------


,Quality,Train beats,Train (%),Train songs,Val beats,Test beats
0,maj,177105,52.949,742,40370,37626
1,min,119660,35.774,711,29633,22003
2,min7,16938,5.064,266,4252,3073
3,7,11445,3.422,216,2175,1971
4,maj7,7236,2.163,154,1060,923
5,sus4,762,0.228,58,63,67
6,dim,472,0.141,44,65,139
7,sus4(b7),268,0.080,27,17,41
8,min9,224,0.067,17,4,12
9,hdim7,213,0.064,28,53,69



RARE TRAINING QUALITIES
--------------------------------


,Quality,Train beats,Train (%),Train songs,Val beats,Test beats
12,dim7,16,0.004783,1,0,0
13,sus2,10,0.002990,1,0,0



QUALITIES UNSEEN IN TRAINING
--------------------------------


,Quality,Train beats,Train (%),Train songs,Val beats,Test beats



BEAT TARGET SUMMARY
--------------------------------
train: 375,210 beats, 334,485 valid, 40,725 masked
val: 86,624 beats, 77,718 valid, 8,906 masked
test: 73,972 beats, 65,950 valid, 8,022 masked

EXAMPLE LABELS
--------------------------------
7            root= 0 track=track_0000382
9            root= 0 track=track_0086256
aug          root= 4 track=track_0048222
dim          root=11 track=track_0000382
dim7         root= 2 track=track_1159695
hdim7        root= 3 track=track_0000387
maj          root= 5 track=track_0000382
maj7         root= 5 track=track_0000387
min          root=10 track=track_0000382
min7         root= 2 track=track_0000387
min9         root= 2 track=track_0086256
sus2         root= 5 track=track_1134297
sus4         root= 2 track=track_0086280
sus4(b7)     root= 8 track=track_0243721

Chord vocabulary audit completed!


### Chord-Family Mapping
Objective: Implement and validate a deterministic transformation from extracted chord qualities to the nine proposed chord families.
The experiment verifies that every observed chord quality has a mapping, tests representative relative-root and family combinations, and measures class frequencies across training, validation, and test sets.
Rare families are retained for analysis rather than automatically merged or discarded. No-chord events and uncertain annotations will be handled separately.

In [36]:
import pandas as pd


# -----------------------------------
# 1. Define chord-family mapping
# -----------------------------------

CHORD_FAMILY_MAP = {

    "maj": "MAJ",
    "maj7": "MAJ",

    "min": "MIN",
    "min7": "MIN",
    "min9": "MIN",

    "7": "DOM7",
    "9": "DOM7",

    "dim": "DIM",
    "hdim7": "HDIM7",
    "dim7": "DIM7",

    "aug": "AUG",

    "sus2": "SUS2",

    "sus4": "SUS4",
    "sus4(b7)": "SUS4"
}


CHORD_FAMILIES = [
    "MAJ",
    "MIN",
    "DOM7",
    "DIM",
    "HDIM7",
    "DIM7",
    "AUG",
    "SUS2",
    "SUS4"
]


# -----------------------------------
# 2. Define relative-root labels
# -----------------------------------

RELATIVE_ROOT_LABELS = [
    "I",
    "♭II",
    "II",
    "♭III",
    "III",
    "IV",
    "♯IV",
    "V",
    "♭VI",
    "VI",
    "♭VII",
    "VII"
]


def chord_to_plan_label(root_pc, quality):

    # Missing or uncertain annotations
    # are not automatically no-chord events.

    if root_pc is None or quality is None:
        return None

    family = CHORD_FAMILY_MAP.get(quality)

    if family is None:
        raise ValueError(
            f"Unsupported chord quality: {quality}"
        )

    root_pc = int(root_pc)

    if not 0 <= root_pc < 12:
        raise ValueError(
            f"Invalid relative root: {root_pc}"
        )

    root_label = RELATIVE_ROOT_LABELS[root_pc]

    return f"{root_label}:{family}"


# -----------------------------------
# 3. Verify complete quality coverage
# -----------------------------------

observed_qualities = set(
    chord_vocabulary_df["Quality"]
)

mapped_qualities = set(
    CHORD_FAMILY_MAP
)

unmapped = observed_qualities - mapped_qualities

assert not unmapped, (
    f"Unmapped qualities: {unmapped}"
)

print("CHORD-FAMILY MAPPING")
print("--------------------------------")

print(
    "Observed chord qualities:",
    len(observed_qualities)
)

print(
    "Chord families:",
    len(CHORD_FAMILIES)
)

print(
    "Unmapped qualities:",
    len(unmapped)
)


# -----------------------------------
# 4. Unit tests
# -----------------------------------

assert chord_to_plan_label(0, "maj") == "I:MAJ"

assert chord_to_plan_label(0, "maj7") == "I:MAJ"

assert chord_to_plan_label(7, "7") == "V:DOM7"

assert chord_to_plan_label(7, "9") == "V:DOM7"

assert chord_to_plan_label(9, "min") == "VI:MIN"

assert chord_to_plan_label(9, "min7") == "VI:MIN"

assert chord_to_plan_label(11, "dim7") == "VII:DIM7"

assert chord_to_plan_label(11, "hdim7") == "VII:HDIM7"

assert chord_to_plan_label(0, "7") == "I:DOM7"

assert chord_to_plan_label(
    0, "sus4(b7)"
) == "I:SUS4"

assert chord_to_plan_label(None, None) is None


print("\nMapping unit tests passed!")


# -----------------------------------
# 5. Analyze family frequencies
# -----------------------------------

family_records = []

train_total = sum(
    quality_counts["train"].values()
)

for family in CHORD_FAMILIES:

    qualities = [
        quality
        for quality, mapped_family
        in CHORD_FAMILY_MAP.items()
        if mapped_family == family
    ]

    train_beats = sum(
        quality_counts["train"][quality]
        for quality in qualities
    )

    val_beats = sum(
        quality_counts["val"][quality]
        for quality in qualities
    )

    test_beats = sum(
        quality_counts["test"][quality]
        for quality in qualities
    )

    # Count distinct songs, not summed
    # per-quality song counts.

    train_song_ids = set()

    for quality in qualities:

        train_song_ids.update(
            quality_songs["train"][quality]
        )

    family_records.append({

        "Family": family,

        "Mapped qualities":
            ", ".join(qualities),

        "Train beats": train_beats,

        "Train (%)":
            100 * train_beats / train_total,

        "Train songs":
            len(train_song_ids),

        "Val beats": val_beats,

        "Test beats": test_beats
    })


family_distribution_df = pd.DataFrame(
    family_records
).sort_values(
    "Train beats",
    ascending=False
).reset_index(drop=True)


print("\nFAMILY DISTRIBUTION")
print("--------------------------------")

display(
    family_distribution_df.round(3)
)


# -----------------------------------
# 6. Verify that mapping does not
#    lose valid beat targets
# -----------------------------------

for split in ["train", "val", "test"]:

    original_count = sum(
        quality_counts[split].values()
    )

    family_count = family_distribution_df[
        {
            "train": "Train beats",
            "val": "Val beats",
            "test": "Test beats"
        }[split]
    ].sum()

    assert original_count == family_count


print("\nAll valid targets accounted for.")


# -----------------------------------
# 7. Identify weakly supported families
# -----------------------------------

print("\nRARE CHORD FAMILIES")
print("--------------------------------")

display(
    family_distribution_df[
        family_distribution_df["Train beats"] < 100
    ]
)


# -----------------------------------
# 8. Preview relative-root labels
# -----------------------------------

print("\nEXAMPLE PLAN LABELS")
print("--------------------------------")

examples = [
    (0, "maj7"),
    (7, "7"),
    (9, "min7"),
    (5, "maj"),
    (2, "min9"),
    (11, "hdim7"),
    (0, "sus4(b7)")
]

for root, quality in examples:

    print(
        f"Root={root:2d}, "
        f"Quality={quality:10s} "
        f"→ {chord_to_plan_label(root, quality)}"
    )


print("\nChord-family mapping analysis completed!")

CHORD-FAMILY MAPPING
--------------------------------
Observed chord qualities: 14
Chord families: 9
Unmapped qualities: 0

Mapping unit tests passed!

FAMILY DISTRIBUTION
--------------------------------


,Family,Mapped qualities,Train beats,Train (%),Train songs,Val beats,Test beats
0,MAJ,"maj, maj7",184341,55.112,750,41430,38549
1,MIN,"min, min7, min9",136822,40.905,736,33889,25088
2,DOM7,"7, 9",11505,3.440,217,2192,1979
3,SUS4,"sus4, sus4(b7)",1030,0.308,74,80,108
4,DIM,dim,472,0.141,44,65,139
5,HDIM7,hdim7,213,0.064,28,53,69
6,AUG,aug,76,0.023,12,9,18
7,DIM7,dim7,16,0.005,1,0,0
8,SUS2,sus2,10,0.003,1,0,0



All valid targets accounted for.

RARE CHORD FAMILIES
--------------------------------


,Family,Mapped qualities,Train beats,Train (%),Train songs,Val beats,Test beats
6,AUG,aug,76,0.022721,12,9,18
7,DIM7,dim7,16,0.004783,1,0,0
8,SUS2,sus2,10,0.002990,1,0,0



EXAMPLE PLAN LABELS
--------------------------------
Root= 0, Quality=maj7       → I:MAJ
Root= 7, Quality=7          → V:DOM7
Root= 9, Quality=min7       → VI:MIN
Root= 5, Quality=maj        → IV:MAJ
Root= 2, Quality=min9       → II:MIN
Root=11, Quality=hdim7      → VII:HDIM7
Root= 0, Quality=sus4(b7)   → I:SUS4

Chord-family mapping analysis completed!


### No-Chord and Uncertain-Target Analysis
Objective: Investigate the causes of masked beat-level chord targets in the constructed dataset.
The experiment compares masked beats with the original chord annotations to distinguish explicitly predicted no-chord (N) regions from incomplete, conflicting, or unsupported annotations.
A beat is considered dominated by an N annotation when that annotation covers at least 80% of its duration without substantial overlap from recognized chords.
Since the source chord annotations are automatically extracted, explicit N predictions are treated as candidate no-chord labels rather than verified musical silence.
No changes are made to the dataset during this experiment.

In [37]:
import json
from collections import Counter

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Classify original chord events
# -----------------------------------

NO_CHORD_LABELS = {
    "N",
    "NC",
    "N.C.",
    "NO_CHORD",
    "NO CHORD"
}


def classify_chord_event(event):

    label = str(
        event.get("chord_label", "")
    ).strip().upper()

    if label in NO_CHORD_LABELS:
        return "N"

    root = root_to_pc(
        event.get("root")
    )

    quality = event.get("quality")

    if (
        root is not None
        and quality in CHORD_FAMILY_MAP
    ):
        return "chord"

    return "other"


# -----------------------------------
# 2. Load original chord annotations
# -----------------------------------

chord_event_cache = {}

raw_event_counts = Counter()

for track_id in selected_windows_df["track_id"].unique():

    folder = track_lookup[track_id]

    data = read_json(
        folder / "chords/submission.json"
    )

    events = []

    for event in data.get("events", []):

        try:
            start = float(event["start_sec"])
            end = float(event["end_sec"])
        except (KeyError, TypeError, ValueError):
            continue

        if not np.isfinite([start, end]).all():
            continue

        if end <= start:
            continue

        kind = classify_chord_event(event)

        raw_event_counts[kind] += 1

        events.append({
            "start": start,
            "end": end,
            "kind": kind
        })

    chord_event_cache[track_id] = sorted(
        events,
        key=lambda e: e["start"]
    )


print("ORIGINAL CHORD ANNOTATIONS")
print("--------------------------------")

print("Recognized chord events:",
      raw_event_counts["chord"])

print("Explicit N events:",
      raw_event_counts["N"])

print("Other/unsupported events:",
      raw_event_counts["other"])


# -----------------------------------
# 3. Analyze an individual masked beat
# -----------------------------------

def analyze_masked_beat(
    beat_start,
    beat_end,
    events
):

    duration = beat_end - beat_start

    if duration <= 0:
        return None

    overlaps = {
        "N": 0.0,
        "chord": 0.0,
        "other": 0.0
    }

    for event in events:

        if event["start"] >= beat_end:
            break

        if event["end"] <= beat_start:
            continue

        overlap = max(
            0.0,
            min(beat_end, event["end"])
            - max(beat_start, event["start"])
        )

        overlaps[event["kind"]] += overlap

    fractions = {
        kind: min(1.0, value / duration)
        for kind, value in overlaps.items()
    }

    n_fraction = fractions["N"]
    chord_fraction = fractions["chord"]
    other_fraction = fractions["other"]

    if (
        n_fraction >= 0.80
        and chord_fraction < 0.20
    ):
        category = "N-dominated"

    elif n_fraction > 0.001:
        category = "Mixed/partial N"

    elif chord_fraction >= 0.80:
        category = "Chord-annotated but masked"

    elif other_fraction >= 0.50:
        category = "Unsupported annotation"

    elif (
        n_fraction == 0
        and chord_fraction == 0
        and other_fraction == 0
    ):
        category = "No annotation"

    else:
        category = "Partial/unclear chord"

    return {
        "category": category,
        "n_fraction": n_fraction,
        "chord_fraction": chord_fraction,
        "other_fraction": other_fraction
    }


# -----------------------------------
# 4. Analyze all masked targets
# -----------------------------------

masked_records = []

with open(
    DRAFT_FILE,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        example = json.loads(line)

        track_id = example["track_id"]
        events = chord_event_cache[track_id]

        window_start = example["window_start_sec"]

        for target in example["beat_targets"]:

            if target["valid_mask"] == 1:
                continue

            beat_start = (
                window_start
                + target["time_sec"]
            )

            beat_end = (
                beat_start
                + target["duration_sec"]
            )

            result = analyze_masked_beat(
                beat_start,
                beat_end,
                events
            )

            if result is None:
                continue

            masked_records.append({
                "track_id": track_id,
                "split": example["split"],
                "window_index":
                    example["window_index"],
                "beat_index":
                    target["beat_index"],
                **result
            })


masked_analysis_df = pd.DataFrame(
    masked_records
)


# -----------------------------------
# 5. Overall masked-beat distribution
# -----------------------------------

print("\nMASKED BEAT ANALYSIS")
print("--------------------------------")

print(
    "Masked beats analyzed:",
    len(masked_analysis_df)
)

assert len(masked_analysis_df) == 57653

category_counts = (
    masked_analysis_df["category"]
    .value_counts()
)

category_summary = pd.DataFrame({
    "Category": category_counts.index,
    "Beats": category_counts.values,
    "Percentage": (
        100
        * category_counts.values
        / len(masked_analysis_df)
    )
})

display(
    category_summary.round(2)
)


# -----------------------------------
# 6. Distribution by dataset split
# -----------------------------------

print("\nMASKED BEATS BY SPLIT")
print("--------------------------------")

display(
    pd.crosstab(
        masked_analysis_df["category"],
        masked_analysis_df["split"]
    )
)


# -----------------------------------
# 7. Inspect explicit N cases
# -----------------------------------

n_dominated = masked_analysis_df[
    masked_analysis_df["category"]
    == "N-dominated"
]

print("\nEXPLICIT N-DOMINATED BEATS")
print("--------------------------------")

print(
    "N-dominated masked beats:",
    len(n_dominated)
)

print(
    "Songs containing N-dominated beats:",
    n_dominated["track_id"].nunique()
)

display(
    n_dominated[
        [
            "track_id",
            "split",
            "window_index",
            "beat_index",
            "n_fraction",
            "chord_fraction"
        ]
    ].head(15).round(3)
)


print("\nNo-chord annotation analysis completed!")

ORIGINAL CHORD ANNOTATIONS
--------------------------------
Recognized chord events: 91134
Explicit N events: 2768
Other/unsupported events: 0

MASKED BEAT ANALYSIS
--------------------------------
Masked beats analyzed: 57653


,Category,Beats,Percentage
0,Chord-annotated but masked,53932,93.55
1,N-dominated,3172,5.50
2,Mixed/partial N,549,0.95



MASKED BEATS BY SPLIT
--------------------------------


split,test,train,val
category,,,
Chord-annotated but masked,7464,38173,8295
Mixed/partial N,92,388,69
N-dominated,466,2164,542



EXPLICIT N-DOMINATED BEATS
--------------------------------
N-dominated masked beats: 3172
Songs containing N-dominated beats: 482


,track_id,split,window_index,beat_index,n_fraction,chord_fraction
0,track_0000382,train,0,0,1.0,0.0
4,track_0000382,train,0,27,1.0,0.0
5,track_0000382,train,0,28,1.0,0.0
6,track_0000382,train,0,29,1.0,0.0
7,track_0000382,train,0,30,1.0,0.0
8,track_0000382,train,0,31,1.0,0.0
9,track_0000382,train,0,32,1.0,0.0
13,track_0000382,train,1,0,1.0,0.0
103,track_0000386,train,0,0,1.0,0.0
104,track_0000386,train,0,1,1.0,0.0



No-chord annotation analysis completed!


### Findings from Step 3.32
Analysis of 57,653 masked beat targets found that 93.55% overlapped recognized chord annotations, while 5.50% were dominated by explicit no-chord predictions.
The results suggest that many masked beats may result from chord-boundary ambiguity or fragmented annotations rather than missing harmonic information.
Explicit no-chord predictions will remain masked during initial model training because their musical correctness has not been independently verified.
The next experiment investigates why beats containing recognized chord annotations failed the beat-level reliability requirements.

In [38]:
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Look up previously classified
#    masked beats
# -----------------------------------

mask_category_lookup = {
    (
        row.track_id,
        row.window_index,
        row.beat_index
    ): row.category

    for row in masked_analysis_df.itertuples(
        index=False
    )
}


# -----------------------------------
# 2. Diagnose one masked beat
# -----------------------------------

def diagnose_chord_mask(
    beat_start,
    beat_end,
    chords
):

    duration = beat_end - beat_start

    overlaps = []

    for chord in chords:

        if chord["start"] >= beat_end:
            break

        if chord["end"] <= beat_start:
            continue

        overlap = max(
            0.0,
            min(beat_end, chord["end"])
            - max(beat_start, chord["start"])
        )

        if overlap > 0:
            overlaps.append((overlap, chord))

    if not overlaps:

        return {
            "reason": "No recognized chord",
            "best_event_fraction": 0.0,
            "best_label_fraction": 0.0,
            "total_coverage": 0.0
        }

    # Largest individual chord event.
    best_overlap, best_chord = max(
        overlaps,
        key=lambda item: item[0]
    )

    best_event_fraction = (
        best_overlap / duration
    )

    # Combine fragments with the same
    # root and quality.
    label_durations = defaultdict(float)

    for overlap, chord in overlaps:

        label = (
            chord["root"],
            chord["quality"]
        )

        label_durations[label] += overlap

    best_label_fraction = (
        max(label_durations.values()) / duration
    )

    total_coverage = min(
        1.0,
        sum(o for o, _ in overlaps) / duration
    )

    # Reproduce the conflict rule used
    # during dataset construction.
    conflicting = any(
        overlap / duration >= 0.20
        and (
            chord["root"],
            chord["quality"]
        ) != (
            best_chord["root"],
            best_chord["quality"]
        )
        for overlap, chord in overlaps
        if chord is not best_chord
    )

    if best_event_fraction >= 0.80:

        if conflicting:
            reason = "Conflicting annotations"
        else:
            reason = "Unexpected masking"

    elif best_label_fraction >= 0.80:

        reason = "Fragmented same-chord events"

    elif total_coverage >= 0.80:

        reason = "Chord changes within beat"

    else:

        reason = "Insufficient chord coverage"

    return {
        "reason": reason,
        "best_event_fraction": best_event_fraction,
        "best_label_fraction": best_label_fraction,
        "total_coverage": total_coverage
    }


# -----------------------------------
# 3. Analyze chord-annotated
#    masked beats
# -----------------------------------

diagnostic_records = []

cached_track = None
cached_chords = None

with open(
    DRAFT_FILE,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        example = json.loads(line)

        track_id = example["track_id"]

        if track_id != cached_track:

            folder = track_lookup[track_id]

            chord_data = read_json(
                folder / "chords/submission.json"
            )

            cached_chords = prepare_window_chords(
                chord_data.get("events", [])
            )

            cached_track = track_id

        for target in example["beat_targets"]:

            if target["valid_mask"] == 1:
                continue

            key = (
                track_id,
                example["window_index"],
                target["beat_index"]
            )

            category = mask_category_lookup.get(
                key
            )

            if category != "Chord-annotated but masked":
                continue

            beat_start = (
                example["window_start_sec"]
                + target["time_sec"]
            )

            beat_end = (
                beat_start
                + target["duration_sec"]
            )

            result = diagnose_chord_mask(
                beat_start,
                beat_end,
                cached_chords
            )

            diagnostic_records.append({
                "track_id": track_id,
                "split": example["split"],
                "window_index":
                    example["window_index"],
                "beat_index":
                    target["beat_index"],
                **result
            })


chord_mask_diagnostic_df = pd.DataFrame(
    diagnostic_records
)


# -----------------------------------
# 4. Verify count
# -----------------------------------

print("MASKED CHORD ANNOTATION DIAGNOSTICS")
print("--------------------------------")

print(
    "Beats analyzed:",
    len(chord_mask_diagnostic_df)
)

assert len(chord_mask_diagnostic_df) == 53932


# -----------------------------------
# 5. Main causes
# -----------------------------------

reason_counts = (
    chord_mask_diagnostic_df["reason"]
    .value_counts()
)

reason_summary_df = pd.DataFrame({
    "Reason": reason_counts.index,
    "Beats": reason_counts.values,
    "Percentage": (
        100 * reason_counts.values
        / len(chord_mask_diagnostic_df)
    )
})

print("\nREASONS FOR MASKING")
print("--------------------------------")

display(
    reason_summary_df.round(2)
)


# -----------------------------------
# 6. How strongly does the largest
#    chord overlap each beat?
# -----------------------------------

print("\nBEST CHORD OVERLAP STATISTICS")
print("--------------------------------")

display(
    chord_mask_diagnostic_df[
        [
            "best_event_fraction",
            "best_label_fraction",
            "total_coverage"
        ]
    ].describe().round(3)
)


# -----------------------------------
# 7. Distribution by dataset split
# -----------------------------------

print("\nMASKING REASONS BY SPLIT")
print("--------------------------------")

display(
    pd.crosstab(
        chord_mask_diagnostic_df["reason"],
        chord_mask_diagnostic_df["split"]
    )
)


# -----------------------------------
# 8. Example problematic beats
# -----------------------------------

print("\nEXAMPLE MASKED BEATS")
print("--------------------------------")

display(
    chord_mask_diagnostic_df[
        [
            "track_id",
            "reason",
            "best_event_fraction",
            "best_label_fraction",
            "total_coverage"
        ]
    ].head(15).round(3)
)


print("\nChord masking diagnosis completed!")

MASKED CHORD ANNOTATION DIAGNOSTICS
--------------------------------
Beats analyzed: 53932

REASONS FOR MASKING
--------------------------------


,Reason,Beats,Percentage
0,Chord changes within beat,53787,99.73
1,Unexpected masking,93,0.17
2,Conflicting annotations,52,0.10



BEST CHORD OVERLAP STATISTICS
--------------------------------


,best_event_fraction,best_label_fraction,total_coverage
count,53932.000,53932.000,53932.0
mean,0.654,0.654,1.0
std,0.087,0.087,0.0
min,0.375,0.375,1.0
25%,0.578,0.578,1.0
50%,0.656,0.656,1.0
75%,0.729,0.729,1.0
max,0.800,0.800,1.0



MASKING REASONS BY SPLIT
--------------------------------


split,test,train,val
reason,,,
Chord changes within beat,7449,38066,8272
Conflicting annotations,4,41,7
Unexpected masking,11,66,16



EXAMPLE MASKED BEATS
--------------------------------


,track_id,reason,best_event_fraction,best_label_fraction,total_coverage
0,track_0000382,Chord changes within beat,0.554,0.554,1.0
1,track_0000382,Chord changes within beat,0.769,0.769,1.0
2,track_0000382,Chord changes within beat,0.554,0.554,1.0
3,track_0000382,Chord changes within beat,0.571,0.571,1.0
4,track_0000382,Chord changes within beat,0.654,0.654,1.0
5,track_0000382,Chord changes within beat,0.554,0.554,1.0
6,track_0000382,Chord changes within beat,0.571,0.571,1.0
7,track_0000382,Chord changes within beat,0.654,0.654,1.0
8,track_0000382,Chord changes within beat,0.683,0.683,1.0
9,track_0000382,Chord changes within beat,0.683,0.683,1.0



Chord masking diagnosis completed!


### Findings from Step 3.33
Analysis of masked chord targets revealed that 99.73% of chord-annotated masking cases involved chord changes within beats. Conflicting annotations and unexpected masking accounted for only a small proportion.
The initial 80% beat-level chord-overlap requirement was retained to avoid introducing uncertain training labels. Explicit no-chord predictions will be preserved as candidate labels but excluded from initial supervised training.
The next stage converts reliable normalized chord annotations into the proposed twelve-root, nine-family harmonic-plan representation.

### Roman-Numeral Target Encoding
Objective: Convert our normalized chord labels into the final root-and-family representation for the Transformer.
For example:
- Cmaj7 → I:MAJ
- G7 → V:DOM7
- Am7 → VI:MIN
We'll also distinguish reliable chords, no-chord candidates, and uncertain annotations. The original dataset will remain unchanged.

In [39]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd


# -----------------------------------
# 1. Output configuration
# -----------------------------------

PLAN_FILE = (
    DRAFT_DIR / "windows_v1_plan_labels.jsonl"
)

FAMILY_TO_ID = {
    family: index
    for index, family in enumerate(CHORD_FAMILIES)
}


# -----------------------------------
# 2. Build masked-target lookup
# -----------------------------------

masked_category_lookup = {
    (
        row.track_id,
        int(row.window_index),
        int(row.beat_index)
    ): row.category
    for row in masked_analysis_df.itertuples(
        index=False
    )
}


# -----------------------------------
# 3. Encode one beat target
# -----------------------------------

def encode_plan_target(target, track_id, window_index):

    result = target.copy()

    if target["valid_mask"] == 1:

        root = int(target["root_pc"])
        quality = target["quality"]

        family = CHORD_FAMILY_MAP[quality]

        result.update({
            "target_state": "CHORD",
            "root_id": root,
            "family": family,
            "family_id": FAMILY_TO_ID[family],
            "plan_label": chord_to_plan_label(
                root, quality
            ),
            "model_target_mask": 1
        })

    else:

        key = (
            track_id,
            int(window_index),
            int(target["beat_index"])
        )

        category = masked_category_lookup.get(key)

        if category is None:
            raise ValueError(
                f"Missing masking diagnosis: {key}"
            )

        is_n = category == "N-dominated"

        result.update({
            "target_state": (
                "N_CANDIDATE" if is_n else "MASK"
            ),
            "root_id": None,
            "family": None,
            "family_id": None,
            "plan_label": "N" if is_n else None,
            "model_target_mask": 0
        })

    return result


# -----------------------------------
# 4. Process complete dataset
# -----------------------------------

status_counts = Counter()
split_counts = Counter()

total_windows = 0
total_beats = 0

review_records = []

example_preview = None

with open(
    DRAFT_FILE,
    "r",
    encoding="utf-8"
) as input_file, open(
    PLAN_FILE,
    "w",
    encoding="utf-8"
) as output_file:

    for line in input_file:

        example = json.loads(line)

        track_id = example["track_id"]
        window_index = example["window_index"]

        encoded_targets = [
            encode_plan_target(
                target,
                track_id,
                window_index
            )
            for target in example["beat_targets"]
        ]

        example["beat_targets"] = encoded_targets

        total_windows += 1
        total_beats += len(encoded_targets)

        split_counts[example["split"]] += 1

        status_counts.update(
            target["target_state"]
            for target in encoded_targets
        )

        valid_count = sum(
            target["model_target_mask"]
            for target in encoded_targets
        )

        valid_fraction = (
            valid_count / len(encoded_targets)
            if encoded_targets else 0
        )

        # Preserve all windows.
        # Mark unresolved issues for final auditing.

        example["dataset_version"] = "phase3_v1"

        example["quality_flags"] = {
            "below_75_percent_valid_targets":
                valid_fraction < 0.75,

            "missing_artist_id":
                example["artist_id"] is None
        }

        if (
            valid_fraction < 0.75
            or example["artist_id"] is None
        ):

            review_records.append({
                "track_id": track_id,
                "window_index": window_index,
                "split": example["split"],
                "valid_fraction": valid_fraction,
                "missing_artist_id":
                    example["artist_id"] is None
            })

        # Verify target consistency.

        for target in encoded_targets:

            if target["target_state"] == "CHORD":

                assert target["root_id"] in range(12)

                assert target["family_id"] in range(
                    len(CHORD_FAMILIES)
                )

                assert target["model_target_mask"] == 1

            else:

                assert target["model_target_mask"] == 0

                assert target["root_id"] is None

                assert target["family_id"] is None

        assert len(example["genre_vector"]) == 5

        output_file.write(
            json.dumps(
                example,
                allow_nan=False
            ) + "\n"
        )

        if example_preview is None:
            example_preview = example


# -----------------------------------
# 5. Dataset statistics
# -----------------------------------

print("PHASE 3 — ROMAN-NUMERAL DATASET")
print("--------------------------------")

print("Saved:", PLAN_FILE)

print("Windows:", total_windows)
print("Beat targets:", total_beats)

print("\nTARGET STATES")
print("--------------------------------")

for state, count in status_counts.items():

    print(
        f"{state}: {count:,} "
        f"({100 * count / total_beats:.2f}%)"
    )


# -----------------------------------
# 6. Verify original dataset counts
# -----------------------------------

assert total_windows == 8952

assert total_beats == 535806

assert status_counts["CHORD"] == 478153

assert status_counts["N_CANDIDATE"] == 3172

assert (
    sum(status_counts.values()) == total_beats
)

print("\nDataset count checks passed.")


# -----------------------------------
# 7. Split statistics
# -----------------------------------

print("\nEXAMPLES PER SPLIT")
print("--------------------------------")

display(
    pd.DataFrame(
        split_counts.items(),
        columns=["Split", "Windows"]
    )
)


# -----------------------------------
# 8. Unresolved quality flags
# -----------------------------------

review_df = pd.DataFrame(review_records)

print("\nQUALITY REVIEW")
print("--------------------------------")

print(
    "Windows below 75% valid targets:",
    int(
        review_df[
            "valid_fraction"
        ].lt(0.75).sum()
    )
    if not review_df.empty else 0
)

print(
    "Windows with missing artist ID:",
    int(
        review_df["missing_artist_id"].sum()
    )
    if not review_df.empty else 0
)

display(
    review_df.head(15).round(3)
)


# -----------------------------------
# 9. Inspect example plan labels
# -----------------------------------

print("\nEXAMPLE PLAN TARGETS")
print("--------------------------------")

preview_targets = pd.DataFrame(
    example_preview["beat_targets"]
)

display(
    preview_targets[
        [
            "beat_index",
            "root_id",
            "quality",
            "family",
            "plan_label",
            "target_state",
            "model_target_mask"
        ]
    ].head(20)
)


print("\nRoman-numeral dataset encoding completed!")

PHASE 3 — ROMAN-NUMERAL DATASET
--------------------------------
Saved: /Users/praise/Developer/vocal-harmonization/datasets/mtg-jamendo/phase3-prepared/windows_v1_plan_labels.jsonl
Windows: 8952
Beat targets: 535806

TARGET STATES
--------------------------------
N_CANDIDATE: 3,172 (0.59%)
CHORD: 478,153 (89.24%)
MASK: 54,481 (10.17%)

Dataset count checks passed.

EXAMPLES PER SPLIT
--------------------------------


,Split,Windows
0,train,6324
1,test,1223
2,val,1405



QUALITY REVIEW
--------------------------------
Windows below 75% valid targets: 3
Windows with missing artist ID: 40


,track_id,window_index,split,valid_fraction,missing_artist_id
0,track_0079690,1,train,0.969,True
1,track_0079690,2,train,0.984,True
2,track_0079690,3,train,0.922,True
3,track_0079690,5,train,0.938,True
4,track_0079690,6,train,0.922,True
5,track_0079690,7,train,0.906,True
6,track_0079690,8,train,0.891,True
7,track_0079690,9,train,0.859,True
8,track_0079690,10,train,0.859,True
9,track_0156873,1,train,0.844,True



EXAMPLE PLAN TARGETS
--------------------------------


,beat_index,root_id,quality,family,plan_label,target_state,model_target_mask
0,0,NaN,NaN,NaN,N,N_CANDIDATE,0
1,1,5.0,maj,MAJ,IV:MAJ,CHORD,1
2,2,5.0,maj,MAJ,IV:MAJ,CHORD,1
3,3,5.0,maj,MAJ,IV:MAJ,CHORD,1
4,4,5.0,maj,MAJ,IV:MAJ,CHORD,1
5,5,5.0,maj,MAJ,IV:MAJ,CHORD,1
6,6,5.0,maj,MAJ,IV:MAJ,CHORD,1
7,7,5.0,maj,MAJ,IV:MAJ,CHORD,1
8,8,NaN,NaN,NaN,NaN,MASK,0
9,9,10.0,maj,MAJ,♭VII:MAJ,CHORD,1



Roman-numeral dataset encoding completed!


### Simplified Chord-Family Investigation
Objective: Investigate whether simplifying the proposed nine chord families into seven structural families produces a more practical harmonic-planning representation.
The revised representation combines DIM and DIM7 into DIM, combines SUS2 and SUS4 into SUS, and retains MAJ, MIN, DOM, HDIM, and AUG.
The analysis compares family frequencies and song-level representation in the training partition. The original nine-family dataset remains unchanged.
The decision will consider musical information loss, class imbalance, and the responsibilities of the subsequent chord-refinement component.

In [40]:
import pandas as pd


# -----------------------------------
# 1. Define simplified chord families
# -----------------------------------

SIMPLE_FAMILY_MAP = {

    "maj": "MAJ",
    "maj7": "MAJ",

    "min": "MIN",
    "min7": "MIN",
    "min9": "MIN",

    "7": "DOM",
    "9": "DOM",

    "dim": "DIM",
    "dim7": "DIM",

    "hdim7": "HDIM",

    "aug": "AUG",

    "sus2": "SUS",
    "sus4": "SUS",
    "sus4(b7)": "SUS"
}


SIMPLE_FAMILIES = [
    "MAJ",
    "MIN",
    "DOM",
    "DIM",
    "HDIM",
    "AUG",
    "SUS"
]


# -----------------------------------
# 2. Verify all qualities are supported
# -----------------------------------

observed = set(
    chord_vocabulary_df["Quality"]
)

assert observed == set(SIMPLE_FAMILY_MAP)

print("All observed qualities are supported.")


# -----------------------------------
# 3. Calculate simplified distribution
# -----------------------------------

records = []

total_train_beats = sum(
    quality_counts["train"].values()
)

for family in SIMPLE_FAMILIES:

    qualities = [
        quality
        for quality, mapped
        in SIMPLE_FAMILY_MAP.items()
        if mapped == family
    ]

    train_beats = sum(
        quality_counts["train"][q]
        for q in qualities
    )

    val_beats = sum(
        quality_counts["val"][q]
        for q in qualities
    )

    test_beats = sum(
        quality_counts["test"][q]
        for q in qualities
    )

    # Count unique training songs.
    train_songs = set()

    for quality in qualities:

        train_songs.update(
            quality_songs["train"][quality]
        )

    records.append({

        "Family": family,

        "Mapped qualities": ", ".join(qualities),

        "Train beats": train_beats,

        "Train (%)":
            100 * train_beats / total_train_beats,

        "Train songs": len(train_songs),

        "Val beats": val_beats,

        "Test beats": test_beats
    })


simple_family_df = pd.DataFrame(
    records
).sort_values(
    "Train beats",
    ascending=False
)


# -----------------------------------
# 4. Display new representation
# -----------------------------------

print("\nSIMPLIFIED FAMILY DISTRIBUTION")
print("--------------------------------")

display(
    simple_family_df.round(3)
)


# -----------------------------------
# 5. Compare representation sizes
# -----------------------------------

print("\nREPRESENTATION COMPARISON")
print("--------------------------------")

print(
    "Original pitched families:",
    len(CHORD_FAMILIES)
)

print(
    "Simplified pitched families:",
    len(SIMPLE_FAMILIES)
)

print(
    "Original theoretical root-family combinations:",
    12 * len(CHORD_FAMILIES)
)

print(
    "Simplified theoretical root-family combinations:",
    12 * len(SIMPLE_FAMILIES)
)


# -----------------------------------
# 6. Check rare class support
# -----------------------------------

print("\nRARE FAMILIES IN NEW REPRESENTATION")
print("--------------------------------")

display(
    simple_family_df[
        simple_family_df["Train songs"] < 30
    ]
)


# -----------------------------------
# 7. Verify no targets are lost
# -----------------------------------

assert (
    simple_family_df["Train beats"].sum()
    == total_train_beats
)

assert (
    simple_family_df["Val beats"].sum()
    == sum(quality_counts["val"].values())
)

assert (
    simple_family_df["Test beats"].sum()
    == sum(quality_counts["test"].values())
)

print("\nAll chord targets preserved.")
print("Simplified representation analysis completed!")

All observed qualities are supported.

SIMPLIFIED FAMILY DISTRIBUTION
--------------------------------


,Family,Mapped qualities,Train beats,Train (%),Train songs,Val beats,Test beats
0,MAJ,"maj, maj7",184341,55.112,750,41430,38549
1,MIN,"min, min7, min9",136822,40.905,736,33889,25088
2,DOM,"7, 9",11505,3.440,217,2192,1979
6,SUS,"sus2, sus4, sus4(b7)",1040,0.311,75,80,108
3,DIM,"dim, dim7",488,0.146,45,65,139
4,HDIM,hdim7,213,0.064,28,53,69
5,AUG,aug,76,0.023,12,9,18



REPRESENTATION COMPARISON
--------------------------------
Original pitched families: 9
Simplified pitched families: 7
Original theoretical root-family combinations: 108
Simplified theoretical root-family combinations: 84

RARE FAMILIES IN NEW REPRESENTATION
--------------------------------


,Family,Mapped qualities,Train beats,Train (%),Train songs,Val beats,Test beats
4,HDIM,hdim7,213,0.063680,28,53,69
5,AUG,aug,76,0.022721,12,9,18



All chord targets preserved.
Simplified representation analysis completed!


### Artist Identity and Split Integrity
Objective: Investigate missing artist identifiers and verify the integrity of the existing artist-disjoint dataset partitions.
The experiment checks whether individual songs or known artists appear in multiple partitions. It identifies songs with missing artist metadata and inspects their available source information.
Missing artist identifiers are investigated before deciding whether to recover the metadata or exclude the affected songs. The existing dataset and split assignments remain unchanged.
This is a preliminary leakage audit. Cross-split duplicate and near-duplicate recordings will be investigated separately.

In [41]:
import json
from pathlib import Path

import pandas as pd


# -----------------------------------
# 1. Load dataset metadata
# -----------------------------------

records = []

with open(PLAN_FILE, "r", encoding="utf-8") as file:

    for line in file:

        example = json.loads(line)

        artist_id = example.get("artist_id")

        if artist_id is not None:
            artist_id = str(artist_id).strip()

        if not artist_id:
            artist_id = None

        records.append({
            "track_id": example["track_id"],
            "window_index": example["window_index"],
            "split": example["split"],
            "artist_id": artist_id,
            "below_75": example["quality_flags"][
                "below_75_percent_valid_targets"
            ]
        })


artist_audit_df = pd.DataFrame(records)

print("ARTIST AND SPLIT INTEGRITY AUDIT")
print("--------------------------------")

print("Total windows:", len(artist_audit_df))
print(
    "Total songs:",
    artist_audit_df["track_id"].nunique()
)

print(
    "Known artists:",
    artist_audit_df["artist_id"].nunique()
)


# -----------------------------------
# 2. Check song-level split leakage
# -----------------------------------

song_split_counts = (
    artist_audit_df
    .groupby("track_id")["split"]
    .nunique()
)

leaking_songs = song_split_counts[
    song_split_counts > 1
]

print("\nSONG-LEVEL SPLIT LEAKAGE")
print("--------------------------------")

print(
    "Songs appearing in multiple splits:",
    len(leaking_songs)
)

if len(leaking_songs):
    display(leaking_songs)


# -----------------------------------
# 3. Check known-artist split leakage
# -----------------------------------

known_artist_df = artist_audit_df.dropna(
    subset=["artist_id"]
)

artist_split_counts = (
    known_artist_df
    .groupby("artist_id")["split"]
    .nunique()
)

leaking_artists = artist_split_counts[
    artist_split_counts > 1
]

print("\nARTIST-LEVEL SPLIT LEAKAGE")
print("--------------------------------")

print(
    "Known artists appearing in multiple splits:",
    len(leaking_artists)
)

if len(leaking_artists):
    display(leaking_artists)


# -----------------------------------
# 4. Find songs missing artist IDs
# -----------------------------------

missing_artist_df = artist_audit_df[
    artist_audit_df["artist_id"].isna()
].copy()

missing_song_summary = (
    missing_artist_df
    .groupby(["track_id", "split"])
    .agg(
        windows=("window_index", "count"),
        low_coverage_windows=("below_75", "sum")
    )
    .reset_index()
)

print("\nSONGS WITH MISSING ARTIST IDs")
print("--------------------------------")

print(
    "Songs:",
    missing_artist_df["track_id"].nunique()
)

print(
    "Windows:",
    len(missing_artist_df)
)

display(missing_song_summary)


# -----------------------------------
# 5. Inspect existing metadata
# -----------------------------------

print("\nSOURCE METADATA INVESTIGATION")
print("--------------------------------")

metadata_rows = []

for track_id in missing_song_summary["track_id"]:

    matches = genre_metadata_df[
        genre_metadata_df["track_id"] == track_id
    ]

    metadata_found = len(matches) > 0

    existing_artist_ids = []

    if metadata_found and "artist_id" in matches.columns:

        existing_artist_ids = (
            matches["artist_id"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    folder = Path(track_lookup[track_id])

    # List available top-level files/folders
    # without modifying anything.

    folder_contents = (
        sorted(p.name for p in folder.iterdir())
        if folder.exists()
        else []
    )

    metadata_rows.append({
        "track_id": track_id,
        "metadata_row_found": metadata_found,
        "artist_ids_in_metadata": existing_artist_ids,
        "folder_contents": ", ".join(folder_contents[:15])
    })


missing_metadata_df = pd.DataFrame(metadata_rows)

display(missing_metadata_df)


# -----------------------------------
# 6. Identify low-coverage windows
# -----------------------------------

print("\nWINDOWS BELOW 75% TARGET COVERAGE")
print("--------------------------------")

low_coverage_df = artist_audit_df[
    artist_audit_df["below_75"]
]

display(low_coverage_df)


# -----------------------------------
# 7. Overall integrity summary
# -----------------------------------

print("\nINTEGRITY SUMMARY")
print("--------------------------------")

print(
    "Cross-split song overlaps:",
    len(leaking_songs)
)

print(
    "Cross-split known-artist overlaps:",
    len(leaking_artists)
)

print(
    "Songs with unknown artists:",
    missing_artist_df["track_id"].nunique()
)

print(
    "Windows below 75% coverage:",
    len(low_coverage_df)
)

print("\nArtist identity audit completed!")

ARTIST AND SPLIT INTEGRITY AUDIT
--------------------------------
Total windows: 8952
Total songs: 1076
Known artists: 218

SONG-LEVEL SPLIT LEAKAGE
--------------------------------
Songs appearing in multiple splits: 0

ARTIST-LEVEL SPLIT LEAKAGE
--------------------------------
Known artists appearing in multiple splits: 0

SONGS WITH MISSING ARTIST IDs
--------------------------------
Songs: 7
Windows: 40


,track_id,split,windows,low_coverage_windows
0,track_0079690,train,9,0
1,track_0156873,train,3,0
2,track_0156888,train,12,0
3,track_0480340,train,5,0
4,track_1045346,train,7,0
5,track_1082101,train,2,0
6,track_1082102,train,2,0



SOURCE METADATA INVESTIGATION
--------------------------------


,track_id,metadata_row_found,artist_ids_in_metadata,folder_contents
0,track_0079690,False,[],".DS_Store, beats, bpdr, chords, key.json, key_..."
1,track_0156873,False,[],".DS_Store, beats, chords, key.json, key_compar..."
2,track_0156888,False,[],".DS_Store, beats, chords, key.json, key_compar..."
3,track_0480340,False,[],".DS_Store, beats, chords, key.json, key_compar..."
4,track_1045346,False,[],".DS_Store, beats, chords, key.json, key_compar..."
5,track_1082101,False,[],".DS_Store, beats, bpdr, chords, key.json, key_..."
6,track_1082102,False,[],".DS_Store, beats, bpdr, chords, key.json, key_..."



WINDOWS BELOW 75% TARGET COVERAGE
--------------------------------


,track_id,window_index,split,artist_id,below_75
2770,track_0961261,8,test,artist_356592,True
3616,track_1037054,2,train,artist_372655,True
5498,track_1087536,1,train,artist_004142,True



INTEGRITY SUMMARY
--------------------------------
Cross-split song overlaps: 0
Cross-split known-artist overlaps: 0
Songs with unknown artists: 7
Windows below 75% coverage: 3

Artist identity audit completed!


### Cross-Split Duplicate Screening
Objective: Identify potential duplicate or closely related melodies across the existing training, validation, and test partitions.
Seven songs without artist identifiers and three windows with insufficient target coverage are provisionally excluded from the primary dataset. The remaining song partitions are preserved.
Potential duplicates are identified using transposition-invariant melodic interval patterns. Matching patterns between songs in different partitions are reported for further investigation.
This analysis is a preliminary leakage screen. A negative result does not guarantee that all covers, alternate recordings, or near-duplicates have been detected. No dataset files are modified.

In [42]:
import json
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd


# -----------------------------------
# 1. Define eligible training windows
# -----------------------------------

eligible_window_df = artist_audit_df[
    artist_audit_df["artist_id"].notna()
    &
    (~artist_audit_df["below_75"])
].copy()


eligible_keys = set(
    zip(
        eligible_window_df["track_id"],
        eligible_window_df["window_index"]
    )
)


print("DATASET FILTERING")
print("--------------------------------")

print(
    "Original windows:",
    len(artist_audit_df)
)

print(
    "Eligible windows:",
    len(eligible_window_df)
)

print(
    "Excluded windows:",
    len(artist_audit_df) - len(eligible_window_df)
)

print(
    "Eligible songs:",
    eligible_window_df["track_id"].nunique()
)


print("\nELIGIBLE SPLIT DISTRIBUTION")
print("--------------------------------")

display(
    eligible_window_df["split"].value_counts()
)


# -----------------------------------
# 2. Create melodic fingerprints
# -----------------------------------

NGRAM_NOTES = 12

song_fingerprints = defaultdict(set)
song_splits = {}

with open(
    PLAN_FILE,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        example = json.loads(line)

        track_id = example["track_id"]
        window_index = example["window_index"]

        if (track_id, window_index) not in eligible_keys:
            continue

        song_splits[track_id] = example["split"]

        # Include the song even if it has no
        # sufficiently distinctive fingerprints.
        _ = song_fingerprints[track_id]

        pitches = [
            int(note["pitch_class"])
            for note in example["melody"]
        ]

        if len(pitches) < NGRAM_NOTES:
            continue

        # Melodic intervals are invariant
        # under pitch transposition.
        intervals = [
            (pitches[i + 1] - pitches[i]) % 12
            for i in range(len(pitches) - 1)
        ]

        for i in range(
            len(pitches) - NGRAM_NOTES + 1
        ):

            pitch_segment = pitches[
                i:i + NGRAM_NOTES
            ]

            interval_segment = tuple(
                intervals[
                    i:i + NGRAM_NOTES - 1
                ]
            )

            # Ignore very repetitive sequences.
            # They can occur in unrelated songs.
            if len(set(pitch_segment)) < 4:
                continue

            if len(set(interval_segment)) < 4:
                continue

            song_fingerprints[track_id].add(
                interval_segment
            )


# -----------------------------------
# 3. Build fingerprint lookup
# -----------------------------------

fingerprint_lookup = defaultdict(set)

for track_id, fingerprints in song_fingerprints.items():

    for fingerprint in fingerprints:

        fingerprint_lookup[fingerprint].add(
            track_id
        )


# -----------------------------------
# 4. Compare songs across splits
# -----------------------------------

pair_matches = Counter()

for fingerprint, tracks in fingerprint_lookup.items():

    # Very common melodic patterns are not
    # useful for identifying duplicates.
    if len(tracks) > 10:
        continue

    for track_a, track_b in combinations(
        sorted(tracks), 2
    ):

        # Only investigate cross-split matches.
        if song_splits[track_a] == song_splits[track_b]:
            continue

        pair_matches[
            (track_a, track_b)
        ] += 1


# -----------------------------------
# 5. Calculate similarity scores
# -----------------------------------

similarity_records = []

for (track_a, track_b), shared in pair_matches.items():

    count_a = len(
        song_fingerprints[track_a]
    )

    count_b = len(
        song_fingerprints[track_b]
    )

    if min(count_a, count_b) == 0:
        continue

    coverage = shared / min(count_a, count_b)

    union_count = count_a + count_b - shared

    jaccard = (
        shared / union_count
        if union_count > 0 else 0
    )

    similarity_records.append({

        "track_a": track_a,
        "split_a": song_splits[track_a],

        "track_b": track_b,
        "split_b": song_splits[track_b],

        "shared_patterns": shared,

        "smaller_song_coverage": coverage,

        "jaccard_similarity": jaccard
    })


similarity_df = pd.DataFrame(
    similarity_records,
    columns=[
        "track_a", "split_a",
        "track_b", "split_b",
        "shared_patterns",
        "smaller_song_coverage",
        "jaccard_similarity"
    ]
)

similarity_df = similarity_df.sort_values(
    [
        "shared_patterns",
        "smaller_song_coverage"
    ],
    ascending=False
).reset_index(drop=True)


# -----------------------------------
# 6. Display duplicate candidates
# -----------------------------------

print("\nCROSS-SPLIT MELODY DUPLICATE SCREEN")
print("--------------------------------")

print(
    "Songs analyzed:",
    len(song_fingerprints)
)

print(
    "Songs without distinctive fingerprints:",
    sum(
        len(v) == 0
        for v in song_fingerprints.values()
    )
)

print(
    "Distinct melodic patterns:",
    len(fingerprint_lookup)
)

print(
    "Cross-split pairs sharing at least one pattern:",
    len(similarity_df)
)


# Three shared patterns are sufficient
# to flag a candidate for manual review.
# This is not a duplicate classification.

duplicate_candidates_df = similarity_df[
    similarity_df["shared_patterns"] >= 3
].copy()


print("\nPOTENTIAL DUPLICATE CANDIDATES")
print("--------------------------------")

print(
    "Candidate pairs:",
    len(duplicate_candidates_df)
)

display(
    duplicate_candidates_df.head(25).round(3)
)


# -----------------------------------
# 7. Strongest matches
# -----------------------------------

print("\nTOP CROSS-SPLIT MATCHES")
print("--------------------------------")

if similarity_df.empty:

    print(
        "No shared distinctive melodic "
        "patterns detected across splits."
    )

else:

    display(
        similarity_df.head(10).round(3)
    )


print("\nDuplicate screening completed!")

DATASET FILTERING
--------------------------------
Original windows: 8952
Eligible windows: 8909
Excluded windows: 43
Eligible songs: 1069

ELIGIBLE SPLIT DISTRIBUTION
--------------------------------


split
train    6282
val      1405
test     1222
Name: count, dtype: int64


CROSS-SPLIT MELODY DUPLICATE SCREEN
--------------------------------
Songs analyzed: 1069
Songs without distinctive fingerprints: 1
Distinct melodic patterns: 281231
Cross-split pairs sharing at least one pattern: 112

POTENTIAL DUPLICATE CANDIDATES
--------------------------------
Candidate pairs: 12


,track_a,split_a,track_b,split_b,shared_patterns,smaller_song_coverage,jaccard_similarity
0,track_1056811,val,track_1072573,train,4,0.014,0.004
1,track_0576998,val,track_1162013,train,4,0.010,0.004
2,track_1085330,val,track_1162013,train,4,0.010,0.004
3,track_1113871,test,track_1167425,train,4,0.010,0.003
4,track_1056813,val,track_1320933,train,3,0.021,0.006
5,track_0577048,val,track_1143806,train,3,0.013,0.004
6,track_1119472,val,track_1159743,train,3,0.010,0.004
7,track_1060329,train,track_1085328,val,3,0.010,0.002
8,track_0076660,train,track_0964541,test,3,0.009,0.004
9,track_1056813,val,track_1072571,train,3,0.009,0.004



TOP CROSS-SPLIT MATCHES
--------------------------------


,track_a,split_a,track_b,split_b,shared_patterns,smaller_song_coverage,jaccard_similarity
0,track_1056811,val,track_1072573,train,4,0.014,0.004
1,track_0576998,val,track_1162013,train,4,0.010,0.004
2,track_1085330,val,track_1162013,train,4,0.010,0.004
3,track_1113871,test,track_1167425,train,4,0.010,0.003
4,track_1056813,val,track_1320933,train,3,0.021,0.006
5,track_0577048,val,track_1143806,train,3,0.013,0.004
6,track_1119472,val,track_1159743,train,3,0.010,0.004
7,track_1060329,train,track_1085328,val,3,0.010,0.002
8,track_0076660,train,track_0964541,test,3,0.009,0.004
9,track_1056813,val,track_1072571,train,3,0.009,0.004



Duplicate screening completed!


### Findings from Step 3.37
Transposition-invariant melodic fingerprint screening was performed across 1,069 eligible songs.
Twelve cross-split song pairs shared at least three melodic interval patterns. However, all detected similarities were low, with maximum Jaccard similarity of 0.006. No strong duplicate candidates were identified using this method.
Existing split assignments were retained. The flagged pairs will remain available for supplementary inspection, as melodic fingerprint screening cannot guarantee the absence of alternate recordings or cover versions.
Following provisional filtering, the dataset contains 8,909 windows across 1,069 songs.

### Musical Timing Validation
Objective: Validate the timing metadata needed for beat-aware Transformer inputs.
The experiment checks whether saved beat targets match the original beat annotations, examines beat positions within bars, identifies time signatures, and verifies the expected 16-bar window structure.
This validation uses the 8,909 provisionally eligible windows. No data is modified.

In [43]:
import json
from bisect import bisect_left
from collections import Counter

import numpy as np
import pandas as pd


# -----------------------------------
# 1. Configuration
# -----------------------------------

TIME_TOLERANCE = 0.005  # 5 milliseconds

timing_records = []

meter_counts = Counter()
position_counts = Counter()

total_beats = 0
matched_beats = 0
missing_positions = 0
missing_meters = 0
missing_downbeat_flags = 0

alignment_problems = []

cached_track = None
cached_beats = None
cached_times = None


# -----------------------------------
# 2. Process eligible windows
# -----------------------------------

with open(
    PLAN_FILE,
    "r",
    encoding="utf-8"
) as file:

    for line in file:

        example = json.loads(line)

        track_id = example["track_id"]
        window_index = example["window_index"]

        if (track_id, window_index) not in eligible_keys:
            continue

        # Load original beats once per song.

        if track_id != cached_track:

            beat_data = read_json(
                track_lookup[track_id]
                / "beats/beats.json"
            )

            cached_beats = sorted(
                [
                    beat
                    for beat in beat_data.get("beats", [])
                    if np.isfinite(
                        float(beat["time_sec"])
                    )
                ],
                key=lambda b: float(b["time_sec"])
            )

            cached_times = [
                float(b["time_sec"])
                for b in cached_beats
            ]

            cached_track = track_id

        window_start = float(
            example["window_start_sec"]
        )

        window_matched = 0
        window_downbeats = 0

        window_meters = set()
        window_positions = []

        for target in example["beat_targets"]:

            total_beats += 1

            absolute_time = (
                window_start
                + float(target["time_sec"])
            )

            idx = bisect_left(
                cached_times,
                absolute_time
            )

            # Find nearest original beat.

            candidates = [
                j for j in (idx - 1, idx)
                if 0 <= j < len(cached_times)
            ]

            if not candidates:
                continue

            closest = min(
                candidates,
                key=lambda j: abs(
                    cached_times[j] - absolute_time
                )
            )

            difference = abs(
                cached_times[closest] - absolute_time
            )

            if difference > TIME_TOLERANCE:

                if len(alignment_problems) < 15:

                    alignment_problems.append({
                        "track_id": track_id,
                        "window_index": window_index,
                        "beat_index":
                            target["beat_index"],
                        "difference_sec": difference
                    })

                continue

            matched_beats += 1
            window_matched += 1

            beat = cached_beats[closest]

            # -----------------------------------
            # 3. Beat position within bar
            # -----------------------------------

            position = beat.get("beat_in_bar")

            if position is None:

                missing_positions += 1

            else:

                position = int(position)

                position_counts[position] += 1
                window_positions.append(position)

            # -----------------------------------
            # 4. Time signature
            # -----------------------------------

            meter = beat.get("meter_numerator")

            if meter is None:

                missing_meters += 1

            else:

                meter = int(meter)

                meter_counts[meter] += 1
                window_meters.add(meter)

            # -----------------------------------
            # 5. Downbeat annotation
            # -----------------------------------

            downbeat = beat.get("is_downbeat")

            if downbeat is None:

                missing_downbeat_flags += 1

            elif downbeat in (True, 1):

                window_downbeats += 1

        timing_records.append({
            "track_id": track_id,
            "window_index": window_index,
            "split": example["split"],
            "target_beats": len(example["beat_targets"]),
            "matched_beats": window_matched,
            "downbeats": window_downbeats,
            "meter_count": len(window_meters),
            "meters": sorted(window_meters),
            "first_position": (
                window_positions[0]
                if window_positions else None
            )
        })


# -----------------------------------
# 6. Create summary dataframe
# -----------------------------------

timing_validation_df = pd.DataFrame(
    timing_records
)

assert len(timing_validation_df) == len(
    eligible_window_df
)


print("PHASE 3 — MUSICAL TIMING VALIDATION")
print("--------------------------------")

print(
    "Windows:",
    len(timing_validation_df)
)

print(
    "Total beat targets:",
    total_beats
)

print(
    "Matched beat targets:",
    matched_beats
)

print(
    "Unmatched beat targets:",
    total_beats - matched_beats
)

print(
    "Alignment percentage:",
    round(
        100 * matched_beats / total_beats,
        3
    ),
    "%"
)

print(
    "Missing beat positions:",
    missing_positions
)

print(
    "Missing metre annotations:",
    missing_meters
)

print(
    "Missing downbeat flags:",
    missing_downbeat_flags
)


# -----------------------------------
# 7. Meter distribution
# -----------------------------------

print("\nTIME SIGNATURE DISTRIBUTION")
print("--------------------------------")

meter_df = pd.DataFrame(
    [
        {
            "Beats per bar": meter,
            "Beat targets": count,
            "Percentage":
                100 * count / matched_beats
        }
        for meter, count in meter_counts.items()
    ]
).sort_values(
    "Beat targets",
    ascending=False
)

display(meter_df.round(3))


# -----------------------------------
# 8. Beat positions within bars
# -----------------------------------

print("\nBEAT POSITION DISTRIBUTION")
print("--------------------------------")

display(
    pd.DataFrame(
        sorted(position_counts.items()),
        columns=[
            "Beat position",
            "Occurrences"
        ]
    )
)


# -----------------------------------
# 9. Bar structure validation
# -----------------------------------

print("\nDOWNBEATS PER WINDOW")
print("--------------------------------")

display(
    timing_validation_df["downbeats"]
    .value_counts()
    .sort_index()
    .rename_axis("Downbeats")
    .reset_index(name="Windows")
)

print(
    "\nWindows with 16 downbeats:",
    int(
        (timing_validation_df["downbeats"] == 16)
        .sum()
    )
)


# -----------------------------------
# 10. Unusual windows
# -----------------------------------

print("\nWINDOWS REQUIRING REVIEW")
print("--------------------------------")

unusual_windows = timing_validation_df[
    (
        timing_validation_df["matched_beats"]
        != timing_validation_df["target_beats"]
    )
    |
    (
        timing_validation_df["downbeats"] != 16
    )
    |
    (
        timing_validation_df["meter_count"] != 1
    )
]

print(
    "Potentially unusual windows:",
    len(unusual_windows)
)

display(
    unusual_windows.head(15)
)


# -----------------------------------
# 11. Timestamp mismatches
# -----------------------------------

print("\nTIMESTAMP ALIGNMENT PROBLEMS")
print("--------------------------------")

if alignment_problems:

    display(
        pd.DataFrame(
            alignment_problems
        ).round(6)
    )

else:

    print("No timestamp mismatches detected.")


print("\nMusical timing validation completed!")

PHASE 3 — MUSICAL TIMING VALIDATION
--------------------------------
Windows: 8909
Total beat targets: 533710
Matched beat targets: 533710
Unmatched beat targets: 0
Alignment percentage: 100.0 %
Missing beat positions: 0
Missing metre annotations: 0
Missing downbeat flags: 0

TIME SIGNATURE DISTRIBUTION
--------------------------------


,Beats per bar,Beat targets,Percentage
0,4,402900,75.490
3,6,56448,10.577
2,2,45902,8.601
1,3,28290,5.301
4,5,105,0.020
6,7,56,0.010
5,9,9,0.002



BEAT POSITION DISTRIBUTION
--------------------------------


,Beat position,Occurrences
0,1,142544
1,2,142544
2,3,119593
3,4,110163
4,5,9438
5,6,9417
6,7,9
7,8,1
8,9,1



DOWNBEATS PER WINDOW
--------------------------------


,Downbeats,Windows
0,16,8909



Windows with 16 downbeats: 8909

WINDOWS REQUIRING REVIEW
--------------------------------
Potentially unusual windows: 80


,track_id,window_index,split,target_beats,matched_beats,downbeats,meter_count,meters,first_position
78,track_0021984,0,train,63,63,16,2,"[3, 4]",1
293,track_0044671,2,val,63,63,16,2,"[3, 4]",1
294,track_0044671,3,val,63,63,16,2,"[3, 4]",1
390,track_0047052,0,train,63,63,16,2,"[3, 4]",1
460,track_0072837,0,val,65,65,16,2,"[4, 5]",1
639,track_0207564,0,val,63,63,16,2,"[3, 4]",1
644,track_0210969,0,test,63,63,16,2,"[3, 4]",1
652,track_0211291,0,test,65,65,16,2,"[4, 5]",1
766,track_0243697,1,train,69,69,16,2,"[4, 9]",1
824,track_0243809,0,train,36,36,16,3,"[2, 3, 5]",1



TIMESTAMP ALIGNMENT PROBLEMS
--------------------------------
No timestamp mismatches detected.

Musical timing validation completed!


### Create the Seven-Family Dataset with Musical Timing
Objective: Construct a new version of the harmonic-planning dataset using seven chord families and validated musical timing features.
The dataset will preserve the existing artist-disjoint train, validation, and test assignments, excluding the 43 previously flagged windows.
Each example will contain normalized melody notes, a genre vector, a beat grid with bar positions and metre information, and beat-level chord targets represented by twelve relative roots and seven chord families.
The original nine-family labels and detailed chord qualities will be preserved for future comparisons. Mixed-metre windows and timing inconsistencies will be flagged rather than silently removed.
The resulting dataset remains provisional pending final integrity and annotation-quality checks.

In [44]:
import json
from bisect import bisect_left
from collections import Counter, defaultdict

import pandas as pd


# -----------------------------------
# 1. Configuration
# -----------------------------------

V2_FILE = (
    DRAFT_DIR
    / "windows_v2_7family_timing_provisional.jsonl"
)

SIMPLE_FAMILY_TO_ID = {
    family: i
    for i, family in enumerate(SIMPLE_FAMILIES)
}

TIME_TOLERANCE = 0.005

split_counts = Counter()
target_counts = Counter()

mixed_meter_windows = 0
within_bar_meter_issues = 0
beat_position_issues = 0

total_windows = 0
total_beats = 0

example_preview = None

cached_track = None
cached_beats = None
cached_times = None


# -----------------------------------
# 2. Construct Version 2
# -----------------------------------

with open(
    PLAN_FILE,
    "r",
    encoding="utf-8"
) as input_file, open(
    V2_FILE,
    "w",
    encoding="utf-8"
) as output_file:

    for line in input_file:

        example = json.loads(line)

        track_id = example["track_id"]
        window_index = example["window_index"]

        # Exclude previously flagged windows.

        if (track_id, window_index) not in eligible_keys:
            continue

        # -----------------------------------
        # 3. Load original beat annotations
        # -----------------------------------

        if track_id != cached_track:

            beat_data = read_json(
                track_lookup[track_id]
                / "beats/beats.json"
            )

            cached_beats = sorted(
                beat_data["beats"],
                key=lambda b: float(b["time_sec"])
            )

            cached_times = [
                float(b["time_sec"])
                for b in cached_beats
            ]

            cached_track = track_id

        window_start = float(
            example["window_start_sec"]
        )

        # -----------------------------------
        # 4. Build beat-grid input features
        # -----------------------------------

        beat_grid = []

        bar_index = -1
        bar_meters = defaultdict(set)

        position_conflicts = 0

        for target in example["beat_targets"]:

            absolute_time = (
                window_start
                + float(target["time_sec"])
            )

            idx = bisect_left(
                cached_times,
                absolute_time
            )

            candidates = [
                j for j in (idx - 1, idx)
                if 0 <= j < len(cached_times)
            ]

            assert candidates

            closest = min(
                candidates,
                key=lambda j: abs(
                    cached_times[j] - absolute_time
                )
            )

            difference = abs(
                cached_times[closest] - absolute_time
            )

            assert difference <= TIME_TOLERANCE, (
                f"Beat alignment error: {track_id}"
            )

            beat = cached_beats[closest]

            position = int(
                beat["beat_in_bar"]
            )

            meter = int(
                beat["meter_numerator"]
            )

            is_downbeat = bool(
                beat["is_downbeat"]
            )

            if is_downbeat:
                bar_index += 1

            assert bar_index >= 0

            bar_meters[bar_index].add(meter)

            if (
                position < 1
                or position > meter
                or is_downbeat != (position == 1)
            ):
                position_conflicts += 1

            beat_grid.append({

                "beat_index":
                    int(target["beat_index"]),

                "time_sec":
                    float(target["time_sec"]),

                "duration_sec":
                    float(target["duration_sec"]),

                "bar_index": bar_index,

                "beat_in_bar": position,

                "meter_numerator": meter,

                "is_downbeat": is_downbeat
            })

        # Every selected window has 16 bars.

        assert bar_index + 1 == 16

        assert len(beat_grid) == len(
            example["beat_targets"]
        )

        # -----------------------------------
        # 5. Encode seven-family targets
        # -----------------------------------

        new_targets = []

        for target in example["beat_targets"]:

            encoded = target.copy()

            # Preserve the original
            # nine-family representation.

            encoded["family_9"] = target["family"]

            encoded["family_id_9"] = (
                target["family_id"]
            )

            encoded["plan_label_9"] = (
                target["plan_label"]
            )

            if target["target_state"] == "CHORD":

                quality = target["quality"]

                family = SIMPLE_FAMILY_MAP[quality]

                family_id = SIMPLE_FAMILY_TO_ID[
                    family
                ]

                root_id = int(
                    target["root_id"]
                )

                encoded["family"] = family

                encoded["family_id"] = family_id

                encoded["plan_label"] = (
                    f"{RELATIVE_ROOT_LABELS[root_id]}"
                    f":{family}"
                )

                assert (
                    encoded["model_target_mask"] == 1
                )

            else:

                encoded["family"] = None
                encoded["family_id"] = None

                if (
                    target["target_state"]
                    == "N_CANDIDATE"
                ):
                    encoded["plan_label"] = "N.C."

                else:
                    encoded["plan_label"] = None

                assert (
                    encoded["model_target_mask"] == 0
                )

            new_targets.append(encoded)

            target_counts[
                encoded["target_state"]
            ] += 1

        # -----------------------------------
        # 6. Record metre-quality flags
        # -----------------------------------

        unique_meters = {
            meter
            for meters in bar_meters.values()
            for meter in meters
        }

        mixed_meter = len(unique_meters) > 1

        within_bar_issue = any(
            len(meters) > 1
            for meters in bar_meters.values()
        )

        mixed_meter_windows += int(
            mixed_meter
        )

        within_bar_meter_issues += int(
            within_bar_issue
        )

        beat_position_issues += int(
            position_conflicts > 0
        )

        # -----------------------------------
        # 7. Save updated training example
        # -----------------------------------

        example["dataset_version"] = "phase3_v2"

        example["beat_grid"] = beat_grid

        example["beat_targets"] = new_targets

        example["quality_flags"].update({

            "mixed_meter": mixed_meter,

            "within_bar_meter_disagreement":
                within_bar_issue,

            "beat_position_conflict":
                position_conflicts > 0
        })

        example["representation"] = {
            "relative_roots": 12,
            "chord_families": SIMPLE_FAMILIES,
            "no_chord_supervised": False
        }

        output_file.write(
            json.dumps(
                example,
                allow_nan=False
            ) + "\n"
        )

        total_windows += 1
        total_beats += len(new_targets)

        split_counts[example["split"]] += 1

        if example_preview is None:
            example_preview = example


# -----------------------------------
# 8. Validate dataset size
# -----------------------------------

print("PHASE 3 — DATASET VERSION 2")
print("--------------------------------")

print("Saved:", V2_FILE)

print("Windows:", total_windows)
print("Beat targets:", total_beats)

assert total_windows == 8909

assert total_beats == 533710

assert split_counts["train"] == 6282
assert split_counts["val"] == 1405
assert split_counts["test"] == 1222

print("\nDataset size checks passed.")


# -----------------------------------
# 9. Display split distribution
# -----------------------------------

print("\nSPLIT DISTRIBUTION")
print("--------------------------------")

display(
    pd.DataFrame(
        split_counts.items(),
        columns=["Split", "Windows"]
    )
)


# -----------------------------------
# 10. Display target distribution
# -----------------------------------

print("\nTARGET STATES")
print("--------------------------------")

display(
    pd.DataFrame(
        target_counts.items(),
        columns=["Target state", "Beats"]
    )
)


# -----------------------------------
# 11. Check musical timing flags
# -----------------------------------

print("\nMUSICAL TIMING QUALITY")
print("--------------------------------")

print(
    "Mixed-metre windows:",
    mixed_meter_windows
)

print(
    "Windows with metre disagreement "
    "inside a bar:",
    within_bar_meter_issues
)

print(
    "Windows with beat-position conflicts:",
    beat_position_issues
)


# -----------------------------------
# 12. Preview final representation
# -----------------------------------

print("\nEXAMPLE BEAT GRID")
print("--------------------------------")

display(
    pd.DataFrame(
        example_preview["beat_grid"]
    ).head(12)
)


print("\nEXAMPLE SEVEN-FAMILY TARGETS")
print("--------------------------------")

preview_df = pd.DataFrame(
    example_preview["beat_targets"]
)

display(
    preview_df[
        [
            "beat_index",
            "root_id",
            "quality",
            "family",
            "plan_label",
            "target_state",
            "model_target_mask"
        ]
    ].head(15)
)


print("\nVersion 2 construction completed!")

PHASE 3 — DATASET VERSION 2
--------------------------------
Saved: /Users/praise/Developer/vocal-harmonization/datasets/mtg-jamendo/phase3-prepared/windows_v2_7family_timing_provisional.jsonl
Windows: 8909
Beat targets: 533710

Dataset size checks passed.

SPLIT DISTRIBUTION
--------------------------------


,Split,Windows
0,train,6282
1,test,1222
2,val,1405



TARGET STATES
--------------------------------


,Target state,Beats
0,N_CANDIDATE,3155
1,CHORD,476303
2,MASK,54252



MUSICAL TIMING QUALITY
--------------------------------
Mixed-metre windows: 80
Windows with metre disagreement inside a bar: 0
Windows with beat-position conflicts: 0

EXAMPLE BEAT GRID
--------------------------------


,beat_index,time_sec,duration_sec,bar_index,beat_in_bar,meter_numerator,is_downbeat
0,0,0.00,0.47,0,1,4,True
1,1,0.47,0.53,0,2,4,False
2,2,1.00,0.60,0,3,4,False
3,3,1.60,0.59,0,4,4,False
4,4,2.19,0.56,1,1,4,True
5,5,2.75,0.44,1,2,4,False
6,6,3.19,0.44,1,3,4,False
7,7,3.63,0.48,1,4,4,False
8,8,4.11,0.56,2,1,4,True
9,9,4.67,0.52,2,2,4,False



EXAMPLE SEVEN-FAMILY TARGETS
--------------------------------


,beat_index,root_id,quality,family,plan_label,target_state,model_target_mask
0,0,NaN,NaN,NaN,N.C.,N_CANDIDATE,0
1,1,5.0,maj,MAJ,IV:MAJ,CHORD,1
2,2,5.0,maj,MAJ,IV:MAJ,CHORD,1
3,3,5.0,maj,MAJ,IV:MAJ,CHORD,1
4,4,5.0,maj,MAJ,IV:MAJ,CHORD,1
5,5,5.0,maj,MAJ,IV:MAJ,CHORD,1
6,6,5.0,maj,MAJ,IV:MAJ,CHORD,1
7,7,5.0,maj,MAJ,IV:MAJ,CHORD,1
8,8,NaN,NaN,NaN,NaN,MASK,0
9,9,10.0,maj,MAJ,♭VII:MAJ,CHORD,1



Version 2 construction completed!


### Final Dataset Consistency Audit
Objective: Perform a structural consistency audit of the seven-family harmonic-planning dataset.
The experiment validates normalized melody pitches, chord-family mappings, root identifiers, training masks, beat-grid structure, and artist-disjoint dataset partitions.
It also measures which root–family combinations are represented in each partition and identifies combinations absent from training.
The analysis does not modify any dataset files. Any detected inconsistencies will be investigated before creating the PyTorch dataset loader.

In [45]:
import json
from collections import Counter, defaultdict

import pandas as pd


# -----------------------------------
# 1. Initialize audit
# -----------------------------------

issues = Counter()
examples_with_issues = []

split_windows = Counter()

song_splits = defaultdict(set)
artist_splits = defaultdict(set)

family_counts = {
    split: Counter()
    for split in ["train", "val", "test"]
}

combination_counts = {
    split: Counter()
    for split in ["train", "val", "test"]
}

melody_note_count = 0
beat_count = 0


# -----------------------------------
# 2. Validate every example
# -----------------------------------

with open(V2_FILE, "r", encoding="utf-8") as file:

    for line in file:

        ex = json.loads(line)

        track = ex["track_id"]
        split = ex["split"]
        artist = ex["artist_id"]

        split_windows[split] += 1

        song_splits[track].add(split)

        if artist is not None:
            artist_splits[artist].add(split)

        duration = ex["duration_sec"]

        example_issues = set()

        # -----------------------------------
        # 3. Verify key normalization
        # -----------------------------------

        key_pc = get_major_pc(
            ex["reference_key"]
        )

        shift = ex["semitone_shift"]

        if key_pc is None or (key_pc + shift) % 12 != 0:
            example_issues.add(
                "Incorrect key transposition"
            )

        # -----------------------------------
        # 4. Validate melody notes
        # -----------------------------------

        for note in ex["melody"]:

            melody_note_count += 1

            pitch = note["midi_pitch"]
            pitch_class = note["pitch_class"]

            onset = note["onset_sec"]
            note_duration = note["duration_sec"]

            if not 0 <= pitch <= 127:
                example_issues.add(
                    "MIDI pitch outside 0-127"
                )

            if pitch % 12 != pitch_class:
                example_issues.add(
                    "Incorrect melody pitch class"
                )

            if (
                onset < -0.001
                or note_duration <= 0
                or onset + note_duration > duration + 0.001
            ):
                example_issues.add(
                    "Invalid melody timing"
                )

        # -----------------------------------
        # 5. Validate beat-grid structure
        # -----------------------------------

        grid = ex["beat_grid"]
        targets = ex["beat_targets"]

        beat_count += len(grid)

        if len(grid) != len(targets):
            example_issues.add(
                "Beat-target length mismatch"
            )

        if len(grid) == 0:
            example_issues.add(
                "Empty beat grid"
            )

        bars = defaultdict(list)

        for i, beat in enumerate(grid):

            bars[beat["bar_index"]].append(beat)

            if beat["beat_index"] != i:
                example_issues.add(
                    "Nonsequential beat indices"
                )

            if beat["duration_sec"] <= 0:
                example_issues.add(
                    "Invalid beat duration"
                )

            if i > 0:

                previous = grid[i - 1]

                previous_end = (
                    previous["time_sec"]
                    + previous["duration_sec"]
                )

                if abs(
                    beat["time_sec"] - previous_end
                ) > 0.01:

                    example_issues.add(
                        "Beat-grid timing gap"
                    )

        if set(bars) != set(range(16)):
            example_issues.add(
                "Incorrect bar indices"
            )

        for bar_index, beats in bars.items():

            meter = beats[0]["meter_numerator"]

            positions = [
                b["beat_in_bar"]
                for b in beats
            ]

            if len(beats) != meter:
                example_issues.add(
                    "Bar length differs from meter"
                )

            if positions != list(
                range(1, meter + 1)
            ):
                example_issues.add(
                    "Invalid beat positions"
                )

            if not beats[0]["is_downbeat"]:
                example_issues.add(
                    "Missing bar downbeat"
                )

        # -----------------------------------
        # 6. Validate chord targets
        # -----------------------------------

        for target in targets:

            state = target["target_state"]

            if state == "CHORD":

                root = target["root_id"]
                quality = target["quality"]
                family = target["family"]

                if not isinstance(root, int) or not 0 <= root < 12:
                    example_issues.add(
                        "Invalid root ID"
                    )
                    continue

                expected_family = SIMPLE_FAMILY_MAP.get(
                    quality
                )

                if family != expected_family:
                    example_issues.add(
                        "Incorrect family mapping"
                    )

                if target["family_id"] != (
                    SIMPLE_FAMILY_TO_ID.get(family)
                ):
                    example_issues.add(
                        "Incorrect family ID"
                    )

                expected_label = (
                    f"{RELATIVE_ROOT_LABELS[root]}"
                    f":{family}"
                )

                if target["plan_label"] != expected_label:
                    example_issues.add(
                        "Incorrect plan label"
                    )

                if target["model_target_mask"] != 1:
                    example_issues.add(
                        "Incorrect valid-target mask"
                    )

                if target["family_9"] != CHORD_FAMILY_MAP.get(
                    quality
                ):
                    example_issues.add(
                        "Incorrect nine-family mapping"
                    )

                family_counts[split][family] += 1

                combination_counts[split][
                    (root, family)
                ] += 1

            elif state in ("MASK", "N_CANDIDATE"):

                if (
                    target["model_target_mask"] != 0
                    or target["root_id"] is not None
                    or target["family_id"] is not None
                ):
                    example_issues.add(
                        "Incorrect masked target"
                    )

            else:

                example_issues.add(
                    "Unknown target state"
                )

        # -----------------------------------
        # 7. Record problems
        # -----------------------------------

        for issue in example_issues:
            issues[issue] += 1

        if example_issues and len(examples_with_issues) < 20:

            examples_with_issues.append({
                "track_id": track,
                "window_index": ex["window_index"],
                "split": split,
                "issues": ", ".join(
                    sorted(example_issues)
                )
            })


# -----------------------------------
# 8. Verify dataset splits
# -----------------------------------

cross_split_songs = sum(
    len(splits) > 1
    for splits in song_splits.values()
)

cross_split_artists = sum(
    len(splits) > 1
    for splits in artist_splits.values()
)


print("PHASE 3 — DATASET CONSISTENCY AUDIT")
print("--------------------------------")

print("Windows:", sum(split_windows.values()))
print("Songs:", len(song_splits))
print("Artists:", len(artist_splits))
print("Melody notes:", melody_note_count)
print("Beat targets:", beat_count)

print("\nSPLIT INTEGRITY")
print("--------------------------------")

print("Cross-split songs:", cross_split_songs)
print("Cross-split artists:", cross_split_artists)


# -----------------------------------
# 9. Report structural problems
# -----------------------------------

print("\nSTRUCTURAL ISSUES")
print("--------------------------------")

if issues:

    display(
        pd.DataFrame(
            issues.items(),
            columns=[
                "Issue",
                "Affected windows"
            ]
        ).sort_values(
            "Affected windows",
            ascending=False
        )
    )

    display(
        pd.DataFrame(examples_with_issues)
    )

else:

    print("No structural issues detected.")


# -----------------------------------
# 10. Root-family vocabulary coverage
# -----------------------------------

train_combinations = set(
    combination_counts["train"]
)

val_combinations = set(
    combination_counts["val"]
)

test_combinations = set(
    combination_counts["test"]
)


print("\nROOT-FAMILY COMBINATION COVERAGE")
print("--------------------------------")

print(
    "Training combinations:",
    len(train_combinations)
)

print(
    "Validation combinations:",
    len(val_combinations)
)

print(
    "Test combinations:",
    len(test_combinations)
)

print(
    "Theoretical combinations:",
    12 * len(SIMPLE_FAMILIES)
)


# -----------------------------------
# 11. Find combinations unseen
#     during training
# -----------------------------------

unseen_val = (
    val_combinations - train_combinations
)

unseen_test = (
    test_combinations - train_combinations
)


print("\nUNSEEN ROOT-FAMILY COMBINATIONS")
print("--------------------------------")

print(
    "Unseen in validation:",
    len(unseen_val)
)

print(
    "Unseen in test:",
    len(unseen_test)
)


unseen_records = []

for split, unseen in [
    ("val", unseen_val),
    ("test", unseen_test)
]:

    for root, family in sorted(unseen):

        unseen_records.append({
            "Split": split,
            "Plan label":
                f"{RELATIVE_ROOT_LABELS[root]}:{family}",
            "Occurrences":
                combination_counts[split][
                    (root, family)
                ]
        })


if unseen_records:

    display(
        pd.DataFrame(unseen_records)
    )

else:

    print(
        "All observed validation/test "
        "combinations appear in training."
    )


print("\nDataset consistency audit completed!")

PHASE 3 — DATASET CONSISTENCY AUDIT
--------------------------------
Windows: 8909
Songs: 1069
Artists: 218
Melody notes: 633883
Beat targets: 533710

SPLIT INTEGRITY
--------------------------------
Cross-split songs: 0
Cross-split artists: 0

STRUCTURAL ISSUES
--------------------------------
No structural issues detected.

ROOT-FAMILY COMBINATION COVERAGE
--------------------------------
Training combinations: 68
Validation combinations: 54
Test combinations: 52
Theoretical combinations: 84

UNSEEN ROOT-FAMILY COMBINATIONS
--------------------------------
Unseen in validation: 2
Unseen in test: 1


,Split,Plan label,Occurrences
0,val,I:HDIM,5
1,val,IV:DIM,32
2,test,VII:AUG,3



Dataset consistency audit completed!


### Establish a Statistical Baseline
Objective: Establish a simple statistical baseline for evaluating future harmonic-planning models.
The baseline learns the most frequent relative root–family combination from the training partition and predicts it for every valid beat in the validation partition, without using melody or genre information.
Evaluation includes root accuracy, family accuracy, joint root–family accuracy, and macro-F1. Song-averaged accuracy is also reported to reduce the influence of songs contributing many overlapping windows.
All statistics are fitted using training data only. The test partition remains reserved for final evaluation.

In [47]:
import json
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# -----------------------------------
# 1. Collect training statistics
# -----------------------------------

train_root_counts = Counter()
train_family_counts = Counter()
train_pair_counts = Counter()

validation_records = []

with open(V2_FILE, "r", encoding="utf-8") as file:

    for line in file:

        example = json.loads(line)

        split = example["split"]
        track_id = example["track_id"]

        # Do not access test labels.
        if split == "test":
            continue

        for target in example["beat_targets"]:

            if target["model_target_mask"] != 1:
                continue

            root = int(target["root_id"])
            family = int(target["family_id"])

            if split == "train":

                train_root_counts[root] += 1
                train_family_counts[family] += 1
                train_pair_counts[(root, family)] += 1

            elif split == "val":

                validation_records.append({
                    "track_id": track_id,
                    "root_id": root,
                    "family_id": family
                })


# -----------------------------------
# 2. Find most frequent predictions
# -----------------------------------

majority_root = train_root_counts.most_common(1)[0][0]

majority_family = (
    train_family_counts.most_common(1)[0][0]
)

majority_pair = (
    train_pair_counts.most_common(1)[0][0]
)

pair_root, pair_family = majority_pair


print("PHASE 3 — STATISTICAL BASELINE")
print("--------------------------------")

print(
    "Most common root:",
    RELATIVE_ROOT_LABELS[majority_root]
)

print(
    "Most common family:",
    SIMPLE_FAMILIES[majority_family]
)

print(
    "Most common joint chord:",
    f"{RELATIVE_ROOT_LABELS[pair_root]}:"
    f"{SIMPLE_FAMILIES[pair_family]}"
)


# -----------------------------------
# 3. Prepare validation predictions
# -----------------------------------

val_df = pd.DataFrame(validation_records)

assert not val_df.empty

val_df["pred_root"] = pair_root
val_df["pred_family"] = pair_family


# -----------------------------------
# 4. Calculate classification metrics
# -----------------------------------

root_accuracy = accuracy_score(
    val_df["root_id"],
    val_df["pred_root"]
)

root_macro_f1 = f1_score(
    val_df["root_id"],
    val_df["pred_root"],
    labels=list(range(12)),
    average="macro",
    zero_division=0
)

family_accuracy = accuracy_score(
    val_df["family_id"],
    val_df["pred_family"]
)

family_macro_f1 = f1_score(
    val_df["family_id"],
    val_df["pred_family"],
    labels=list(range(7)),
    average="macro",
    zero_division=0
)

joint_correct = (
    (val_df["root_id"] == val_df["pred_root"])
    &
    (val_df["family_id"] == val_df["pred_family"])
)

joint_accuracy = joint_correct.mean()


# -----------------------------------
# 5. Display baseline metrics
# -----------------------------------

baseline_results_df = pd.DataFrame([
    {
        "Metric": "Root accuracy",
        "Value": root_accuracy
    },
    {
        "Metric": "Root macro-F1",
        "Value": root_macro_f1
    },
    {
        "Metric": "Family accuracy",
        "Value": family_accuracy
    },
    {
        "Metric": "Family macro-F1",
        "Value": family_macro_f1
    },
    {
        "Metric": "Joint root-family accuracy",
        "Value": joint_accuracy
    }
])


print("\nVALIDATION PERFORMANCE")
print("--------------------------------")

display(
    baseline_results_df.round(4)
)


# -----------------------------------
# 6. Calculate song-averaged accuracy
# -----------------------------------

val_df["root_correct"] = (
    val_df["root_id"] == val_df["pred_root"]
)

val_df["family_correct"] = (
    val_df["family_id"] == val_df["pred_family"]
)

val_df["joint_correct"] = joint_correct

song_accuracy_df = (
    val_df.groupby("track_id")[
        [
            "root_correct",
            "family_correct",
            "joint_correct"
        ]
    ]
    .mean()
)


print("\nSONG-AVERAGED VALIDATION ACCURACY")
print("--------------------------------")

display(
    song_accuracy_df.mean().to_frame(
        name="Mean accuracy"
    ).round(4)
)


# -----------------------------------
# 7. Display training distributions
# -----------------------------------

print("\nMOST COMMON TRAINING CHORDS")
print("--------------------------------")

top_chords = []

for (root, family), count in (
    train_pair_counts.most_common(10)
):

    top_chords.append({
        "Plan label":
            f"{RELATIVE_ROOT_LABELS[root]}:"
            f"{SIMPLE_FAMILIES[family]}",
        "Training beats": count,
        "Percentage":
            100 * count / sum(
                train_pair_counts.values()
            )
    })


display(
    pd.DataFrame(top_chords).round(3)
)


print("\nStatistical baseline completed!")

PHASE 3 — STATISTICAL BASELINE
--------------------------------
Most common root: VI
Most common family: MAJ
Most common joint chord: VI:MIN

VALIDATION PERFORMANCE
--------------------------------


,Metric,Value
0,Root accuracy,0.2875
1,Root macro-F1,0.0372
2,Family accuracy,0.4361
3,Family macro-F1,0.0868
4,Joint root-family accuracy,0.2431



SONG-AVERAGED VALIDATION ACCURACY
--------------------------------


,Mean accuracy
root_correct,0.2675
family_correct,0.4234
joint_correct,0.2328



MOST COMMON TRAINING CHORDS
--------------------------------


,Plan label,Training beats,Percentage
0,VI:MIN,79505,23.899
1,I:MAJ,54501,16.383
2,IV:MAJ,51634,15.521
3,V:MAJ,45640,13.719
4,II:MIN,32574,9.792
5,III:MIN,17038,5.122
6,VI:MAJ,8342,2.508
7,III:MAJ,8256,2.482
8,II:MAJ,6595,1.982
9,♭VII:MAJ,4368,1.313



Statistical baseline completed!
